# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.

PubMed dùng EFetch GET theo PMID; các website khác dùng Requests/Selenium.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.

Khôi phục checkpoint PubMed 6.000/110.000: giải nén ZIP đầy đủ
`ids_5979207_7061937_synced_vi_en_zh_efetch.zip`, đặt `RESUME_RUN_PATH` tới thư mục chứa manifest,
`ID_FROM=5979207`, `ID_TO=7061937`, `POSTPROCESS_ONLY=False`, `PUBMED_USE_EFETCH=True`, `MAX_NEW=None`.
Giữ `PUBMED_UPGRADE_EFETCH_SNAPSHOT=True`: chỉ nâng cấp transport EFetch; XML parser/schema, core/runner/probe
và dữ liệu cũ giữ nguyên. Log sẽ ghi completed/pending thực tế. Không dùng ZIP diagnostics để resume.
Checkpoint full ZIP được tạo mỗi 10.000 URL mới hoặc 15 phút, và cuối lượt. Nếu có Secret HF_TOKEN quyền ghi,
upload ZIP/checksum vào `checkpoints/<run>/latest.zip` và `latest.json` trong cùng commit. Đây là backup;
run còn pending không upload corpus như hoàn tất. Có thể tắt upload, ZIP local vẫn được tạo.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
VOV_USE_CURL = True  # vov.vn và các subdomain dùng curl/Chrome
LONGCHAU_USE_CURL = True
LONGCHAU_WORKERS = 1
LONGCHAU_DELAY = 5.0
LONGCHAU_ATTEMPTS = 3
LONGCHAU_CONNECT_TIMEOUT = 10.0
LONGCHAU_READ_TIMEOUT = 45.0
LONGCHAU_REQUEST_BUDGET = 180.0
LONGCHAU_RETRY_FAILED_ON_RESUME = True

BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_USE_EFETCH = True  # Mặc định: PubMed đi EFetch GET, ghép theo PMID
PUBMED_EFETCH_BATCH_SIZE = 100
PUBMED_EFETCH_WORKERS = 1
PUBMED_EFETCH_GAP_SECONDS = 0.5
PUBMED_EFETCH_ATTEMPTS = 5
PUBMED_EFETCH_CONNECT_TIMEOUT = 10.0
PUBMED_EFETCH_READ_TIMEOUT = 90.0
PUBMED_EFETCH_BUDGET = 300.0
PUBMED_SELENIUM_FIRST = True  # Chỉ dùng khi PUBMED_USE_EFETCH = False
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

PUBMED_UPGRADE_EFETCH_SNAPSHOT = True  # Upgrade transport compatible with existing XML; keep saved runner/core/data
PUBMED_MIN_BATCH_SIZE = 1
PUBMED_MAX_SPLIT_REQUESTS = 50
PUBMED_FALLBACK_BUDGET = 900.0
PUBMED_PENDING_RETRY_PASSES = 1
PUBMED_MAX_FAILED_BATCHES = 3
CHECKPOINT_ENABLED = True
CHECKPOINT_UPLOAD_ENABLED = True
CHECKPOINT_EVERY_URLS = 10000
CHECKPOINT_EVERY_SECONDS = 900.0
CHECKPOINT_HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
CHECKPOINT_HF_SECRET_NAME = 'HF_TOKEN'

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import os
import shutil
import sys

NCBI_EMAIL = os.environ.get('NCBI_EMAIL', '')  # Email liên hệ; có thể điền ở đây
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '')  # Tùy chọn; không ghi vào settings/log

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
source_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
previous_tag = source_tag + ('_efetch' if PUBMED_USE_EFETCH else '')
default_tag = previous_tag + ('_longchau_v2' if LONGCHAU_USE_CURL else '')
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = previous_tag if (WORKDIR / 'runs' / previous_tag).exists() else (source_tag if (WORKDIR / 'runs' / source_tag).exists() else f'ids_{ID_FROM}_{ID_TO}_synced')
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and (previous / 'corpus_config.json').exists():
        old_settings = json.loads((previous / 'corpus_config.json').read_text(encoding='utf-8')).get('settings', {})
        if old_settings.get('pubmed_use_efetch') is not True:
            raise ValueError('Run cũ chưa dùng EFetch. Chọn RUN_TAG mới và RECOVERY_RUN_PATH.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'huggingface_hub', 'curl_cffi']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    pinned = {line.split('==',1)[0].strip().replace('_','-').lower() for line in requirements_path.read_text(encoding='utf-8').splitlines() if line.strip() and not line.startswith('#')}
    install_args = ['-r', str(requirements_path)] + [p for p in package_names if p.replace('_','-').lower() not in pinned]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import importlib.util
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pmG-EZ93b${1iaaY@doMc4VUT4vcG}?G=Z{ud|HEi!9fL7pAoI5j&I3&#vNfSvxi#+tHKo=<x1Vw;#f?|;dK~l6pQTw3}TIByw|C0XBx%ZNn!<mu1*b49rhrIWkd(YQBU-ahY<|pHE93@gjNhv3JRL-}<6qiXUMo}<H(xQyQLIlYO|3sC<!J&*}ITE=%j^v3brc#z-oaQ1BVIG{sBCir}b8~Yenq_HTiXfk4L0-s>ah}dZm|}8RMrl$AJ>Y&?0bg#$Pkt6<=6V_w(>OY;f0naY1O<QlvIv7rnn&sjW4)@3VsrPWMVctQSx^!rgYzkVnn5}Cmd@&5HOOLKn9C}O!gM4@K^d4^r?c3TWiF-nFqSir>jkc9aSvZ8jN>W^%QTJaea3m^he0^il&JEUmdms;LvXLi;;7tELt-Ei*4_>7Y#!u!dO~wwR-itxW-m83Hb!!c^#>!DcH0|l2>j);tn!2@;l*h%D)^yC!~9a7maa^&wJ4bkovIvf{gu<}X-Hu@%Yv|UX;d4SBOLFiX_UAYUA!dCXF(kOOuEjKkM28t!Sg)ck6QyRrh!Yt+W<A6Jv_bhgZI?{Zz@km(L@%QZT1$N_{@8vyL*_9=6zw7uK|uEFGvgpJ@(YkdXIhK<T8tcP&z#x9*sqkmS&b$lzEi78kDuZCut%vSI)9>?$atpX;=}{9UfWc^Uf4nr$<~Qn90P|D0+Pc#-rtFD6>+yJXFcjQ<8@KA-J3O#b-fWNxojWqAFqe2@Yequ&#}bAJzir&VthdoEhHj)pUgRjzAV^%L<VUlp_sf%FVJ|7KKa7qJ}B4Ih>cWAeLI|QOqFj(p7IQ`m8#f$&nA?_H$s#A(K2wi$n^B453Cg^%PLeF&g2O!9=_>6wXryb}xsOisf@zIu@yd2&&ak(WSwUAslA!A}W~u4?mJ+I2Av6{ITblUGciGj<XBCK?Y9UgrVdWG<f1P49KEkv;**DP47n~-*R}pJdamATwJjuWQ;{RAExQigO*UyUp9+Klr?~Hl#EzqYlPny&i2#Cj~+nV9KCXD^<E)M&^xH&(j`cJv47C>f-IBC$W>!nJk-;)vIW&BD*c96dPZMZyn>nkOy<#eE{e3uLC`WPW5}gEmib%+hu|J8h$xiB=R<_g57@+HBxxPkZaf$1Sd>$#&JURjmg$@i5sM1OIO>J$JGkLsLu}8K<~rMrUTdOFWa&D37X--Zk=W4H8bKLvYDppDZh!jhV=;@0SpXp!h%<v8unsOZ6+V6@NnTb7H!q~-!kR?}S>zXxo~po7NnU`ufDWkDH;cw>gSD1T1&Cw>a~JNutYDpCzFM$@kW7%Y;jSl$UJMha(W7gqV;>z5)iawPzJ}-w#EZO=7jHwEQ3ZK`DoB3RrB5sqSJS8`cEz3lAClCo_-=^OY)WmYO_tV{Ajr2D_hggym~=CeB=g$R5v-faJx{<14;KR}YGAF{jZ{t$6hN<T|JDIN2(=}|6Fr9QnA@?GJ=&fbvd&hv!L$sgT1%jY1EWE%#~L^{MrG!!Q=V*_pWqY+WR0T@_yvv7i{R08B4}5sne9dJcG-^icp(;C%_TbA^|_$Vf+Y0|_+kVv4rYg=fSs!W81EI;p_@CO7vJYIpf9d>$t=%DK~-yGajI{%><Mp+kC@&~gonp(<21ca1#&r-Iq6E@91&ze_j&#S*-6DteT?ukH^DmY={sN~oy!Rw$u>iRICiZ;3t&7DflfdM^$E_Md>Oo!=rF*Ap}pEKz;OjZ>O&KwNu(BIa5dz@Ahm;o`WEA2wkH}!cl9sWmbXn2anX{jMVV@i<>R(5$ev8noV+_8&V2oto+-W>&$n+qeiF~_>uK#lEd-+ohjK9Y84=4TePd^CjT53)Uz5xB>63fUpFjGWCl4PO@5gDwG{Rtfl{0mu4S45R3gf^}y*Vd|XTHIJ+vc+00^0d=6u&S*T`=HeQUw!<dzvAdQ2I8L0UzZ&TO9L(Zt?Sr_GU=KvFvVa(UJ_aRkuY?FV}v}0HO(JZy?yAY}4G35Cd4h?Gun;4;I&>1(mE`Td2E>GZZ1uW+g>BvuedQ+fsOHy>PHrckBrF%)r0s@y=>=)~0Cl8_S_<P3B#yF-4i?((;*kwFa$k1j}hTVRWXdd29puryj^&RFKTw)&goP37BvpEU!x$$(ALRrx}1rm_8nH(B6v%Zvi@za|Te)!;J&B6}TW%;cEF`6#RZI(Y7QTX<e`joUmb}{TdSiN9d<%(b(jhS^Y(CzqrW>puz^<ShUZR&l7FcGG$sh+A{oRi8hMVJ;p3xYYwx9O>ea;ns<{OA(i3RlZMRP!m!>fowr6YP(xxn8_@zmG1Is4S=Vey4;Cj;IZdmw-s8$snw?aOx|X*}aZc@KDqj`EH4b}6?*?t{T4SLN44W93ZvVW{kt(IN{5VY};WVgR>_~H#0z<HX(-(ACY(p>fFuzEGte7H^kj)*N_#92j)2Jv5+gRJMe19ClWAiEWB$H^vt-%%0DNJbRB|W<9Q(i-XFo8Tw6i3?z3Ivq%Eh@H5M-{u03$WCE1&!dk^wF!4bG8prv^)h$HP%&MEw-v#t(d<ir5*A2byVI&{?FtVWPyd$NC0J)SggpVAc~1f5mq_TXe%SDeuz}>D99sO^eNy^RE{y1NS4O2IwnofzYqe};!X~PM%$KmJCv%qJT1$tPIdWd63^Ggne{ZaRTAt%g(mL|O;X~shtHlr`sB%C*rk`0<`_(3M0UC*+zoK;sUlZ5^N(+{u{Muzu*;xW`lw2ZDkJ_=h^QX-TEXsj7}$O+CqXzD9|O0z59C7c4zyUb5j&#7>_8+a1t9TXn=4KLl_{=kA%Sdot76}Jn=zyI8-y)>A-ySTPUtZ@4{UEAUUU%qlpeCOASPdA7!O#+E)#o~Cs5QNj(@tNRJ6C&<TfO%C48H7WYr<EV-t)52}M{#P%f=Bfd^DB9x0bVTjxVqlp{Z+iV8{35g5LBPmi1;lS^5p_Sl8HOU!efj$*Kd1ySAbRSv%;yV~CSaA1n%K<lBZs)VngP<~qT=rD!J%;ESVHBv+qu$Z3*6;83-;nY6Om6V`%)FBi{C{N)W;tf_Af@1ZntvO&abCk*1TVS)gdA-WijAr)AGN1^&<}*B926WL|amc=v0<)H}Xs)8$!40Ld_60_=$Phzn-O|Oq^$vlhKrI!f>H=yRwFY<mgN-iRand8_6&o3{W*FuyljU2zb+EK7uYtCx11sRQ1>OSEQH2%&trn=O=BW=zxp9B$`Hg}ES*o2<3xJwQH5i#6o8|8>%sNmWvf`39XO>JajP9wximOMRm6lvaQpB)!x#TJB)OcUB0N%Q(-B^Vsk83|ZPP!o1IfHI)eAyr+fy26R?A^;JoZeH#F;vou5N<Sy*3hSl7d$a*rWcH$<9dqgY;D2ybLf0{Sw#qIh9rV($1B<2qKXxz%n&>cBTDb+Qp)i5ty^p7D|%t-RL0rR`ONr6x<v9+7j1LN-X9)ttrx}hG={&1kf2abht6eAiss1UFG5ymVv+l<y;8dCk(;Xv5=k^h?UW;7RD@|D+*EVhqk3>tE1|lb8BzqZ40$vffb+9m@gv)4K<w+0#^A3rInHw)%5~n_CTjHTF0MeHB6-j)P*c4d5r7P44N5hU@@rP5oAS#l*wP&idKw4)xMSk613MFNmMd$gN`5?vVP7=K0sg)2;CCL>H37Y_M-K|U-#XexZBe5_B3FAS9i(=YY{wwXZ7orlVlhQYU*qT0EVq?e^UT_VDzyXrLM^^=*qBk)QWSTb{BSu(jFZB~k#<BWN>`<?2YNJ1mwWrxox9?Dg1(SkJ9L~SNiwxOPo3c|!)ztG^mD6cU_n+}`#!klJa?YAb#<BCYT`pTU+QiOGLWuCpgtV57@IDJ`!4ahPbM9o@Jh-$E!69wgjvGO&-r#cDaEjAxvQrT$JfdZ&%3a*NH<W<Q7TpWL3u1~bJeMf@2f_WUX!LNI5?^8h#m?pp2X>)>wHh!9Fvac<kK}upkdU(nx16si%3DV(IdOGTDS3M6=mxw290IOdPJ>HIL``{QECiO-5qffM7qlfZaj{x@F+u;sx(R+B5Jd;`&tO-9I<)`7Q#zBs9Ql6cigO+AQK3OYk`_p<jv->(~t}|7F<R=Tkc9?=r|50G-ChY!qy?>KJY5o!=zIouX9&yhNV6P^zEDK(|D8W7;QBg)SR~~qT~lJMEt{6#Lp`uDb>Ts<sfBquJ_R&>)JJW=*COAZWXJ*>!1@)K>|8t?UouCH?0MLT|+qNbqWJ!zM`o9wS=H6HuMY|xdBbvJCX@f#M01pOr0-F^8U62?`xO`<O4xHVv^}@dx-7~>3|{$HilczZx50&R_8YD9d&2N6wQZ8Fk{W6`cs^q;0Wn;lPk3+n@mCJDh=y&9F*HT=8X8s$%(h>@sb0FOz?ufqB=_oJ6MXktK8kR<hP}AlaMpHxCLzAmguBM+$RTRLkWdWc=yox%cl=M+FC{v2seqkk|Q*T#Yo+aeB+;~jEQ#AI&-PLP3I7OvOij9mbCNrmhqyQxSag7-+gV^4oualnbcc4X({=s4&jn!<yUWhnM}5irkDQ-qoMjVzxnddf4X|}+fY3C<nfL;l{4!s5XYBav2B@Lz4<cqdR-b&?6rLP*nFf85Xol&JFDCg$X=5(ijGy+f}@*rw-Dqa+qjiMMc8D!x@rri6oqDmQ@39pEXB$J=G$RK%VM5=dyEw5Q7ee)EzLVcnbixmrqvCo(&J-DFHcuicpGYOE8r<<SrVWV_sM<Wd+X*d#^y9pt&(~Nc$c7?x>JYT%lBr6(ZQy!bOM>d3|5v%mWsSy7$EqRahzLA?5#qtgl{#Xrk_TQG_5_?Qr1ox7|NPHjZ-=kUB}zr=25FS-{sZ}g@kKiMzhH}hzS0USndNQe^Zngb@NAHH<Kte2$T}E#$HE(pe&Kupp*dRp_O3mSB#+|3~B|O^+TIxkcU(1;_=*UG(Ja4<Me~`)3`YOpyz%t_>1%X58gfKQEv~WF}z1>kn6yKC5KwxZvWsG%hhTU4%RkFtdKDVz?!GiDuthhxlTeltLgr(c+~WL>d3IE?s6W43@D5%8d0c;R7_+d^CnBeU81aATr66Nh|NH3v$By5l|4GAt8yMox`@U(+zh(H`wskg+40vl<=!YyGj(*W-tfv3rml0Bt)g(bD1EA@hnq;+h3}i6E$dIZAFJany!;Pjp6JyKtm4C~*RK!(J`SQCaTHy>`8f!m2hG_=Y)FN1Z0(ay7xFu1y<MnkB>r&}!b+#(@)5lPlF0HQ>W1h($j@g*x~2F-SEah#edqIi2v0OV-{l*7=euw{_jG5Q){Zt6mP>Ft9!FFV%>7xAA5|IZPQj70ECA?a;@??aN^9fN%_$3T)Kl(1>1`~X2>R(z8vWaD>(RoLx@(Aj=s=XTnoOI1tHvF(T!nm|aFEJ{BmUWdRY7Yt12N_tY${RUoU@hj>XRd&f@Sl>#NAC9XofXWQha6B-}vezKd?>8j!+mQizTfD%}YqW30slg2VHoh-3|iByaG}9<j$(o9h0y-S0}U=&+cBU@3Kv<8zN|B>90dYJ9}$1_K{k&k*rA2i#>7cdNkdfyv+m)%d&=#zW4}LIGtm?37Sqv$a-YU8n<rU0NX+>YTdSOP`0}HE^koJT84Vn=(pVLy295wwB7MHLtKO@w?R7hsxqDNAq^sK<Mg&)9o%8nqHG$y+=m*6)lDF@RrI=5VP{$)du!^Nw%NI*l2#_qs6u3IymXGyJHd;L%0qFv9x$qAS>ftI6lE4wjzUTiM$wR&+vhh@l5%*b*Su3!R~(igRQA^tDlB}aE!V=EFT=>Kjc@~++@GfN3iNfUwmu%*ypuOvcW|{z$UuW?{7`q3=XGY$q<K{}Ipy@FdO^~A#$WE;JBac4Ja7%PJ`zxgDoN;>R6Oe|-&~LB>9b3{FFF);mYBz*rlx6W_hRwbCCyl%^^D#3>$9S&Nuy4N>Yk0hXwq_nZmMr-ivs8iA|vIc`h)cgct$W2RPFp*J`(wNtcT>OQlY8yY-wYKR>!289inK^I#M)1-QkcocR(gS4jkvg7KWOAO{k!V`JD(V(|@I)W~yCy-`RZ!XBgE_`DMtxU4DI+8dHax$I<5AE_F!=j=N<!+++~A%PiWvM@j~7CeNy3cU#?CFcOaT0V#2GHbw?uaQpr2lwRA#^0umI2fB__=ituW>~x2?ur-y@WQt+l`xiPVc6JD@k@?^@O-y6FXZCIwhC3N*y4cs?_8m;sVCWjQ2Q}CzKhqu91H3Rj+M$mv;N;enExh4e&5~kpdz=gW+nEFzFX00GJUf|!c3a$)I>7I(lRU_FsJCZ~+$R4Tpkwp|z0z_=ZRxJPr3;UIpfjLI^TE4+{r-CwyDEt3(q?*l@98wUdi_6@?ZTIrUqy&LuipGa)BCZzjj_Ag-s7v+zpq3xQD4*O^53XG6$a<-9Z%f9dh;tZ!RD8*FduoYavD(XOCcz)-u#|S|9~R5?Z17F_PB4pynKzY;*ju7)2rA271HaKLbagoh>*c$)2r9NRqzz2J=1e``I}0NF8`YY6Z(8vp_}aL^&dlV$m>clKf;?=cyxUA=3gRRAcs-Y**XQ8kDG74#NbfGSFit&dE%RYy8O)?EC2JK0yF9en<#+-Kk(q{^}o(&eP5&ZYC_q*Uq`f?SIJaNqRUs_?sm5494uCCYK3d86xtN1oZ7sr?&(+7o1W51-5@{S+|?qvx2un-y*igK{`c?xS)5_En|I&HAra6Mz(h65WY8~YvAam)2~N@t9THv#7?e8&m3(iHj%)OaU%mMSQhFsQpD@?@nnnotelZg|@7IH@yazP7REX@bl2_JP;-AP_6ELb}(ljjA;I<cWpV^O;qQXP3GSn+sMB2?6c42R0&-AHDy4hH!Is^n%vPuhwyLBGX<AtNW=}rfvl?HG&V)OT|_;I6nss3LUl$`5#wZ@xULcd>nG=&&vh7qXa*umAs_L&uhU3i<evO8OXvelG)=IFalQCn*N{73h;P*!TYS%m$eruPJ6|Hzv1H98L};@Yn=!PZI{D{-cUZ2!9W=K!l+JC+2R?E>W-G{f6lqLqDUMjS#dq0yL#&mX;j5ow6D3Qs#&I=JY9H@1<TxtiRy7WXcen#vum6tu3|I4F|=6vR`TG50TDg|IP~zsqc1B$r>!#VI{XF!2v~^()5#kOY(gBPL?PE0jtJF}kzaP8QH1*snLgq<oCd*Fau=hi^DuQ{vm}ujiV5mT3g({vBUz1vBVBgtqXU&N|P@y8l|7i|2&L-uF2TJ{MbCTY^6axO1Ro+~haL7;pw(@&ZH7IQqnuV_sP4tBi!zAD{wvk9`Q&c!dqPeNqgz9$$1E?JszXKW8#NdaxtDSkPAUoId|CtgYH{eZIi1za~SDI$*VfVc6y0C$@E`v1tpgLaeBE(6$x#K7H`$h5zi~(@&ng@Mfb`wqiC~?JD6vH@HEB!Yym`e=5K)_I;{|`aZk!zE9nEzOVC8@Dq?E9ASALnT})}O#sAQVWqflZ{z;}v#z60',
 'checkpoint_snapshot.py': 'c-qxj>vP<;5&y2g0#!edlqjBLx0&P;)}zL9?P<pG#BMtiiRIu(+#Mpm3Xpn^{`c<UL4rJ<Wv8EdA0!f3U>A$s-#&PFc=&+{Ugta`>#itBwq@CMQ}e2o-;<_W6<lt~ukSt(TIGbdQWMCoB)lSZ#mL)kv*Fc-yrCH*SzVUAjSdeF4|v(sq9t1@w*_C>*WYAa+2@kBTl-wggSDtj(!dyy6Epn5ArZ_zwY=<~A9%Co1v@x+eeve}A3uHiNMiEnv5Kw=mU~MOtB@=tUe^>gbq54%>XhS%cV1<jRqemv7;@phq`$m4f0uszbpE%CmuW7&@PNS2V>e9Pr=2JytUf!1UuXv<%jzn}@$XJgqn>X{+m@A0ixV$lbWM5D36=`hjuxH0fJK@~TVJy(Wz`Kz`1Vcu>F2*+y!T)&71@^Gur%iarryIHJRKYy<ZMlDXu)&ZvJ|#k^Np|n1xvM|enIUV9a&w4<mgC`KPj4ZEyy*y$E)>w9S+JfJK>M7NqgTg!{QPaB>~}*RCP;MbzLlppj<NYPg->BLWo-UYwyS2w#Xx_cp_ydft;xokr&L!Flc&4mtQCk@`FvCDe?n3o0afgrF7_cftK_$!r9B)d#ht9l|YhLvZYlvk~bti@C(;)#x@90Vx&bUiMU+zino*FP0tjQ(}5NKe#Zfggn~tg2EaVq`og<BKlzoOJe;0<mqsVa@nZ50XE%cMUX?hf-Vwt!1hb$wh5EK)VgXi6C2OCTDoyY&3X&W-Vhqj$g3JhD)u8AI&~Z^$CAdAQXvrM5c0<?c5^GrKd&@l}F_EqCXXo_hU?iQW%xdGna!XT0Gaq5lXz-S|TZMyqFsd6?`JPyL0ihC^Fj_9{ZU${5+yt0ym~4Gb0_F+3zxeYvUJz~BotX<PW4dN3Q@UYyS*Z151Zde^YmNhRK?FBe0M`482DZHvET>t^@&rUeKh!z}l<n8J`@kj2ZUxtOwc4)J!IF~4Z^Ngnl{eloRLkZ!@L%!ZuZ#*NuH^C6M>IoEzmqTBVcWKie6?IGF8_TcUnR%E)hm2bCJhpvz1p*pQ8x};E&;Sp6HsJUP$`KGr-O}aHKLl+6x}#YeaVV7SgWA71rs5uyVlyS<2OhY^)%{D5$bUZ=RJY45g-V{eSu)SGawL9pi$xNn%&8Qcmp=125}W>gt6P3*<%nQQq)N2Pgp3+n2bg2j>}g1fi{-Mr}5tPN-}Z7a@gK4z~&(nX?@=cx&WxjYJvC~ZHjv3dq-NpAQ*UA_p_la(VR{vnt*$2Q%N<i5$_o~Og?}N|3zgDFi=-K^XGOnaSDhPru*Iruhsx9y*Xrc2Wgf%l5_1Pt@s+8)5@|(w@L?=)*Rgxf}z#I9+u?Nvx8nT(9yCf^lik$u6k7MS$J;n>hZ~07yKc1Sm<$sG|BSw*srM-Xpxdjh3k=X3I%-*D~)}u%D}&>P3(2;`sCXY4vxYfyKB^W<41AnVN6=Wf5;`k3^0xh6~*z8O0^OUtYU$h!Y7>7yW+`l1WZ@(#t(ugS0`wimJ{g0D*4Nft}I~o?2;k4(kx>Q9sz=(AB3m<G)k#l7tip#PfYTHc#mPJ`y7$N0gdvMK0qi;JM4L*4Txpt;p{@aD-};TKAQ%Br?wQbB9a5-@C+bg(N&fWdG}hnx-o)z2)`c6B~*1{gP*oOluHpLk+K*D7w1O4UR)%y7W=hgtAMiCe(gO*G|ziAjb`or9$R7l(*bn1_X=@4xW_p5EY0AFCi*-M9=)_rp$8=xSe6<@(?tG3=z-&)&kTeo&ScMFX`h}3kXm)T$fp?P<TC&<D)Ni~cbxC+;PE(B(y4;6;i!QCUfF+h45RWI{`#6g#-Bt7<(vv7X??9;1J^&@m4eB-xB*Lmj2g1^g0?u**1ZP}6u~p8v}*-e9{2fc*Gir<(ry_MT}6N`=(`s8u3;FNKy|Jn7lS89n1&vE=@5z|D0H12FtXHwiu+-Y2;nl7-FnUMd=KfPwshC%aMOns9ejrAM${WPzElPDHLp}@tT4h619L2EVNNk^$c72M`(<x;NgSVsBaLF^i8N|81Bb`L4pxq!scRv$VzJ&;DXPTITNRDB^km(MjHNanS+8C?foosr#Qb;^yk-}TB4hA0o2o^(0!p=Mq;`@jt7y$Rw6D(Piu<wksX%Bt9nr76d4p2*CJ=hJUWmHKU3gu^J>|O(DU|v@oqxJ`9nzwxZ(;2>f6_jJQS)IJ?%cp4p~1p($2m+;@q!tgH15{udNq~zCB)2YKX@h(qL1|8Pl;JJMbSwCZJDUsQH8J?(YCLPb7lLX2f;17_7sVirp<ju;UdyhMGdtEV5F7vr!ifVmRB8{C3w(Rt-@wVCceAyjunGBm<b{()25c-s6p@*H@)xDKrIhZ3&#9_`N_CEeHOX;hMs9(Om*LL?<+q8I0G4UC&0WWMmwzQDrE`~BBAlI*+psafO-wE165P&TXzV1Y+T*L(PE2o7#RV;(YyHr8>qC~#=X;8A;bB*waZlMyLQ}DE*?iW`LuxhL!1}`@IH^92V;XV>Y%|ij+-T+*255ab@63t#W9iNf4ba`p@SNkK8DI9M)B5ZhlHLNO#4x4+FHsl0LoC?7+ZN?6We=K(2#cmODU&+^gRI=qlGdhFg$t#R)i$38L8IpRyT^*U_t4&>s!2U*XIon$&adY{By$ucuQTK=R@s98xH4Fbh_yf>-NMb8Meh1W`nfaF#jwZsbJbWOpebcAhv_L`1>R2MS`;li0KRyqLwfKlAh^`Wwetz=)Z-dl^qtj65mU5y9IMJmkH;O4t6mHkB))7;gODosUGu9cPt(qrSGwY>IwUvv1dAF-n_Zj{UPGZT}>+hgVscf`^Cz@h8zv&fR!Sth4QU>^;?Xs9qVHId<=fiG5aFV({XYykD1?8T2<xcUD-%q9WP{+#7-@h8RxOeVuGPeNalIzJ4s6bC%R$rxSu_id`TX)8V~z){4jZ1*g>0w!<ZLVU75C9xb|GeXS-p~fCTt7SQ~Z}QHap1!2m+p4@|-cG~FPNCl~4;=zTV180(GW&pV0%hMgT$2SgT(HGu4u|KjM#tO;M922Piwr1vYP+A9XijzUtg+df;&Au~i5&X~D~oqqXzFQ;#MVw~I4Iy#QWta`GlzWrPiC-2y_6?8If75yci2sV7HNObr9_GHpFnqOi+6eMn7z^&9xVEb$4f`A9syhBUP)ZRnG&XkWB)!pK8ClJCeV^J-tO{JaxypimL6JK2>QPD#EbB`0uUz?UPUD52iYpm*>bEz0Jb1dBc>vdoR)pDW&#_ad2<cBZP$Y(5b(ij9T`?LOW$hS(-_4u^OFfz?>w_RyKeP|9O;a=n4>F(OR3~-aU7ZH1k9n^#)l|wM%(tbPC^yVnzs4?y5%v$*_a?Qr+vCL7|VEfyjh^1dqECc0`ENbvGsg~8a&arl}cnYhSj>FjR6vtR)sJ^?zlTR>Tk<JA(SGTVg9lGz~@IGGO2ncAt6EPnVr=av-LN%|yZHyWMZpUfH%!byJne1BvyHVB~Sqn6)9CHaLe!U#zzno0X?W{EGF49<`hl8iNgaAIpc5*wZRh*DK>~-)}2C{1yW2gH6vkRxWp%ohL>ig0RCI6$YF(*GK2B|SLbHH!|#3Zkou6+T4GAeTYVXAL!MF;-{2G21<',
 'corpus_runner_snapshot.py': 'c-rkeU2oh-a_{pidUX$+*&9*P+TLvBV*sD++cnl}JF>isU^3uQGd(jraY!zkl&q8WACiY4uQ@D&0}>n(<k5b}!yfK`=%1vjzu4Vu&WyZ=!#%_ctcL9Fs_yEl>iTN#?d_d3IjL8auvJpgIhhyhB+W@#6=#&te3|ByktCm!vN_9Awj$4-?3Wc~w7N+1C7Bgf*)TF&(b;)fq<PK4y}i9bx-N^VCP}p{lZw&7qAJ$VH_Pa(PK%ro+2L`~<Tb6vWKI`JlhyNdR;$4?cA<WXKUsPfHg%fmw}txk8&>4%_d2Op0$&NgV2r}`3LE5acEtvRZ=asLe)jSOiAW%pv0pWLPOJTkFM`37r@wmi=K1MsXuTc~_!l&=<~Ui>ybg{@@J;dCG|Q5M2jL+ZewXI+;*yaUr{w4`JbXyt#pm}QlDD7VACX68nbGg)+1F`(@ZjFxg!eut!>_+O{pR_YWa&92Ujpfh5qZ3VIn#rqdjJ6cBd?Q1Ql)Z8Fcy=T(=53H{72!K1W8@fby+ibz6XzWx~4^gvzvT&2<`VDgj0BZ*36d_-yVGdZv`0Gz!J71hFvcB_+KR%qpFd=jEj<Q6*OL}R}Mn?c2)q@09cFv=z&D!`KLxzG?^Pi?%|Bvwz|5C^~C?yY+EF0)>JgE>r4#0k<&h_U<+W2*2SFS4q-<+kBdCJ3dU`-wT&-|oI=}-(>^{+^Q5{`uLV?K`!zseG^2Uitm8#mF&ueXHBJ`*9Mc7@XDe^BZn_T668IIf^tU+Aqr*d(!DUgM(+UZD1W(JP1dKq}Ilc)W*n9n;kG3WU93jnsa25yLJ8a3775Q?uN*X@zSyN^9tS^d-o)$HbFU6j>;?4(~x$|f<ciL+Li3J2jp45C-#a7{l3MSFJ3{1nCHI;L9nIbDJZoAKCs2VSmv^MB{WHlPg_-s7584L!Xs|e>_maIXv*>F}Rml<j?06seAh}TJ;E-0%(eaP03PXRU65W0~A5~w~Qz6?fuT)NQx7qGB+m9UlSFVOMcSui4>L>kLPw#A^tKYQ?bFg@0Utdf*b@@<ke^l4QU)i8L?yAp&9dOOLNbRJ6VmubBsesp0`(tJqsSuqE_9|cXl*#81%kubt)pvJmIDais$pWtA=a5l`~BpH!KQ4#z_;EWODaLM>4gvYcTj;4wZAaKL$Y7(ULV9Li;e2@Ts!`F{p<cra!B|mA(ES&)d$>DH_F|}r3qo&eu&F!s#BdWQzyJMJ0Hpd6iUt-0R29_%bCwkzIy=&1+tl@@}a!)NUFKRw6q;FGJv!S^FF(uSOa*8++FDoGg%{fESF$94cBFfR2m~T*$N1Sd$b6#pK6W(FEiI9unu|DX5h6is>o|ART)~L@w8{lMaY_baclB&{#=n$408<M^jU6V3}Rrha((ygMA1$wxm2m(kMr!`X3B(WqZS{y^@HYUkzMj4A)ozxABf^Q$ajGw%D`8bep1Pw(i@O=bU1ASXp$*iXHfKWI-N$~YoFJF8a>yaakPHo&}mDbc!0&42L{G#@7<g$k(t$6c&Bb3bsTFFkVpIQnGP|T&4&5%+zZpNn+=r(j7!m{(UEW4*Jw?h@~O_L<#XXFHCffj0#;S0$G2^gU`2iP;QCx+!33YOzsFqWq87}^U)b66>7(oix9zy?5cD3>r+vl<P$%q1Y0En!p6K^YBgj<YdGVwfO~conF&k<S3o+bh9&sXK1Ft?Ih;_B9%U&6p@Eh4>tdUmHUE-onUVdwOocB4`$)a)}muE*Bs8WJa@?+!j~v)dnaQm1K~QF3D6clVVoatrRn4MpD5Vuc$z5kx@%_1j)PNKCG<-C0TZs%+As7h5^x#%SlTjF%yjn7@R6$w1u$GuJ8TFRils|1>TvA5~I;bZ=7Is3xCkJi%Me~>eyYErClnT!vU?ew(K}jou+-X1pBZCw;`^tN*V<<pgUOA>+B&~QQ+j(k+6yShAYmsB|yCe;hs!UTDX_v^buZKFKfZXD_Ay|svW79@B7Ds#a=S5YPtCv!OrLNp)I6#7nteci2U3pu&>XU+>;ynXFL9UR)qj6b!kP>zu#n9C`+*{$|g%hG-891&Wfg*(HP*MJuXf4NdU^Z2FDko7r-zBA!&6}Zi1wh4R#c2K!W48+tyy%P!w!?eIqCW-Qiroe%#`oTCo(cx<RaXN3@`&_1l`!9)oD31>rl%4SQB}I3BoCljaR=F%_2>R5_<t$EJA-&%!-`A2`F|z;*-Y=<}h4ihd>!QsFP_2*nrrs0`Y7rhc~NAo#vXvb4T30H`;MELjSOTsx~D;#f1y8?Gu%`^t~Wcko)nZ1DDL7npr;nbUEgTc=K7Sj=kH@)-<VW0ZVZuF;9OF?SijO%#o%e9v*K0vX)EPzr2OJX<VM^bNsW<7|P0oGDLvD{ry)aEcC!VSQsaE(|tX!N{Id+$Tc8*g1cB+eGf|bYwE;Bmk%hbcp#<P{v_Wk*P2^!zyJZnQ@9L&xFIm1dLczrp4u`L&J@WwwEe|d*b4ocHncQ4D}z#P?RO>vw1?QV=~;Yyo&<eYEEDJ+EQniRa$^%O@$+Wt@<6y2r)`w2jX#uNUHOp{{T`!Isn1q6k<3HVnDZk2^?DX3=#yvsi2CJG2~&@H9iSO4cddfjUou>25Yzfu#cKG?}q&a#t;sgHB%vqmOvj(b4hB5Tdg7Rh+2_bJKTmjwxSy-S;}*fO}N|m9(e#99cKX9H#)h?l-!q|Wf(65^uwylk&z{anshx7O(XS!TUQ#eRu`5vC**$%+MSI}4$N=Ee$usJcU}$b@mhq0|6*&|u#)H_R^mpVed~~oTVd&tu?!0PV72;R5k)b(I4SZKI5kZ%Q?No{>+k~3^5F9FGVE&|ZJis&t9PH4a7R)W!!YbCB41mP2yG=%v|5Ufb9Kc9hRr}(pomH>Cn&+Bq_+A)>Y;usS@D(76T%)GE48uB#BLf0eA65l=B=E80$|KQn20IO=3L$PnOj|j_S0>!J6K`bI@0FMb}OvU@^a^}qaF6~w%5QfnY&wJQ*FoR$r_v*+ze#6N$W+%7fc7uLP($n;~rL4|AxW&3S=$5NQ!t4f+q(}Dq9w~*|NoGnU0LKgN*G*kfaS?LtsB&K6^RL=_N+`qrixaa(=}~A|MW*z<K^1e=!sQ#)2nty(|*!=?)bfzXGoXM)@0Y1w*7(Hz1qT#X6nEXIC}Ist_1KT!A0;Fc;(@xL=t2phJ=Q*k|c7kjHn%bL1#BKO>KS{N0RP(lZ8V$?V7f$w_wm;a^iySIrff-M;@riM#4DL-42nYDk^__#ZqKUy%Ct!@of262t4W+Yi4_2)dg%FMtEPMWh&lfs)?7|6xt$x9@+Shd#)Qy+8ibk3Rs^+Yf&r*V-buIk+~Bg&RcmkX-!uZw2||KmX~!Z$JDnBiB;DOiXKGdh<}OsBM;75j=J`3>BQ}Ww&qv3q(wA-~W3}GX4sKw1A<+plk+>t0X1ox9|VACf_%(JW?THmXJ-SYx4JR9zB0{`VaBR(^oG~PQ&%wyXxQ_c_Qi<-jUOmmXd7n5}(w@AVI<IgtyQ6n-eKZ<edlX{{BAU|39|=K;Kq6(Mlw|hC->=RxUWND8qfR_l{g+@Ai<qA?>TK@o2Pq3}?~IQ*QHSU9zF*DXrxxMvob0&J#9E(}-vMVdvlNnOyvHWMd@h;}y^*<-73J>FKMnB-i@({qKNYxQDN$p1}|~a_oN!Dj$<ycU8)-#Wqa`TVyZz?&&Y%rza;bPhP{`;+TKTgYY*Hp6B^P=18HEK$XK29I0}6P415aK}2GaF3054F_G!(=>8aTsc&OYP?0DJkY+3#Nj&}|-)6-!ruaI7OuCFC;lv|eT!L$mhcU;1_WY-xszMUL_o;Y)qcgqyabj047+!U;SOnuvAp<x3Vwyk+f#8}LEE-i7!4D0Ht_B<Pp>Oi@ytvGXi8yRC&=}wxs>wN}B48!ST*_HdWSfxK1IfC9%o5ZT;jlhj{$h*WQF_PU1+!7Zac8XU*qtzT>o0c4*crPMMysU5LFPeXcK5<pvj9HVnL&fAJqt9}R1{il;+U5@#G??3#UQ}0I{{lZ&>*wxe8_>}g|HlA&a0vh%d5bRL>fce1p5|%793-7Jd!Jh4rQfStczgACQ!q^aEQPoNE#8C6e)(TV$p5b?JXclcJkBL!?JgzFPfSTEi89&BZbvqPyNC!uq}{!ojy|A9_&~{?h@PXi4>Q(`snQpXanUc2JNpO>MI>GNTApd36!!$n!{PF&;sx&4^0`*<QEY+bRwMH>I)2--)LDBP=!BRfWetw^eXXAF;gr4Sjy4P#nY4gf*fs9M(zkbzeG<9k<cX^`Eq(~g;0MTl~~6d_!ay>9vXv#mqB0=+-F(2i4G>MG0TrWwLI1b-GW12)v0Qujy(ldT0%UdW$#VH)tgOVxB;>WhzLn&5ZXu(%xQ3zB+0~;RayFFq@Y^hQVuk9U1KE`jt_jRXE#CCI?aa7E6G%E*Z>fq+4-=ok88y?wu+~e(2F(^s4#`f7}KM8j0(Co1!oN70YSCO<IYBP`TMafaRwXdKpJ+kZ0;ce<qK>MW+ORy^CCWd^d*<Zlc$eg0*iu|0xw@ZI{hj%9A26k**-o~Yq3U}kC)1^dg$g#N8vIn&W6FKa>Qth-Uy!4Wiq=$gZOyY8I!soO}VU+)-ddeG8v>DYr1wvym^G|BbAI%gFEf?JbC%cm#6WgFP}m+wSP#+pqvllqX(dgk4BTj7Dfp@Vx#_$iL|GY@l={w9yzG_BwqBfhfK!;F}5KjEX-$TX_#m0Fi%(EvbZoN&C8f9SHa#4A=QkcRoZYjdeLOD3fDLZR&(|p7epn(<9S*|-D0jD2Gc;CV$&YsfwC!HMGGvV1Nq!JMtMe%h&RP>4tTkYK&2xQvqOHVD0jKnJRdU;Z8{{34NNR$k^<?IyvU(2GsE?l06tk@Feu6Sfl*ZrFF}X1q`st7TvL#RA#K@KdpTjo^G+?^Eg5e<t=%gUjoF{BtO=Y^RDujifHS;Y6&&M5AtnS;k<YRQb2KV4bfAjZ(En^FyrNl&Ne7`2y|x=y8XEOR1I>VDn83T@kZQQCjx#Y}?G7(?@Y!qa7!XebouLz2+C>sp_=-Zb1oC-pfvGJVZcPCyOr9jM(DfL0o}Sk?s@h11DZNOG26e_Y6KT-n9V;P59Tn4N6@`}#Fw=P~LPPUd$+3tK3A8G)sMHCzwpFVwyfRC`l$jf|^y)X&CIbn47FHHtJ2xmhUFT!cS6e@Iq(@9g*I{j#k~R|{=VB|`!IoT@qR~WxpPHoc_i$>yKCw!5M_qHmb&WrIl~!8muV95bjR^;fdmdUxC7u|FQ8NZ68I0%dwryAy&X_~<Q&++E6Kgc>%w7r`&c914y&ik5d^0wXwTnfj(RHP-NW{lF+arAx3yoW`$;Le~z0qJ-ZgQt1l{ud02c44Y1FV^|D_^y$Z&5N(e5j#OZ}7yUm;%=}GU}<SsYPko8pEwdM;#n(VM=F@&LJq8KsXX&-8p}P$SWM&92fXzLl_I!n|Q2wpa)}g7Mxdb2>CF1ZNW8TMNAjUbTVpQY&e%yVzTf^rW$poF+i`Ftwh!1H~_V+{FgB=*;G#>Q`z!W^B9Y9%70C1NigV2;DSSzDccqcVlG#p7=_`OVrF8NAc+Z>b{Rn=3@saUfit)PNyE-D6C0~%@_ab<kwhOF{b$CFLDo#Sb~w|fK3%7{XT=f}d&dX_n+&`)Fv`rGb*x&PqmgO2RvU4wOb4F4ln7!rgWE&c)>=h@&(&5HE^)m|F|DCHi;THuI?SnSu7*Zn&!K)E+g*LmxUf`#+;LC2%|1Txb92P6wF#caiX-{+#+yxnbY<4CRfIRtYz1kOQi@?aMSE{ro0OaQe(BJAq$XtBN7}BAb~16@8z$;>)L_(Q3n`U$N3d<8wy#h}E+~_?byr@|;*|+5=>T}7FdD82l=#9eL#517dD&Jp$H=%=i}>VBGsx=2--OA0&QaOnOSL)vX78jY&<%qm{<T`;wU&Gr@sN7kPgRAZ{iXSCB*KMuAE&0Bl~8j>VN5c|R9WEefX#CI7DN7B)TXwMUFG07h)+S+NiXc6wN>X$l|X(NbRK9dYWid1(vC3lNqZ4bKyYBAnVHqXNWs%c`j*e0Kt-P1#u>;i`Y5&SgYi^31S@iMMK9ACwfVvDW7f$P2I7(niuDMw);SSwm*AU`A(5~T-N3fstwWc@Zm6z1M8@H`5IR1E_WOH$m^MQx|BiPpj&pOFZeU?oINT4w4UZfZWhO@TudzY&sQo!|4tb@BvlO^aW;*%}6Zvvqk+j3)op`33&*_N{xz=#M;XCbsHo9vNy!Opm+$Wt7e`j)nOWNu;n@>!Y?P45z{J8#|ZKG+CFTjQ;f|<OH`jWx@80?P-v;gdH3m9DOP^%yE-Pq9Yr*PBH-W~IrXs2;Mvi7J0b3@$G<*dzaRp;PL@~e;brTmi^Szf<Wdw`xKmK}8Y)V{$52jC)0mn$bq<6|t(hZm+h>l|WznnA;~KiF^waAqE_GeaxMH+4^ZUI`r29M3tl;fyG<Nzk4ehFys6Lpau$g>JwLKAnTdnaXa8rw4)9XV_m12FA{6hF(H%iegK8MGpP$DEGUnqn*{xpzomEA;2z)E?49wtm_?VZct<}y%*L;SLOJd_rjbDb3M+#A^8GfPX%autD`s9akIgzY4w6gWXVbH#3hRRev-!6O*`C0>iu}kG=B$xRRaLnR3bd;&#C*d@Xx2{QQ~7N0I&bR>oF#FI0TK8<<G@oe$z%|BjF=cNMV_eeAS$N11eM(5Mpu&O?zpIKZM_0u1VizNU7z}9Sj54qU5Bx!(r@brfo;bPt6#blXQ(<k4@4Vza5*zRNRA(bz@@aI{;eA)fvehMAsp4TSY1Vj{6w7BN*ERVyl_z%5*EnM0{ANl|k{Hp&mr-VPueP{MT^oj5pi6c!$2pMfl0@hIvt%R%FyuA2{gFk&DwEKQUrx3Wsr~y%>zk2l6elv_F1t#_(IA-x$O1F!0XN`^{-1&pb@!<F;!3<FnnTeK=j_^Mo6+thf|e!p`TD40`gLxmG9%OSDHJ|7rlkgunXoKpbq%VY%N~&^62u#Kq#fSY9P+a+WM0XgZ%`a4GQIz`QdJRSE@K<;pv>b4Gcpek4bnUB#(MIPe5JwChe;=4#^hvp5^3yo;-1Bz^9FGEa9fNd3lO_hTWZnaVRl_DcJ#O7U5b&JEq2J^~{4Kqu`+clq83klYg8IDqd|_+|PoU|aLSS-xlecTEiZ1QXg}tp7Kq|FWhug!zGk9sjTPIfOeDBK?$tq#YvAZIQJs-4owQaSPCT>g9Uu-Ds1&JK7W`-5dq|m?>_Es2Sxt-AU;ZRq@>z^}*Wk1>QI;lNm)F9AVvRcWirY|IVMFv0!-_iSO8kL`;8H1IPnGep}OYP5?RgJ18sf9V@f`OWd2}m0n^gXs2Kb&iZ(7h;L*3KCh@c%Vdhz96o3-e{{#o>lPdW!W!cLWLfzYlg(^`;4m2{p7nfqO)%^?eG9XY6BmE1n`Cm7SR}e#L;a=4N}TYTQyx{z1~XExc*}53*$lp8iA#!NxS%yt713fOrW4~t4#AQ+peDa)443F2YsI38s0mgA<Q=uYFK!`-VoV6GwL8_*Uns&m4N+fq9rK$6m>tt!7K-j~90|_UJ~&gx$c^1WweEA(>P3YY?KX{8$*g^m%#tDwJ-8Pj#_&HlN(s@{+c8pj`zjnE#mR407i|$Yp2J2lruv2oenq?IS_`%rP~>gVt61I?=m7KGXT&vy;!<O$=AEyOcM!XeUm{SRI)AO)^?~aJ^H!B*?4%EQYo|E5Jt}#-ab6Avpr7Ly1u~BLp%KUY+F~rT3@Em&p2IRYSez@Ur0%z7<%e}q7zh6cjv0q%',
 'crawler_snapshot.py': 'c-qZ<>vJ5}b?0~d6)mc~Jwpu!1CZ285Jxsm$uY|krBIZ2QxGy~%=BPdn3uX|2*4mdI4&o)H{Oajk8*aCiWF}uDLamw#G8%9?D_*$`af7@O7f@u7xs4^_x8R07yz`q8<h?Abl-dK>)i9c=U%#W>8r^Xli91c?gafbIvlUhrsLyao{VRUc5r(Zyt9nc<6s=^#pA3MOlAZAcacWPG@0%PvouJigZK!~?Ms&~tqs%JB<S^q%f&K{d%Yl;%xCE$h^EuoB3dM~X|}dz-lhBVD9vK2)1QsUai2?=N;hZA=^{>B!5|(+%kg56^cSY^w7*Q#c)DmyOSaD4Q5r{s*Jrcw%SUm4xtO7Pl=b?v$$T6y;(@Kdh>sTIWY5&oH<M@@?Z>H<8bphDkxXJ!#{9<c@%L-9sdhb{M9H|lTqI-L4IUVk?=5CMQ_l>1|7*$IJddJmgl?Fh@(*6LC-EXe#mIh{M}q+dnwb}GXS1n!o<xh0^kt4GXIQV(2YpVX>0maozifB15p~`En)MIjh5bc~mMxOLZ8kf$j|-k_th{~U401V5`al*Q0{3j0j%i#xGp2iZ_x3EAw&)?7kCTO0G@b3u7F<3x74FchFD2vnb$)GeX?`4~>Ff|ojTtDsY}4W|7cqbA7e4|%P<+83-^nKj@3KWc(li<-;|Q3hgISCRrc3+rqBk55*4DoI@|`zsy>>h32I0NUPPq2w4SF=Y>rR}-QQ9Bfq)is58=dxMxOV5YFTQrScjGr+zI}H^*>G)b4O<{sF8Yn;_F91d(inZ2T8wN@XNL`gl)%J(vz=tK;VcDWG@4p{knCej4O$T&isE298%0~!o|OXaQGBEoOnuBi+@rD$LNWz$m@Gz&W%9B;o5#~enC^v5n$e*Ys%VRWX#>Mn5K7x&RlCt*Hc9%uz2ilkaVxFB83Vn*+~r-cZ)s_XLQD<@N!pMfS@&+b#0Cbz$`-xZ0soXLi+C~zAg)QH>P@3b%plwJ6g(3Q!}f{A`8KV4G62j*^R&H~%)=Z^`!G$g^`*z$*Zfl&&&N?e*1pOJr8sS99p7y4Z5>|4;Hg}Du$;`Z2FhV9(~M|sl=YLO`=w}{VSGfBr;F}ZvmH-q8#cn_Vz~ZX*fdR!XVD;U(Z>gFfC^*{XJu*JJ))5f28k)?7J*X5vGx*sE2|$v7`AESH=MDgRU>I)+QYY|VE|kXh}8((x>1d0z#C8xe>@FCUHDU*3h{~pWC)BK&kp0%V%wB9%{cj5+@P^4AbJFDI5e%DmkV!y>1NoHG78cxa`;w?&bN37-dpp|g`URkY`ND+!=1Ml!CT8tC+h4zLj?nc-gF0LOTRTexzainfVMH2-J^E4NRxS^$<ud4(?@XHG8x=~X490>@z2pqY?TJMvsu5{a$$BDy=~*SbR`)E4f?ndgkYS>5WBWbD~hr52Fnb&;(kX72>(~3@iKlHls9dJuPqm#%Rw|I)H){85JZ6i^a4h>Bu>GT+8@nk_z?xu_%K*sN26_>yll2i`?1$Qd8LG4NPGI7_Ul6CiFGvG?-;^-*v|SRF!7#n1j3Chc&Z!hgvo&B6Ic`OQZFFGfIhSr$MZM?f$b%b86Yh*iV%iLJRUTtc9RRQoQ0Q<=8$`0@DqU+Bm;;`gMgL=0u~x4*=RYLW@gCAAd^AKmW=l7m|CW2m;1iklz+@fQL#N;j>iy(AX(9f=KVH5gA{|yPy34o$~4=MnC8bo8%)9&^qk(U8p5kdmZ>l}SOT~DU`u9UZoOpt%`H*aRj_Nw>tT|D7T0`yI(YNWD?zkejAkH`$3c<>*>X5c`U&V8qxC$UftjHB7lM1(ki-&$AqIII?8W$wAMv6;B8n}op~_x_H?$kDaG<<tk6(Z|<+I0Y?hKl$(Ndri7j8?@Mh7cA!(@#U;#hfkopHjL1akv9YKol?2rXLJrokrc;t7GnKK|{|VlmG?zrC@s^V=J|@pD4v{#Tm*VR&oGBR5ly8QTs{Q2VL$RHxqj2LTX*JCU+<Y{<X{n!3{5+1j;~(5)fsvLh9DgJ-(IW^s_&a?)@Yv3-v^ht1y%t_Pc)PNzHww|E&gV9G`Yc@17<_$A1gz`P8aS%sf=ej?HW0;*zg^6B8^cb3t3otOeB(RkKpEk)#-G^TBlCVNYMM>s&`GWj|LDI#tpc=gT_3JxZgl$18fhzId;hHaT*YisYDt)L^V9*z>~2gb$Kya}_T@{vzh4*%mSB~0VDb}FwI*NrL4dD}Dp`7rJBq}b|RRVUha_#Y*_aXc_g3ex$#8CIF5Bv;=Wv`>_9$MLjG3s&vUW^qRP5WreyTSBR7tk{|@f8TC;9}C;;;=6<fQ@=;%P)C`;n@KT6y;9?3k`SZyX1%z~M2-m@tsGY<uNr-BM{@|`gT@Kb|7|-}OxCsoCPSgLz+dU%)E^Pl{0fX<U7v;ma!Lq*|L=qOIHPZt>kPG8ZI4Z_cevax;9sQQf=#luyMgYwMk>sKW}c)*JM|X#bCHtgvlU>&cE{0VZx99Pc91Fu73hm7T@dA-k%|DxKUyqO#(DH78T9x&5ac%0FF>j)Fq@n2R&?T8#L@UQG1~^WVBV}EX>UMiNQU5XGf<-S^<=Ov&}g6QPe5wKNB!|KOYX%@D?AJX+b(e+YAJ6hi^n32)X5iMwb+v2kPQ`%UhIY~frx>dm)Ap?tmdw_&GKf7t95eua_TNE3-YO9!>9RLnGUXAFK4qzQ#`vhl&*@QzQod43S42a^u&R5%0!=3NR`{x*O>>MSn#K9hwB=?a3W*d4mt&LE`<8juh3r3Nswt6>WI<l-0V7YAsuQOA^p6XuBd4$xC;;#+sCwO0#mqDu9lj@j0HUrxug>6oY=mHG&NX0P;O(tg1|ZH$&ArPnDFvK?8Jj7YwG9dNIyqM{^z_6D`J8)P-7ibWwwlGS|`hF`K<~W_$ChyAQ3`HK<W7<UFoD{rl?Scmq~z~)=ihM6nPWX$KC?+^r8jPH8SiF`eVMbnBIgpZvDnGUEYO}^=QMGg=lBNPB~2WsYHL4&X*azV-7{H*eV-3Bb4>RCrB~6g|Vr#+4x=zrq{9ys@9in$Ht>dk_(Kc0dh38grzbYB&;MU%s+xKCgDnE&XQoagNAkjni4AREK^%UH+z<d@weHW`VC0~zpT{3uY@uv7APVY`(HWMs;%J6&MKsK5dia0p!}@D_R{6l(EgnAK+nFH%$Av_x!Uqg$8LsKM2~lNJ?V$6ot6gp1Z_5E`fVE6pnd>buvzdtymW%9LM%^tte2{k7Qm>`;d7;63T<&2`*f5WLCd*q2~e6v=u%8NKvdcGhD3VvQB-d8m4BT>6AJIJ$>7#Y8>BQ&l8kk$c39H~GU35i86Eb>C`A@aFwd<{C%E4AN|0hkXu9C-)*;dBN|jOrNdCN=(by(-OLg{UuzYmcCD0zkWT_?_e*-my`-;)5St(Vios>W@e{-8tEZ?mbw(4e%MnF5FcJ4$v?xV-BBc)MaT5s8=*^L5o0$Dx6-TeJ9nSu@oH!shd%(b#SfIq7U5fvB%xM;cTa2)OTNOeOc2oYSfTnaP>i;IEC=0C?$NsS9yl|3ba9WBs6=Ep>PXpvhA^{f#K73h&(*(F+Tkj$gzu-v7yKG28xN|(7x8E~0RIn;r+J=wp^%2oA?dSdhocQfZ@tC}FfR^o@2!Y-%Eq$&p%>hOi%%MYcwa~R1Mt{M5HjuF%e3Fx7=O?J&wyc&YrjC2jx*81Zp%K~=A$X<E*s2|VCAfSYitfoCMuw>EeK@}PgSsn(lAjjnbuE8!$!k&l?b0HTX0hk~1kIOmfmTe19AP{Mkl6jB5L}slxt?dIwLme1L$KA~~WF+t<_#c3)S6abkvd=?x$a?lIv8Ps<SZFcq>nSF+M?NM1?rvTqkor;71pRWQgD<skV7yL`dwa_PaK8I|#{n|LFv<A8Sd1A)amIK!{KR9@XVZPdMQI6j)cJ^gK!t&$rhO#NQ2464n20llkFKgg1g}vAziFhRL@chAeemp8)dad&WwM09YL86f^I4WGARQS4vP-kw)<kq<<+=cj@V}y4txDdJ{})yx2W`>H59fvF;!_UHeS3aLIl8%!^0()O<e+B`$a#6{31?7ETLBvyRA((L0@<}DS%tmWu@!T5SgqLDqvqoB=LQy?DAp`mG;J{o2Y`QKNc2EU@;B)bq?^vU%*rPi#;HX#fyOeD!MHk1(^UehHEJ-SYEi3Ud5h(Jab}+^QdPyvCH*$`mk~>>xmWiUgg-P2J%Lg9Ak(&|kbAH}{GPZ4<Q46*)0Xr9%CW(Okfg_4w$x%Y_WH$d$Vq3YJ+;{bt|_%o<9#w!AG2A+!7H{@=VL8U8RJTny3nK+YO8H91{bHrX+DFP0T#%uJI$%~02z|ipnNc_pmCM3fc1Hqww7R1sRUrO-_UlNXVgL5Ks111$_Pv+;hRu5*Kh1YJi~^4^9DL_qfd;R6=kgA#%M7aw;;!`F##Yp=7Zr2?{quu=UbOIF7wm1@U&TsU?{gOv+@OwmFYu89L)xHo{USX;X89AK1eLgxog^;MkyhF#aJ0gAi4<Y{Ol|Wo4Hq#OS`UZeFa%5ExejTd^ntykbwz+Bj<#PxJ_7Wuj(6zznGq}{17|`hJN+xmF7i2V>GBtWNkpZk)S%wJssS@5*-JNnQ?Cs1&Z%cRGJa;5vWEl#M2k>+H62DyhpA;NU3x9!WZhMF%+5zFw<rZcOpHH!mE<$aMmcyWy>2bsx~md@(OjkW>G{|c&^i|aR-_}u68o->0^VsvvYNKr?X3uJH7q!Y%dzu5ovcKm(wg7#s*H<Dq@1OWpz*WX2d>(YOq0_jq*lRlx4guA#Nnk$%_GY7i>Rqw_wHK&EVm9{)xCn*Mvuj>yrp^FN5wgU5t-K!Vc7_=2s@_M$)k%zn0QjVhrdWPMRJXiOrups@0DA?<|2Y4lRktx+vavGjy^+cJi?Y>pUs#$qY4S2n7K_JQLBeDsvz=Vx1HUq5B<Ki9pyC)W%T+CJTCS7=f?CyXIJksb|i10Galiq$-#x?W@b9*myzn#5RkH&KI%MokDBJqus`Byx3he6X(Izi_OK}v6+HCM4-X!a9XfMa2K+19M2on!NSMlJXtpmvDHH7E0kQ>D!CwIo@oC+*MgO(+9FdigEbZy3L51IT5npphm{rBYGvwFi*TRO%vYFvYUTun__#rad=*CX4sHZn;$X8vaG`pIUBXH#Mz&}&1~cI>Te1L!gDK)@qV^Z^-yd4jPi8{M;J`5xP(5k=(Q-;eWdW1^m`$zNGL<EV9E*g~>QrQ);-yy8SL4}h*RDL<tX~~z<Qa*qVOY)DvZPpY@fTOQPV3iw2ChgRDRRmU!&fO9W-*GU@MjUU(;&Av@`i({F<jl5!p1Qei~e1`#(SfdhLoy0^rU&wah378zT2|6#$dH#!dFqiRlWbUZNxyOZ;NO(qcqkg6@gpEsBH053iK#;v6YL~*gDHT2hX=}!tHbmC8wg-y_B%wRBJ0(il8uIn=+@>y)o2UnUZU=;eQBCI<+gHhB1;TuWmix3a)jotiTSTF3w1e7QlA<JERM)-xy+d(lmEgAW0>=STQQ|Q&n%(vFqW-Y%_G!>(#BD*`1YLsyuw4U2M}cCV4V%Sebx7ujsrpfJQ!oR&W<KFMWIsga$Kl$UmEB_nlObYFu^%m>m*cb}RTbf`ZLxYgm<NYE}DU%@0F^TVh-#DO7dCqH~pi)vD^{iVcuYcV#D7AWdP(#?JKBei%%|d|>RnX|kV8-Cz;=N}3fpH~1q3QEy14_Tp#}u6Y6x>!GzGktK;;Tl$$=15G|A(!^k|=YTi~@()j>GpE}p7VCOb7{N|M6VBai?rd*9n|og13P2b;NB5qDQXAU4JlLQhJ6ycsW{X~R(-(62>SMLlFEpePB}oJb)etI9e~mQY;ZjHpezSa$n{MVam=dtcF>M0$SyU%uK?KE|k`U|K69f{9G9*ty)wWsA;2^8bR<-T;nP9W5XkLt8nRzzG6COY}yd$P8?q^v_)N%F>5VX||T`cIW5+&VHlhjuBh=S`KtW|Yyr?4Qqx$S<ft_go)y}{0B^G2AD!3)oFtPeVVMt%gS)3K*maS(!;n)~2yZjtAgLL7)=`}PUQAThSHA6F~q*No+o5cq5bMj}w>Ia1UKnrNS4pXesh)^ePt;0bwRoR)qfP-D))S930C_Rwif6J}NHw%IjO7B^qbQ1;?|4y`;>2RoAw)`>l`OI%}z9sOo!>uPYB|7o4cqt5)#C?yr9Z^vR{h1}W(bpWo}^9<R{h-CHh1}%Wu&QAP7*G!u0sPB4G(@puZPTZZurm%U=sj{-Ig6p0fYzOzm9|CU(A+2dh-5*4$4oo~PzCY=bG)&yV`a(Nx;<!UYvDYIb2GSPAQ}gtk!y>HGaEh)QanW2M_8%I5q!RxPyn?ZI!!Sc3A358jxSa6BaHrE=bfTN{w3nV>JD#@pzZTZE=bf)I&QgkLp!fogEU|GV60I>IEbVrhXW)+;4Z-X`<+v1z$b$pus7_>5QPH}|N&m*X?eih4@E&Rlbs!I1>vVQo3KPJ3?ELPsyOj#SQ%+{iUaz@Y3#-=V>%_mpG(c*bQ~W^&b$N;ygk7fEOAxvWg_sE(#fq<_nr66#ZE53Vx|U{nJ5@VW2(wwcC>|>q=M@L*=xTRMtA8GK?Mep{%9YL*{kcMa{yqJ9PNJ$j3iFEDoGTQCoIUCp;hcyoteqUr?h5U{DpIXp)TobquhHjy*^ZQXBQz=Nm#BWi-*`|d5MdTI=1eEECc4ZyDM;(a(PH@#NTzdqyfCa?tnfOyyh3vf5VAHxU+sAYKD&7RDa2GUR@IbRwO5Qh;K)?8Q>?RxTfd00q%^r-zWeGc0lC0|0>R8gYp_n!tT|OcA~`xyI5^(0Lf5K#a#+2QpP3{+p4HMlI{{RIhfWU__*JlTbgsf_w%KWJ|N4foQ?W(L{lEskT5E5qUEiSuNS+9&)9_+j2QU-0PpT~a<-2!Z4=l9iMfRs7_<I9`#U9oqM)7Nfz4mOU^Z#<VIWnJrykcT;2Pc|kS+}XtZz@%r3+qBP%Gr6jz0r#C(S*PG0WbK_a;B`QA>?)Hb9K}V18G<}T1~6Acd%5wA<&y=qfny|a>KAKXe*K+Na2lF1nAj}n3!Rs7G|-#&2vh`M7rH5W#g5v($RL-H+RpZU<I5ni(A#yWYuc2+df}=@|Tyb;WebEyaUSnsnVFRnlaTFE!NI}V5Q+)LalQ2=iPP8Io)oh6=(Y#JFVbQxAM>`r@9vMmwPYjAIyeyFn%kzd^u>CzOBQO1ap!e>_oByN+3dB<@VOKW}VYnf&GfE=O^+oyVy+&3HvD4C)UNisTSf5GhC6uGRI4qmJM9th3rxOt}^|g!Bpr9^tz-M)H<<?ih{+au6=@9<SQoKTBSryx72*Sw$P7Yr|ON8g*r})kx$mXsa;bGpc7%j5{VScBTU?|r`m`oHB=LOJ(+7v%_+{p<Hs~!K%I9Ot%;IYVMNNE$$s>RAFx+4og2%0PZ7LgCbjXk(!>pNbGsk&S?xf)UJ(h%+s2r;C5Ur6Y*x`0X|cv{Aux9Y&~6?^lY0(y;A_<~V`+thZWZ7vPAtVGLd)BnY%#AnjZz?NW&&#H+T6xARjWuG4Qd@P<p|JUg;LcAaXPbI;RkX9Ppc`3>7eXEl&>;2Q0K2)L?Es}F_QGm3|Gr|&gl)un4}Ds4v94CtXOa}Y5tcSg_>)h4T3qoAWkHZL&(XN?21;tg$snyzNcxdjVKi==Co%X*NTXKM(~%4EtZnA@xut@u}GoC$XV6e`q8w<=D>!NBCIMSNWET_WE>8|x|K(2#R=K9O;Oz*zjPbpz1DF~pz|KpI>x$AZ@o(HaJUMQ0(vl23X+cL3+-3+k#cEWH#TBAl}jv1>(~{ttVkx$iE^aVYHpy3XJE?TjwJCz;jm>d_qaS)Sb$0wg&B&81<#S&aK}o)Om(SzHS1*_Ck=rYWbGp_ov}KtAoHc@<I3(BRQK>#-LOrPWQkY1L9oKG<&ELmHHIu&jU7W%cB?j7z4^9WSsvGluCiJm>I&!S#l2^%9A&1f=k<`)I=r69*`=;7<M9%-Y_6@abh}FAZuxjxF&g9?diF}|zp<}ZXjNB=T9t#5MP;g9i>fD+z)!@$AM^}8_VqagFLGwjqyAc*=TD_}R-v>5bpcT|Cg`I1O{`QD^VFd~TO9pFf3Ax5Y=(XPJfnU(cx46|S-f2~Tn54@VgQYzL_zfta=XGa#^*i7X}z4b&m*Qix&G;n{{o^If()O%Skx<pK(YNDDw0(WIpEF)hFPtUGv10Gs+wT)*SPd$rIIPdDvxc2YbcL&Wxn;9d3AqvPi}=v7Tju|U9(ZPSVtb=WGHsu+*jK?E$KR+xqrzX`+hRF63=s<T#1N;#~&6m{FFAx)hvW`TZd7>iE}ALDxEje0t-K0I9pY$sYZ4176@$`*x5wWc#grcrzXWEk~e<^NNo<;u&0S;c8`x@?2-H&rAaKg&qyUe7%^-LNq-!#BeB>d0_hmV%apSPb1G=XB%CJ81uBNdr{h9C#xzLoF$yGnrfHM^Zt>qM{P!yVy~cl^rN0aH;rESqpCM8*mN-1k#lL&7L@r>z9k?Rj)4}*?LfI52H0}(i)b7#wSqM&JHq!w?UkF~K<bpBH`yin<BBuh;P7=~}+>%Ultv=l!wciF*$@y1x_AWoy5$!AcGnJa9L}L`!g<`zBvObb_E_f~$$79}L$uNadOi$C9^Z6d{Ix^*jtGkec*HyxH=WfB6wwH#*y0oxnC*-+I?L^QZFg>ek*+R`VKfXSp?8;LH;@6jD<pL#A!Ef7iTE5~eiq~#lvU+Y8OqM7k$jUJX_k*xmJ9}qO*IGUzvlZP0uTFwljk*>fHrXxgnj=^;d-LU<Il{=%z6~6Si1S^2hkT_Jvu0?GVsDs9@&qM(Eu>Z@Nr50_BK;uzlyVeN&K=tMc2g?=+Q!sW`)JOgE*l${cfODex|a;)N0i9o(r%u2j1D}Qx}30Nvy>j1ED~#eOq8jCvvH=Swry$wpKQF&<0!UdVx-P!NL}h-pp$Ia09vDC9ZV@zO_!<RM7?0f1CR}*?YzDJ@ZIC!!|#3gjp_bd+2w8gzwy@KWV5w(y5anL^|aae{8MiYnqDqn?J5I4v!kwRP|W`g!ZGJCZU(hOAv59U1{<H7Y<%vwkc5`e!p{kQMpU}nx{q$U9zEhdWgc=To<NKY_?UY-;WG@3M3j#X9O5t`(JIpfm`=bX5<jzzhMt+M8f4Ra{qtj+g&$mse!Hm1hp6PWzn6x5K6bz-kT?gOR^YWA#91HPF`816qw*o<a}x3rhvo>SSk(SSL+Y|z!XM<!E+vT;&AaK^G-SU48)@sTOYV<e4;f72bxL@)k11T&-NgYK+EkhHD%mz$ojG~POW@Wk7{2+A2ei&0>{gQvu`#;3LoGx6p)|Qubf<ACxYWwY)Ot3j<knwc*f0R%EB|G%c-}1t)Nu10)V3BqVPi}WsIi?d;OzRa*k(S%lvbg^T~OU>dfc!yfV=E&f0H8{@;T>c;mN&w>vr(Ut%u*a9sK5-H*SX)2wo(tlxZQf<Z1UUXVOi7d3c@%De|Ea(M2DOJaYid&zQn<s+fi=ouDZ)0`jgx0oi`dw##R<xJmLq>vO6dIeKX}JF^tm{Al<}zR$Jf#&7<+8-cDJ+bNtqjvgebD`1{;ldIzCpAshJfZj95-rVX$*#U()66ZcVJZw8}oc+p13cg~bhIJtWVFp#{(r#F?%-XvCvaAC6WH`Ks1o#J3wXkWTX?7?q@iS{@no-qWMPtgD?C6Tzsft6n(?Jy3C>j<B9d9oThfC0yc5HE$+C9pxBj2s9tEip|&f6MVfJ8Gq?AER5*`s2zvI1$aU|-(!kVc`LNG#a4E6s75ejzkFrVBW7?{*mh8fEo7<X%YLTbU{87R{<zFL}fCX;b58!U{;5ybXpgB*T!c(t9VOP2ao6N|@&5GTrMz?@dkWEc3Fei&{^|V$RL5;O7i)MNJM#HLX#?Y918{OttIVtREnCUDAQg7~oF$wC-?t7TR{B(TLmrc0h8(GMVX54`4=i$&B;HAu||6QjabWIqsr=9=fbXu+b)e_iETBYz20NBsoZrtdwS#6hqgB!_r33{t_Sbugc)tPvqnasr~2@+)D9AQ85xhrQCgJ_fJThH0@bgYA_PBv{*%z>h`d=Fk0W!DSp-7_A9z;*R`s_(dMtzUQ|Clh78c`m=rM;RO9>;XR*yeF+v4{UqxT4#m5~o@w9b0UkvgeeKr5&-$tAI9QhbZ1CoV1FB7WJ3Iu@1fSyP<xVU~$YsGtFJ;CG;H{v~|;!S6+U?1qv1RmW(!V>G-2lJW$Y&yE{X%2JI8GNSfl7dF~$$$Ho$KU*?kH7QRzx?_i{hxpRue6@}>Q8_FlOMfHubR!OILo3WyH*Hqw~gxfN}^bmVX2G-MHWo|w0>P&)FH=}tjw~p^D`^^T+c0&32v)n#%5fV`Ko7WpOY9z<AwL0e%Bo7D{CoM0&}(o5Nfg?HkCn^4F#}AP3R16z3T8LU*+@qtG8ZS&lQN5r~BB2qafPLcsg3v(){{z?^P6-M)#6^$<&pSGl#szxlu~*0VCTCE`KqXs%$iNHrJm=UM2jw+}!wl*s9=gR-uqgT8mMQ8N?WuO%~_|Cs@i0%gq90+|H#vO}^))LUEYn5L}YL?R<e?buXRJKc|;=H@;xhd}pZyej9YRk3?B83zo!9@}xC*jihYTqO|_@P+CSbpYSnlJE*Xq$W4ts`jXq!xZG2ELE1iyHS>n+J1?25SQD>=D?8Aeb$+#77Kct&Xa&Bk@x;t@BuO^bt?9s>a_LCrl$!zObFO4#B^g;U=;92Tjj%$H++R0^N_JLL`7E=dDd^E%Q`z!M?Xme#4{k6^suvv}LUT7?s#+V3pAJG0CBThEgkh)oZB?k@CCppcBre17rPpq46LKV{E>OheFX;PDctbmi{cifa>si-NEFaKbtX9+*d{HU$)hI*uv^htaVgr3<^Ac1|4$M8m@XKj<L%IB_x}to`YU^sJcqec37H-*>v?ANQV5Y8Dz6WoB_-MB|w!Q$lj>(zvJNfhNlvI~nJD=vpWRj_*N1%3ThAY1Dvhv5L-F-rOow)&Vx{nSmd+<ds!Kaj#l`dlC%TZZup)86d{>rVKJqsYDj2Q6TX;>sZ^5I`xgG*No^qNjGYd6)!bI#_9@+6PAVYMxN%3S=<uer?OO8Rk%&a|D$rU(fi;qu=3v6JTmSCyc)Kd3gn<&@$A&-mdRChMW$jP##$s0>5i3lxIapg&F$z<eycKAc8{e|vr7;ACVjj&WzNSr7Tg+q5NUx|Dt{dN&Bd6+AG_d~nhcV)w$&z5ddd)>YOn<N_6e+5uKR18MJ)ogi>y=trOHOSutM)09aEX|_2+BcxV1LJAMQKszVFj)-JVkLVTOI?6PfZqAWl8{vxdl`k_I40o$U<DB48r|d-*ED|b17KP(%Qn(*y&6@C$0!fr9U>b;8o;^swP+SP2XO4Sw6=aRcXD+ua$k2FPT;3XGLOIH+Tc&E2BznP|dDVin06W`T-nmyTXHLEbXI0o<TbYg%E&{&PsOKHh_R-SZJ@+~bvTT~iao%x6F<DF`W88mf?_Nd-0u*Bjd7zsaC@mP|XZ$k`e&dT<S2nj~^?^q>kJAKqC-=p|ub}j9QNB_bGO@meqq2+t@4R*Ct#J4Ajh(j+*LR<3hQ<AZnzuX|K&x-NF|$w5icIY0vsa$Gx@*re^Ubdu%dijRi^HbdzPWZ54JmLdhUSq+L)#q|xU5`4ajd33+w%7JXyT}?{k;SF=V+ZCoYi&OLz&IzI#)VEQ3*kybI+INEVgO}GzNs1r<&F=Kc49{zFnWmvjo;mi>b@O9w~y~Ju{$0V!kO?)4s$tG%43Bv)RFNUX_xo)K`0WE^#i=m5G{;zzCMJzTq%r>}=ng8VhLM=dx;0Vz_OeZo96Xybo-QV5y`%?e8V+>3GtfCeT1;_neRvb|A4g%-pMv$*iAh^>R@_yRWQLHNL%3ZeMNWYtk%4(N`rz<JvLH>T*WAYVPy4J!?f`eQ%Y;?d>Q^CVKV~<{UfxRqgsen^jCff%~Ca&SmFzh4OjeY3OoD{fzJL?MX;MH`Gf#`0i}OU%F+<vGPB@{%JrQLPD+bd#vR>UqWZDSwS8Ja|M`tQcdoqaW1dccp34i^5B>zXtH(bJ%^$h&X_Yi!17pQvx+O`h3Ly_?ZBDT3YEY@5=Efp2>texzA3g+LI^TO<pje=0)pqgIDOI$hs0P@+=rc;Zzu|uyw-f!buS~av}n<hs|48LKzT=F4k6^6?79Cq52us$G56l}_*g-TgIwb=9q8m}@fB2SrAN3GoZFSNny}=1xyEmPetJ^3|CQ?+GT<U*RemrowWB3=<$5EYMt$R76vp2SY0=7VcI@K0vL{{>jv<r)B0?l$5a%b}F?;{(XR?Gc-tMAKtqzqd`|WlRnn9AML})NU7t1Hbt1>~c;mruL#TMsSH&z%Bf3(Vq?|3%IXn)bhz;qDkb-{*w7fQF}Ldo4Fl`&owt`pV><)l=-vEB}n1Sm&=Z~~B<&cJ0p3hpy;2*_2LLvxJ|Afgt%bagrYph+X7&@<XBC*mw*7xPh{+Ue$mT@3ZPO}}e!SByGa;VO<+GK_R-_NfB^Y3%mYKj*9TeHGLyvq)E*YenQxzgiW7?w+IL;#25akAgun-=ARKiZ{Vo#wt(2XGKy}YLO}?V5#c1w{4FCd6x>ZJ=A;c-c84jFzX?xD#0ED;W0GR8+vVEsg7IP-rk`#JlR-^>y>sr8jf>JEDz$Rd(q;<un;HaSI!S^QS6^`)>7bhT^-7Y!ENZwy0`3Y+gRC_HGJc}Np;i;3IY?KFj+IV@n!h|03sRI7bZ&T+kDdl^#1Uz<&~?QPDuHA`t%1co*xcjh~cnslab1pggD#PL{CppG#Q&02ny;9INcr{E5)f1J?Rds%R&OE_(u6QosgZ6BE1SahhMHpX`g5enF$2s8kwo#C5N&X51X=_e057tv!UC#_?3Wd;)oNMi|JPMDLW2q!}P(tBzW8bCT$%x9lXI2{=tZ@HVJTH>G2CNBP5y0T+YWDlD(9}9mf!CmSiDdRzXJHv~6|>O}{f<_~F}IImXzUSd6cYdeoZez?|+S*Z0Gn55Mv7uRi?V!yi8U=|}H<^pg*N_u=n;_=gX_^YD*9diSIEcTLc9HQ;-R2iT<$ZZSxgAP`94;-vz@^#jQSx3MxYrD@&j@Qm4tPS0fquV}_6a-mza&4qOpAXl36=}Ro>8X>mO`=+|f6<nnq-REsy`D#l!AA|$)JlF9~vE0B-lCzXk1Jc!okh#sxI7ZM05^&137c{6JSI*$Z8?j5~zEgM}r^~`2meO=OlT|&Z-Z<Q8q*A5WvNNZk5OQXwV5@WX<hem-rru^{QO9dx3SKSFaXz^;*#X71>fXXxSKKW`=N<U~C5bM!a7m3>F`-kKIK9t~NK91IG6C*EPqfSx4n)5a421|#*)MuqUAl?A-sx=C0Z>gbTaty|X*pL`ech6or_%9`rQ;(d<ejUQkdG=UE%m0hFGi!K?JsAWe1-^m2nmYM4yYQ618E0D>}Pv?1eOmJk)1W0%no9`Hn}Btg44x28ft(O(lO_jaif%l&L}*j1q(m^=Wjp$(H}hi+0P$;?>mqG{s+JK*LR67JpSfC|Kh=)e*B$3`Sks7fAWXF_sJjs1Hb))2cP`$Uq1fxzy0KcpFjTQcR&5<&mRAmzy0+6zoKs+fB!#!{QbZD^t<mr{-dvR_{{(M-X}l$v&ZlM>!;uPVOhoufAO=QJpS&VKK`@6ef;;|XVCxsyN|#1&!7C`_dosVPapr@Hy{7~&*<%ke@I7Bg6Zhd2S1+UV9KKh-w5^|{@oN8V=eKD4R<$34}Xjo*~9m+7<-RC_`xig;G46^w2m@lvsk=#`750d-aA+7mThkaS4YYVQpO|0j;{yptJmrv(1k8ikdjR#GY^rOlllzHuPtRjh?#*_ps?$FAWBdN2wsfdl+FqFX8qJhJt?T3XA9UGJEbgLbO^(gs%bmB6`QGA{;lm%UHS45KU>bWf>T`wqgqYEtQU55H!C>@cS8ahSJ3_1`2rx_<XYi8Q{}0BJVgaXuz4=_g6mzcoi#I##_bFFMc^9qRkSL5Fu4pfj2@+EfCf>4xSjx)hOZ`l=-;#9;ua`kiu4FCAI-;b!ld^n*=(Ipnex*@9<M+3)(+ZAhWD@Yix=--2g1CllAii)>c%|dL$*ZBiHt1g*y54FT}je6r*riwUj-TdcH{HgNRoOVxk>MnR)forC2Q#Yx5=Ow!J|y<>^@bQvXe{)J-Cm&2M1)*F_)OeQ!*suJtgEPI)98w#cBN6A$HOIU%vN0e)$)F^^1S`?&BYS9~4`EfBfe^{P^4d`|-Daa(_WtM1q6S!=DnTdGz3qKp7u>@LqD?yn6U|6EK-aAN=)kz-60Zd8WPbsWxR)1)Vkt#{}At-%(QNI19z{<T3S}4<wJ4xK_JnKr-Cm?dV=4T$6$6q}>1I5C7oP_rLzhyI;TGkLK9%k-vDgt0yDC|4?NNF=*e27xq@`T=}&2OI8Clqo=Bwg@{fzU{%1Tt;p54*ZgcKqO!g%mwo%eJq7J{+vZOxDB9ZsnGC|%uGT?4Eg0F2<SQcYw$(23tjmOe%AvMh8H*0LyX<S*u)RVH9;b2v5sy>ngGtrBZPLiIbfN(zng_bsH1REx*9k5rBnn0zc)1!SWv`8{te67G19JH?mkBjZpnrc#@m3T&ry-#ONq*u*Sb4L9=YrGRHQJ@N<Z40>)M`?`e4a9q<4)O8ew_@b;P<|}5i+64w{I%Kg}+Og%Pb&DRfe4U&T%9+iN4LB&rr{sL|=Ms6E>o@rdv0yIw{u8gOg2PzH$e;YOp;nCkWGA;l_;PC+w+69bS%6;3{s~n@U(t%i>t`owcS;ArpO2WiqRoAhIjfd~!`s?p3gNOu`dUyk-#SQ$C-wF#-$VAPADu0#Uie(9cPR%o#9J;Yc)Vd!(&oKP;K&Tq~JQgrPa5q+O+qNB-C}txKK?S98JHI}g0G+4XsqE1@C)1$q9MBKEvYb6w=>DrfOMMGz%koQ9g^y}4F~O!t1PFQuy_;oNCQ?Og~0*vTwtk_vKDoK<jm>4ZrDA{BQ}IgQ{YpAgvFs_B(W<luVP&y=Q(XZt<5d?!zkMxQy)jD89!aZJYwK|C$?W7BB)lp2A~=~{r-y)fG1b?G^HD{MseqI3cl?}OAS!IAqjnN(5d+V(<Bu<~R`cL8?grZ0D{%r{N3Xu8j_dqiDKIniyo6z**%bGNCvqu=bE539!W1{V#OT7o%(6Aw)HHO1g;ZVtZEP4U}>`Q{tM^*V8}m%49=sh_On?9We!4Cdlro>jO%9^W^wwLAHD_a1qCywbtP5_C7+GYUQ5aUnPvLKVQNNp2+UO}P~X_Q+F%@A={N(|mvYY{q~iotOQI(j^;^kh<V&-$TBib4ItCef8x#Z`^wAHt|=nZ&+rZFR%^#WXC?5q&#$a`{iRI&U_a77Sg@oy(qsPy<AyR37R5a9?Hv?4e!p^XPE3)0GS^{=bHBNaZ(JY{xYsgNmoHu0?tw+Wz5eim{4>zT<~PR?L?&7s77soq6lpOVIvo0sI}t=XnrUuh5XPA4O(oTp1KE1baRPRCBv1CXwOFwH;<1Hv6pF3VZMVomX>y+S^QNS@212TZ%89%@4XN#`P9rbKBN?FfiNt}sR34~hwLe%2C{EV=}em0s}9W-S;ln1Y~@4tY&c;vd|8#`DB!D5F?E%cr7fF|J-H?r$w;A|M?^dyDlF!%AZh{4b9P4b!VM^dF2k3$=kvarQLK4+fJ|+>8oor7MfW&lBVV4pN;v@;6_&PW{cOW*>y&RS<k*k=tL`>zZ<+H^T9N>pEuUO!ht(YsLx8=Pr)g~>x<c9`=i;84IXJURbcdB%kloId{kOWTMmfYBC}K$#`8)g%?s5f6jNq6<OpMmB42&rg%)l_P9XbK;7G3mdXWK0(D5QzUT7j?c#aaHw_#bbwr8gz<rRs6b#%<pn5njKJ{2(tQ8hwc^)}S^iStO;x<4))g3Ef6XM^j$%t!;$m^r4->q(&*Lj50<^VT;<ByAyP<e)?G?8;Xxw3Y+hBm@X$VC*O^T_KNQT=VGLBjH{T{l%RJNKwE<03RmvwRPHbm8*z7+?y24^<h$@^nkood7Ibb2NHM;;6_CWZVB}apDnS%o-Iz|x2VuQF(@u0`baVV!^`;!StL;d#V+lF$)E$uJeP*kRH^v6ts9RVTQsZ)r1*0d?+qE~VudsWZITz(^&B?GDOwR|)$vi`vLQb<v3Ok6CRSQBs5@ZRQbDyf!!_RR@G`9wLi_nt$NaS_Je4h(+$%s>nY8L`$EI&0<+oE|r7eJXw2avolSdh=gDO4sL41jdW0qz>KN_R&uvymL)DB?74`QUS|KGO|4fguL;i;qbtEj<?+SVGaGNC5;P-7}dkw+)F2DUmZpXAjIV4Q=28xM0q42xSh71Op#S)cQ%5&-;B^F`F=+K`tb1gq<Yq+faGs3@BaNY4t8jrzfXvj}}mrAXuiXT0+R;dm<XS!3<}OUwebI%TvCxdr<5$GAu$%;&&vROBRiUE*0Usn-SOw76PVD$2i8v0bfH&Vj7#V@eBtruR-x3Gb@M89U%Qkl6Jn!BFs4zOPqYXD2$%jlY>MM76-0Rm|~##6%s8XUmg2a3Kk;$&VnLNL=~?|*Uvn=4$sinSsWiUI-ZrvmEPHvgY;8ZwhCcL8tFO7?)B5f*$LI998UlTe`=p7>F|vBxPE`-E4|xaeC6g>hyhk*R@bD|wK=IpB)S8c0IH-4MK{#Uq{sdO&Z5krHo%Y<U>sw{z?M^p=VB7!c=HL#;N?8hWf4l}V%VQ9&T|Ja6~l56YRWvc^xZ43-TX@L<^PDhl+N$lU;H;a6Q$uDC-Hc&`d;)Jw7UrET@0Vzyj_V;l~-<@u{3YqZrZb_{R287R<vG8fQ}(NmP9$KQC>qq<y6i9`OJ(<pi(p=RmIg8sU_S>q9Ybm$|I3+k3KVH^8c6J>(B{hG;v)$x$l=G*F5PYnU44{FGKThK&n(}sY0zzJFijWQg3bMrTviRnrjZ9#S=$ul`rSkTknY1=+;Xcxb%FL#+nwj)hg(RJy-=Ps3vvQ)>fcD&4N{4Okrwkc&)o0+jR-&)qTVc9g1I4myYrH__{OWq<Dmo$c9yXzguTzqj%<Qc<JUWO_y_4tzo#*r8&J+trNI|X-9)WLkC-$Wl==fh$68FGQiY?=9c_}It<sLc&>G))M2tK>eigLhqVL9CwsgHyCxMq*Vua;6e($rUi|XzniBwD56X4Ycp9M8=cA&MiHi-;e2EaHeaiPN!?X1~87g&^K7Y0Z@J?|`Z3wq6w9dyki6rOK&Pl<iJ0hd*tdIM@S$^g-#0~t==z6x?am2AS=w{yMBLoAA#R@ie&m0aqWY5p_OpAQ^hP0{hX~4xnv2N5XO^EpQl`Jy-H8dIEmwcd4$#N_2P_tYbgk`=23hBz*I_5=16`&3*O$gbGZRv|3+EX5)Rlv71m5)HzPzvuI_vkQzZ`G98ETmAF&6i_wLJ}*G=~bQycPr{i!~q5T>vFnB1G0Klmlg!o<vF()kq_QBqKwaLYo}O42a(@vpy0RdNWy$5ltIA$;9UG;F>X-UM35_)sz9dM3f<A!rE>JYeiu8_uo7Z^>_GsYhr*l}wxhx_`7qW)x7RKxbD_myoa~Q;bWxv<cHHUQ#q~URlVc&!4!`ZR?&hMCK{nukSAujzyW)f;O7qB$v>0Xi)<#F2IG5cDGSPJHs_de2)|ZbfFhVt}RvHb{dlMDnmDj@#Ql=Ey%C@1cqa$53YpXlMQs-}t3b-zMQS<pQTOOuxLK;J4%3w5L>1>SBf@k*2MlGEOq!v9Bs8ln%{9GtXfGQvrO)VazVSymguOa`cwWd|I;e}L{yQ@*OUffkHHzzl%P`45RIKtJkJK=3+#g&kks~KG23+{S_9bgd!yUyJaB`9Xw_dIZbV(EeMZe9mt(lZ-|vy#JPvzit(^E?;j4lwWLI8+?SA0*LK1K~KU2wW-ruAx-IND@pj&Ife<{56>qs<XovmiFUh+-N_44MdI4Fez->|C_t#YyoN;aHwZ@ouMc%<hC>w3CPh&#x`<$CK=z>!a=$+H>j#lr#0ax23JLiZfgburGDmQ!JQc32g`m!p_I}ooC%4M0tM%;5K?+13L_H3^SQ!c4og{};RKmEfkSs|ar+XlsafAmB6fyp6+>_QDt4@S7fbU8-`8A&My^+NhEw_e-G(VU*h;QC&-o-Gz?lL}@m2%?OU#8CPawn?XRZZPp`uXN>BRnMme9R-wrJJ0O*D2Z-{3YYAutCTm?H533E%rdiZF$R7(FnLcq|#Sy&J1stzX(JSo!eHc)1|x@}W+OSnEopi{Yj)a%8<uZ!zm_nJ+LTPucq6f2fsHXkWwH+=G*FFQ#p((E$SmDEJen`V%OHh(kBEp-8||tfhk~oP-RY4!-i}gP&6pbsQfNyOiFc_SNKz$?VlzclZz}3}BbmgPWsA557Bv*6U#AX#o4d3)nIb-{mmszh3BfM&r{Lg8svQq?vs5;4kpOxF@Z20W-;^!W2+TU%q?y^#GEDTg6HjyMVe#AO0AJUXGSdi#ZFS0*@a210S3Dvnd5%px9gv)0qBS`d{6Hn~y&Dn`v+aL>N<SxZ4q>Xw;1<;3vUgi8l5gJ@`+s;*<+3<F6U%-A51pCz^Qh`+g@uK*%nGqnJP7$>V9aKx-J=?sj|YbJfk<9zFc03C-Hi^kx<_3(7n(mV1vr_z~37<k5roWg&w;;l!<%g8fGizSl2-4H@bWoa7(lhwtW>hhgPMAN?ePWf@n)QZ)U~g9E_+2K9p8K6>yszzxLPw}Y=p-2EJdevn|yQjDEBP-nu5;k*veAHBzQ9=<ovXQ*H*6bxfz&`HMms2?p&>_5;0s|Aq-Unpe*P^OQ#XLOZ7&JdDANUiGZrB*q8aCkrX5AhynmcLIc`6mEM%XU9_mD{-=d^z72RK&$v{Jvpk?mq3Z`@#D9dcc3TSN{{{N~$YGI0tTNgWEitm+l8AM>(NB4dl10y_k17E}1vEeqdiYMj7)e*L(2hl0#+kWS*SF@oCHfCY`SldH7zU#3lTl-}xQo+K#K!qPs5=@3z>@IbC$Ono>1vRc3Y$xV@X8JhHbI!T6(}JbLhh>HhheEga^rZq`_Y%5mo*?vr`t*4^Lg-Ff--*Y4bHPna*+{26jCykys_5pn2X(U62--@=cJfkQ<0aS%x%-d}C|rhHW(VztwtEXlD)RKxU&wf)$<vZ1Kf8dPQTsj|~lN_U9rGtP~of4p?ZrLL!U$0b{gjC@)2>$C6KcSKt!jd*-rKMfqdH`ACe$>yj&4O$dGnFOhUs}j4Wti}`z#Zg1~70@L2No;_gR=xF-^&-ogR>>KpiwhqUI$Gf)!Wi*f;clpM1Q<8AIb)^+(ac&lLebcXEB#N}_4C9x=S4p&SC0V9>ODR*7n1{wq#>zRGbg@)b5T>?R=JST?NY18L*|@d8gcqmx|@K$=VO!PtFlEnzqmJbMu3#w&@%F|0jceznQi4L=SA+@w>>Y)xLXlpTt`rfbj19&3+AkoUeR3@h3HHG-vUso)Utju1)>IYzSEv8Q)*b>&Yf9S$!Ivf$IMCAQHw(7c0Zb6aL%|YiC|PMaz~(^y=HrVJlku8mv#9|d0%|@otJQz0wZOtVu{y-DC|mL7mk}}rs~K~h0hKXnt^sG!ndoE$*lI-si)Hn)bt_unH6ta;bRyDUmJO}JkQ!%wU3U^Ke<C)?Rl#RXH|!r_Ltf%Ue97~16KFn>hVZvO}bn`7Rb4g&^7D_6)|1K4bP-qAeJV;JL6Thfwrsxbo=}l$bV*sxXQ_dDi3tk7pp$pRh6S?#i^?b*Y3ahTu*hwym;=XDrfi;<bpb@Lshw<%8AL&I{sFV31=RKD+YFVbFH|1M$hMX+-v4~d0OsO$Im#JPMEKM=xkLGp@|>lTR|YC83rd=8Tj&At#>2cXZcq0!Zzr7A>zM~_qsiH!uE{kWslr<T#EeuNSm=IzcQ+XG^>;%6<8Ry>?wDL+RWL-w8u528uC403{h&%XwUguq}^$R>+3=uLwhu<OIb*x4Za++r$RUY;Oxwz76u)=s_sy^Ib_(QgXWX~J)r~Ardc|HSX(LPyW-20jJ2{{C80ZKq5XK(M)7#w4R29a*^JKRA$e)G$i?SyC9v7N0u~NvDl7{n;9ElF&h%P{X$()|Is~}TK)OS3Y~Tl$ky9PuHXCNUk_BL0$skg%4k!{&uXH9&(6)p?q@GSsh9d#fepfM|9!1!E_gO5i9*n47-`b^;vus)i;d7B&t6hr}_H<iSYgcL~p2f-4mBlbB%kq-EqY!BXKO&2k)9@K>=&8kZb1<=HgxBhasDMaZIs*!wSUeL7q=R)m|5y#>`Lm&@XJefXN?yDxI7+h)9nqQfdbD+;B<pckDD;ycZ@EQ*wX!W?F=r4{-Ph;ZG8gTjbU?MCl+QLPyWoEacGj@USm6wGzCSlPEe**)g(7A%a)C}o0cYn35bu^A>PRf|w>Gw6D%gY3=2-L#0XGJj3HyYu_PS@N7#k&M#5UFyV=<@JS)}RpIR8;kMH-6y%@zaF*NJ`8$w?q-S(#fiscRd}wf_rt`U$l',
 'diagnostics_snapshot.py': 'c-qZeTaP2hb$*{;Q63>^_KL&V-FR(*a{<=7BY0z7R=cYR@nq0yHj89yn%#6?W`>#&NT3HJ4*{}BjKDC2NEQS|c8u6jf*?B&c^KLMWAaP#ol{la)!ig#Bnv_Y*k!XWr%s)7>fGuyH#avQq|(o&@I^NDqo^yA(I^UIDTbjxj*~181M%ZWkHsiWrWh2Y{%IuABJPSX4x(Zh#$z$_(>(OKIqPk1Zf=CrS(4_$PsixW<OWX;k|>fv9wu=n^neFR5$7`PilH3&MU)T2AUBhP>_p8Q0=+z(N;AoP?+W_={UnxV$|Rpg<~dBtC;o`SL2oK^AG3Ya^In$3YK@tnPcXtP@fcstpjqVSqa>Z0@7X*vkF>sdem|Uz!bomxY+x?}G#z>bzC)0^;?z%LP%3l#8v=iMI#&<$m&&|IV@9SICH^quIXwoE%d@;A;~*hA?>R+2+Iq!tU7jn?0y)b?=gp@FX_|nDf9^+x<Y!m-8TD6-7zBF$Y$oGjXXKpoDzQJ6i@rF=;DzHhaXMc#_Y*DOP&g8Ck~0~5@+{2qtmC$JczDDuX>yv0J@f4}%qMDI0>s*TrJZCs)}L1Jzzz`?(*a2i%W4=4i3XWMG&*W@SKP5M&^XfI9{R&sr_tHsiv33z8^T#^B51DLnxXkpaihfy`)@sC&eb*>9Mbe8v^0NkSiHeJmsy7zyJ8f|XlR(D2m+FaN$)8(ACDis*@@*T;lJl-n5SuolZZh8)q99@_zpLe)MSW+RJNypcd@5D^)sK3^qvz!TcolEt;Zh|$PNIh4q_cpB#sfLmVjC_(9=0Tmh-+k6vJXV%Q`1SPb?Z|MJhc%3&L>kRX@t4OPmd}FwSy64x||f_GCFU3%26ALSmrco~2wU#R{u-)i{k}5BN+W&9n$Qsa5TNQR9UoQ*SJDp4@SJEXlH?<ajF!#~_Vj7SslK;AO>h>ZkKgQUJ|3Ic*&(c_QPSj*4~qFj#C0CrZYdLl0qmk_0|0)gF&<EHIk;(9V3|i0&5sHF)fcwh7ET4maV<?d=JN=Q+ZUhgCn0Xv&M}F>DQ-BU=Ti<yvR|+!^>;m^ppn6tREehmk*sB#?D5NmBEVCYS#KTM}G;A+S4<Pp&RL48=I~V-Z|keC88$IY^RYF}=F@MYto5)r72=i_{nS)y4BbeEIz9;^*}IYmuQroF?fpq{>-TqyQxpj)z-!H0C7@P2ZtKS1%mu9nR8ZoXTt&E!S|!%v!;ohK4FhT}Ob)T{S6uUK01o<i;}HIt_;s29~X<`h?a@iZqbo(Zft6I2XbX(gesKyejixvK<s@wEaZ>q>xb7+fQXA<FJ_aG~k1HOnOyA%wWkSaneWQkoO(Qu^-G!o@6X1ID_LzZb>~E?gZv4ka?2(k++6|WtU9^)97UZL{AG{MTSc9>frk}tLk0qV8z3)aUd|OaU5>IBLJh-s`HS^AW_RxJIUaoBO`y7$)TtF^tMQ4Et}VYt`J+L8`L8QCd?A@QT)84<=4-hBiHSz&Fi^Z1;d{K&jHBLgpeRaOQNx|JXdEXNT#z$=3x7to~ia;wtM*`%sh)J-R0xZep_41ESSh)5w&2J?Z(cLW4dRduw9Y?=vK`vd%9y^+*bI%L5`uVfl5+Vsz@#Fu^O|s;GIHrJt8hCs5+PX7K^(TNG3ao0lN2v#RJSAg>jf+W9Yz9i8{o07wt{1_aRsXJOKT@EB@5A7O3d$=U(IkYr`%~EIR?VB4WQHv;_#~KJ%c*Js6>aRj*o>iG53a?=~&WhkzPNQrde}2r>{rD<Rwge`wOR0H8=(yPo=I9V}gm##Xf}w*-8r0?bS=M~4|(A6kfe=!-2{u)8u%NwSt=RCf25&*}>0YFJZ@-Fjvlo{3CqmW!T$7ia?({o%J)7oX%hh8kbJ_$(Ij<m$zz`Hr$yljQOv%%K)B#6(!N^zv7+vWMZ-i~ot$Iz4gl?7j#9`)@^b`H>*FzX&w^c-(Uonw9g87$$(vlm?UVM0ViNAY_5-(G|K$I~)kz9z&vJdIBQm&?<qMJB1n)rxOG?;+-%ak_SIXLph4(KpM^vXfD|Ar*IJAB1eJ7r}A@JF=spJ$79)f#d5w|PIrl2Pp{oENY;j_Q@zn}x*y9Fx-i4pT<v*7fWTZJm3GjF6LSh{MWexG5(}4w2gr8rL6Yy{AH><(2DJS5+rE72hgsdXWaXSiQqDR!YMtA?+xKw3?uuKt3@B*C-Yc$g3kO9Q4NE4mS6F9{-2}BrK`IS{F5$--naD=$RW<@&J0@E06?ve&bnS^$(b&|F!;!KMRS2s+8q>#)Oxw-$>98(!R#n#u|3{w9kQ883a1O%?&kOEWUo=d!0<qBgJV;W~kE!-v82zr<0?ncbil-%{lsg6yJ=1Y}<0u()oLg!uZY@-$A+_x{0^SWvj!00{jJiNTwc#XKu?8tE9;Y1#(nDIxN&?VstudroUGbg0a=@YLRTP5PKm(5I>s}M(KYF<R_LJ8{JcgbL;YPpsby2#}`Q>loi8#6ZD5<L&$6dcOQql+kiG3WgSzEw&*9293o@$V28(lU>^f64Pa4sEXF5$B1<~+<*lmL^O7a2WG@_go1Uw=~gQJBxmmiii|cNF<!9v)2m)MK}nzHtN_<b3oUqXSxoxmz>ujTMY_=Prntf9es*&2u*K@80_!Zv~5$U+XZG!%TzxAYga5+@6u~Wlc3o)FGB*cvIdnbariOxz_<j;jrTsc+=9LvuV~d%jMifQxpyjGmrQfRT8b6S(z>u>iW1gV2!_KAi+o2(y`YtJ9J1NxJy9{Lrho?Lk(F21e{fAE(W0Wi`hyk14nKTt}&$rRsNv1h#BNI+*M1(nPaTib!!PQ4oy_-xoZ3UhQ(BU2I{%P?ySv#E{u~p#teP=DK<w|3A{FCOHQVe6n9Z3O4Oc6Clk=U$e#{|K52MX_X-2I9(2It&T5vS9&|HAC4jUgaz5H&owiIWtkY1m?yaq(%?C8an9Wf;#^>c~@uA)=>uIA+;4TU^OO0u0C^eQ2`izD)9f@T#mG*LLeU=Ei<o&XLs<tqgd+G?02T&!}J38CmO2jDwRl-uvKP*?P)U)Hl^kg#P(+f(zyeICoGJA?ree2*9DwA~`=*EmgD}1z;vIZdU)>%zypoDbOF}xqB6AOS&y^&;94*gihoIut=TuiyYGOVV64Y?MNEGas1;Mu5J@jy8A*h+YOFOtC%!?@CC(q_Lk#BD-IW?<`D0JRZrEiThs>9a~xY$e1iBK9q1$3dt*Ho8N9TNZbk=v6~4gDgFz9twl{O21NkXUkrv#;@gkS|UlVjZvM;H(6)BTwJ%FrPgk;n5_lXN-<RX&9|ivdt(J^T&9btiTg+~uhSc3$L!U#n!{-f3JUHrEdjdS9?Ag~^3|kq$d=Ji9H0(s)0!nTYG~P3Er6vG0HAaAq_?Zct4O6fx<+?Y-h4UTq~_g)1zaW90r~nYJ3B=1@=XIKnMYfkIrWG@|7i@jZ~~D>;Qiur@{g0tkAa|epl4|PBoscbL~#@8XA^{kr`zQ9;oDrCU49&JwEvr!0U*RT@w$lNXoq50*r7YCRK92+2jkmtmcOpMTzAU2q2^+>R=LK%Ug}e88N0EE?b-x<nJD;;JK5icpY7_42bt$&t*q-XE!WE~#OP{%h%>!+S4~$Zf$lccoI`U+l(|h#knjm5<qHlgvETg;R6{<WNxoIz-=+VN8^g`)=1I7@ze{)Ng2W?}`Q9d9a-o$)aQ}gQKvcSGcUyH?G<0^ee2%6;GMt}}5Lfo^+?$=z{W^~IRuS$1SFo&q_q(&R9lERBn#gcG!LZ)FMNZd8=L4t~+<s7H0f|)KJnY~;B%k!}+`j!M3k2`^P><~2p#_pC9Ez8Q!{N@%ACmFve;+H5^lU-=$8kTBqkIQcLG6Pd={A_%1>g{|Mp1IQb(;FKz8}v|Co+`_aOfQAv0QFt@YH1=-+9DBll&alEl^+mSrU>yU+iwHy<6lMIlGfP%Sb2w_pkp^oD=<fG7J0}=Z4t(Z~+H%>Of}uMp5<avl<SKyXc7@S_#K<+cQ{fpIa`(Vh7i9|IT{r;zKpicnXW&?)I$C)!F`mafCTL`Q`J=Pgv!BJQt+~*%8JA<}pw_;yApz_+W~JAYbPFPURn<61n_@t_5-L_w!t3N;!9-ZOb-@#DhP8w>v1trA8)nKL;@0<bd+C7(OeqWhj67_m{t%Gb));^B~K|p~>cc`CMa%#)L51p$zT6p`zgG;->|pM<YK_$&(rF;L`w2(ENnj&q5#-Dt_XI?#t)C!VffQV>7wCZzq*kr;Wk=yZgJ4wS7mtENos|ib9?BYxIPR`3?|CWgC^0aph&t<WxjVmTi1q{1biB){)=-kddiP<}(VNkmBED|ItI}fC)c7O96v&PLs=z!?N|Q2j$}fk~6(T*@)DYdfo;^z0grJ^w9oG6?*V>yt(Z&QEtlv?XC{jx=RWM9PwAe>Ai=;e4Wz;x^|o$LHyBP;-R*jM{fP5QR#b?3~)3?I>}dsMMhVf7~Uc35^zVO0|k%|_iN{#)>Rb^KxuQ+C9Q!m4wJ1cbx?HDazH@cnsGo?PfiOSQm^9cknV${Y=2klxcz510MEhJ<I7)(KYRM-8zjFkK2rzaGiW?LObKkaCa=oEkw>P4aCf_ezRmW>WCqA#n^Ky2<T%btVu`wh)-(DM7a*@+UHr3nx7-E6dKUq2!N+kNvdb1<ejaNDLIRGX`!nnU{ET}WTjE`fH|L7q?bouSC_%GDxhvj!(B<j_v>5-@DoohsYnyBCf_P1nEI~02mNa#1dM@<f--Sr#YRyXLZz6+Zwk<8?2fi^xeK0gcK3SAOC#r0x@;we5;cSzaIsjE-F(!Sb&o(Z3u3r2X=`-4VL7puG)mA_Q)#>G6@`vC04YUNYVU;S?JJ#7L1*<Vu_Y%fAIXypo_Uy4xCzNv5S{d>6+oOjXo=)OvfWf5TG5DMVWGp6pYsRdrX36*@Op}-*BE#>;^L&D<UIB!Ds_^-nLQG&%I9m?4Kcp)<MY9}0#R-`NB|20Av<(=gP0<FRtHJ~-q2l5}0#wyoX4*asFaH~~Wf5dMoDnDzC!&il?k!=|wju<<v%4V^ALFL$`0`(-#5)8W#6?WnVTo7Zz1jtjB4RQ~^2V}vZWG|&84<&$8r93su%RihPhp}xMZ6yVfB_mwCXi;iE*Y0kiKhRmX*#%sU<B&FFJSOr{Aaw=XjH~F`2FYLBTjxvU^>>w)6^|2txWYev?|@D26X)~lq9{V(h4Q+k|N^S#JDMhgniCMr?D9k6x^V;3z%+iV}tz^zLeF6?;CSi#1qU=Ab9giL&f9EUt;TE?S!}jw01TWLacrDRhF98WCldN9)-ONRhAGlE6uYp@Dr$}i+`9Bhd$BnA8`G`-{vd$2(ATWn#5GAyYf1TWfz+D3*`gVs<U`|H3isX+Er8_9w<MfJTSV`b`mQx)x&M#p1IYc#Va+`+ps*|{*7JUe$(gOfbn5-r0-mND03#2F+GOT>8LN+-m{bo7AbP|l4Jg=12S{P4LfEaP#r917tK8`TQD-my?i>;*4R={l)H3hteYqwGtYXkadh!bNj+V%j1KeIGY_1}M2vFmk2#ZbzF1h-rrfNu{Ij_am*mtAXF14DeVlG6-qfeBv-z}`4M*G)aqLTlWnM5Eh4dvln{sL*)AeP}x|@G%cZU^<M~C{6Zd%&4+-BPyJ<4sC1T2~ISPgw6$zP?~6D}3_%BYkVHnn|lIkT<>@BKKOy-JrqR>6fs1f`rRRqjyTY&rJPWAEX?tFPUEcJPpko(I6p&dPIUp4(KBEV*^glY{#YUq9$giSo)ds@I)5jr!ykfSPASochaHdy=Nd{fZE6cL@NMXZIChSW1Y<RTVdztAAbJd<_`O=c$3w?KcHuc{w#OCiT4{J?CD(%gW_xY*Tf#Am8xH&25&-XjiDJ^C)joCP^mgo=B7AuzzpafVJ0fYRF<4+?TJrMQdEgm5UvHQ-~;aK5XxtQip?Lh}a8#u%5T@-`(%(k9xfjzvlu=NoE^KG(>B4`mVeAT2oijC`!TeXtUbbf)CK<x^m&l#nr)?xzVix-CMW#B6qRw#(2eLcfBZi6{D8F)X=IDsgr^8x1hz1DA0lt_?cwu*W9d$=5?d~cGS*$>j90{qwdOA12l@B5d}cd85B-IH=7qriCAsDM=RnL0r&-3_`Vdod$#Z`C#kJHIu<M_u}u>mXhLqcXy`8I5(s(N+8Wt5a#h;zwbbg7R|3k8!<h#uC-qOTOgouliK@{%9{f*dZS8wi8?oYj96pGq>cU(@nUV`w)+9m&-SSOy`6wkqfg{`)I9gtwR25?WsbfJ<*PMF1s^jRd6cPiE4moDn;3*+d0qyyyolUavYz2y?z-V|laJCLwje*4rTPjy_wBoF$N?R@~HMxG-x_J7hT~qd#ZBu&6?8RLR#L~%IdrJ^nJJ}{t4|)C0&DQ*S^jdWi+EClaq1>cC5=)9eZm`Ol70iV#E~FrM8D0(HyIZB_lXTu*PFu;c-UzU-^<Av=O2c{4J(c%DZzyntkXW~q*Sbf`vJ#kY-+?K#ugPtHMc=cT0;(@*_5M-I@+*uK;Bl;_qg>CVF?zE(J9?BPYBs8rQM1*`9@V>6iz(i&36qozAl@Vx$R<RUdhpZ>M$JyBuN<?~J!+$?7))^rQ8|y-j8=3)7ozH*wPG2kH+v#vCUDnS)Y_M*;+kJU<XvWF*aBap&ui*TnOTREd8Ewv>aE4iRW-+)9BY>xAL4fMsJ{}wKh{IgIO_x+VO<siuM@=PVN;Tci~M*Lj>|^%W*LbMhpwjC_*1Cc2Mu)d8u@$c?fb7idiEFIlY_@^K6$2sSR?cGAkw-2_QOZd>I2QIXH5mMX2Eg_1z1p0T=+tHOY2;<C{Po1R$WzBUDO*mS3<k0gN{K|s?c`3shF>N8jP@Js2Yqh(sD}>kov{@vg;%{Su*F_K2c>hc=f5WbY!I+iW$9t&D^MVYxUP&xwIHn%*H9+MfTo<{Gx8AnyOS?gSfgC@m|g~^(C;n+N{l5BYflrBc(ELOITU*(Y8EQ-3VTHnW>_X>z6W<pu73)k0vnWWbhjgD5ba_l&O_zh;UKh^*BU;qwvfWW0|gZ4u0SrJbCiwlc#)8*HT1R$(&@KNf4T|aE!d>*gNqPZ&)M&xBfqq&X>fG@M;g;9^qa!98)2d&~Lul)p^vPkq{3k)hOvj#T>t}p)&v4@zU?eF3BgTK)a1I*ZtweE7TJ3;V4OIVrh%pR#Pd1<(n|e$x)-LUgu`P`BIv56!mq@+6wkmQSZ4z5c&J+U6)40iwhbK4=H!lu+RLSc4?iPZdx^wQdyOpwR96aTsce;ZKU{dh_TW~)aKdOOx)!jt#lpA)%ssojvK$G;iQA*zo5|y0MySHWKtfRH!+v-Ys+Sr>)wc8>{RpZTg?mUnl&jQ)^Y8h9=Y~yMP1v66Bk*j_v*#LB1X9wHe%u+`|id3UV~C-`qev9V1}gpi!~|V%&1?l>D^DqTqFCKdpbkZEZ`SJ_*XAHDxmc|*P2POCy-ZDI?mP>q{dM%tHI?MTBxiTuXnk84hE7C>#6)flg+!`RbhMmZ9*E-)9(+`>tQw_Vb8Fz72}*c28DiYc%zcMtR}Q-^lKI6<p9%kwBVN=A<G_Lb$K4YSnqj6f6sIJ`W-}K#>W2v{;%Jo',
 'export_snapshot.py': 'c-oCtOK%%D5WWl0e_+)Exv;c0j3h8r=un_8(4t5U)X7B^2x>WNG1-?%u5H!ufA0**)x%2M?m^;m-ro#A?qo8#e(GB5nPNp-cdGDH_S-MB&VmE`V46GJwq5UjVqXC|X5qdE=b5*v0LM&&JE)-DIdjk{t2|WOd@`AgN2ZQ`?H%qr*llc-+f-(~d;j9vW;`lwTeD7i60sz?!PoI<r`T<kwe2HQ4i#VT<`@K1^Few~R^yR|64O&!Qiw{B_iR2MF?@bx&fDB|mGPBnz=@2h%C#))IQHr6^5XL9{L|UR#npSP(isixK~+Yx-~amaBN-DkxPxfq(dDD@n@~aCcF+iDidLi2g7?0hUGa=ELpxA)%7GpWzPAlqQigfeD(wX2ld-b3j8+73v|^?VJ2?gKM7G)&Su4|sp|pUuWs4p1`Oo&Quc7fbK_WD`0zG)sHVY|rTS%E5n&wJtsgjN0vsv4FJ`Jrm7WCrQ_K*xv%VDHs7yif+;f!*$L^1VI&~}dRN#fex7J$No29fqT<8k0}RTR*9(0LH@WCu?j6u66)&vwa;=Bqig0WKS`3VvUrG9uQreuzbgQe%>bjQy|(8&RAcc10Nn>@#)u8i&gYeqDEdn~nlG(mf7JfzytM#SOWY1}aT>2+C{9cyBBED@~&+nO;0j9E1ft5v*Ld)IM6_1|BQlT6*dFb!FTJuobAb&n-UOsz*t~Myj-S8+G~q!{GPeb(d-Aw9$?>Rqr=3jD-gSj~HdW4H<VDRYS~F6rL?AY5DvWwup3o2VUSZPAD!2^jP@IVHYfVNytt~ZBeOutrfHABesnOIG}dix`Q@g`%w43$8+bm(s|{3$5+%gJ`4w^W>Ak&F#x(6j>r5MveTA+Y@6BOhP*&%aip`M_>@JWtURJB_OZet2a#p@2A;IJLuwRZu9C{oht+>(bBo#`4y%o;l`l3!4zAcht->kG<612D-l!K-5}C$-c!$9;UFymLK|+)nR2T@wI7?oNz21@H>5S~`BGaM%KIBQ`qiB^>yjoWf)4io-PSh^DEoC5Rnx?6wMf@AM=RSY522%+3BK%I-F(o99$3dmL3a=<f9LE1~e%G;AB$BveaE%}f&hsxV@@1f0-PfHHcFLe}y+!hJg)xg?@h(6PK4bjvhVKWIlFvx>6|<*<0z_q0_<bHoEf(_a3bNd{118yytda|w9}M*-A>E|w&bED{XEgFBgJa6?lWW4|e}$qO?~k_vVI%sAi2fPLtO(bJ<MY6+2{If->9-x=E)N43NGKgX84o_qolM^yc(CV;?wM2KO9;x4kEDF$16YHTxU*qAl=rquX)2&BpI=_~Jk+Q<+_>Txr|D_`4iH1eq~n<oCKp*8fc4hH?kd~^wM?;BOVV8h+=6PtjQkO1i96=6=_kk!3FqZ_^dWrE$yzn0M0u#;_y`=1en@GGMQd!^;`00R-60Wt!<3jIe5933N-9Xoz$sGFwI(G`NX2iI@#tR>HBWf',
 'longchau_snapshot.py': 'c-p;PUvC`AaewEhnD8EE_PiWYy3^r8X%DARI-g>lBuhNU0FM~V?#}Mcadu{JW|kC>CSV{>L0)1wFM%K8i(wcEV#Edl1d$N<L59Bv{Ymnxs_veie@p7@<X|s%d%CNutE;N(Uyru8w;yNe+&Nf;YiE+BWt5iAI9sRFFps0c3Dc<)Iz{x=8V}+yLB~8>MmzCx73D>ihGmS=pB^4QbMkCmM*jBp_SOtt04zzONf~Ep;h=Yxm(Bs;mQlW?eonI0x%pn4%CKozMrFK=^cekn=+NiaSsF>dXc@+dzb@m%fM11q5mEo3%mO`5PcZ*FUg__PuvjGVSpSSO{q0MvNPl03<wB-f;TyW^X`ayo^*uN6?YY)zJjtenN+=N4c>(}FBdEcktym>-xg}kam&-&yT9nIVYwHIOpTBtY^ocWa+yq-R!4B=5-gdV>efr|?1zIoMbOBV>*+f6~C)v_JP2Hj6o}HcfeUC4<?ms_#bny6L@Z#a)hX;pGpVJ&igziqch_Jg+g3q)ZZHIXoPm*Z+Se_P>Fo||%V3T8a>%pTJ5AVNt7<}^R;o}E{kmpXLVv@%z+H^vqIA5;HY)M~3Y(|!3bMO@X%;VyO{!Zg?o@PaKKb;=tQCOZ+V-lZ&snGN+k4}m0P9t})_2A(r_dkEER(_V{C*VJK-?>Flq9i=W_d7n#RF)B@rDyNqC+!Ou6Tgmb?cw?Rcl={CkJr<AM6I_zMC+Dg{pVJ^9if9eVUB04qcT4aW?;N%8f0mZN5y(ceGc<=bh-6ckDnfVFF3saUBPiB3p`~?h$P7B=yP$4KX3Erd;Ix6f8OEG59qVZ>2sW9lasI3S;;8O;`t;r1fG`sImMQ4ZB3(@6BofED@qT{HQ47>0q1p|I-X`Xe&kc9G=!98x#PM6KgrIb+#5J?>M(mLaA}q=!zBJX^2+FCsUXr<EYM<dzv|~c`Q!jNaUTUck5)-I!J{vDJzu2QObbpP`Newd<?hiJrSrvlZ!g?C{tk6?6r6#*4%7RK^y0nYr2zaQRe=33vpDsVI7P1#jEEJ@fCV+s(g>O_RA4j)1KmxncbY8&>g1(ZnOg2DjwVoqBj@Ote^bA+Fgfv*kf?6-UrZE(J}Kg~D8m$MR(<?3oDUt$HP}}RF=Oxqu&sW2FKnpP3HV>QB3{f<<$1y!QJl`{Pdv**;Zn$x>4nv^_>I?P36{BRO<w`GxECa2w4fEnIoMlB;D@VKlukV%c!gj6saZ+I0j4MI#bgmDQ<_h8+^@+V6PCLSYVGwoxC3YvC(RjbT9s24q6MwGu>$8uPUoO+MFYl}8OCN*OC%Iq3WR~?HSE`%2w;nr8cX`$*Utkw9*9lTxo3!a4|?UJd%N_BMNUX{Mgfi&QrfI@^el9C1@V-$XnF3PhRIs<b(>lFX`M}-Cv@QQ4)Ebha<3JM0MoGKd8Js$aA>p{EyMFFP1Bx9{Z@l%0ZK~UU9+soI;XYLDIa=e9s%u2<<VY{z*4*ak^o@nt_a1^$MJNu9RyDv{_rq(aR2ar5Nsd2W)(@4@>UiD=PyRiErZ;}<<L1Y$8fNW3y^IdXf()fa78`fgs5106`m(qNK~U9K7B6){?XoXr--b=d4%q2;E}t^vsK|9;|m@<lOLutt)4s2BOj+3&R;aR16oL3-76yEG%wC#CdW8R7a%935Eh9AAR?&(OI=(JY>*pN9Ca8v(|A%6*Mu?P{eU>*VV-BX=N<@&9A!y3+4xJGg^r@21$h#t8IBZ~n9ZGo`aU`OIFFLiw&}Eey!-JYk7gEtoQ%|>WG1b%J<-5kfhS9-j!N2v`Vc<Wm;gnx2Aa$dW`>$>qW|cS0qYp_gnimzO$Y3M9ps>{2)p@C!yyKQeL7iK%%DcdcBr!|dO3+!rQ>}sI_K>eI)~@0NPVq#$(p-o-W0+fPn-NMvIs6%WI61i<eYhX=p^A7d^Pe03NTrq#MNZ*`_m{|(HBc)DyHIYdfWZo-SCLF7~*N<@RJbyx~0z&nVD2cYp83jNxJz{NKZhV<!MD-#M3qN8*B~H(E^7AGf>WuH&7OXF&12B|L)LNj29-P)QIISc-8(SETUPKOuc4zu|%j3AuJ@iWSzrbMG`Mum~u69M{u;j(r7iSR)$Fwk}*1mRp{keRt}vd<d`@?Q2|cb^Jtl!M$XwHgAfWI&0-)9pJ0VGr)9CrmS!<@Qdr%&0~32%IP9A&qbUm}lu;ob9;1kvTANCZn!GgO6z54ihbg_zl@Vo_ljuP_z~qP_i8R*Q!*`E9W~U>Wta6eh`DO6srb!YO1%HdD;1he(%Q7$Qt);10qECjTae4X^np<=nPs7%9Z5C?bN!Z4kVu~*NEKJY6GCPS<-W;Bpz?&w|RqAhN8W#WupM>Ws;<DKku1hfM4oE$V$-rx=noa+~LrGu>On20Dj5R<_*OPK5kFY9O6j&fR<!l^<W8jp{b`-PMC}?e>HhV5GK}fPsZc&Xy=EuIQpMXP(@Ut?6cn9jTHMp$XdEBKbs7gS{!Te#gbjrn>e`BNe@Xfz3oke{0`ft}&qv=7P|B2WH0u<bN5iXsR#nr1{WUBMvMqIe6(S$XX@R3+sB##B#F8drliPLpd+4kb$)LF1cxzyRITch55a}*y0bacv2%VyA@n~<CjkAw_ka+z|a$pCoJ&_ox7-9W@FEcI5$H>H&`tw)B6)aKO@BAi*H9v#=(-*FB;djZu~WLSARg^TVqSlS5-Xq$0DA*p+-kM4yZ`M!T|H-x3U`e@hyMkmgf>jIVu@4N3j1-TVX)eh{?4mhF~op6TG6&~PF#g<LtQWojg{TFy9k+o3p2F*kK!4|Q3;wz8ClQ6-WY%VuTpqDt}W;1{u(%AiqRGI;*o~@2G(i<GQ=}$)z!)?#^=ixXlcGNXs>Wc7n&-pFqt<+96%{9u$rm7myB^Mh7=N;#9G!G}|%&X?)gHm+zc)loifYLk#cZ-?_8I=7nK5um?vUNU*h<_qq!G2!rppRuT#q+g<T6Zb#31-b<)#JpCa*z$78XN@$)E?MTTLv&p<DB@DyTM`qKi7sFv9Ltnt6~^-l*myH&0|(6)6K_;jT&=;5Nn?)>(Z(O3lcJk23bzglll7U^-mEds*Z|%w;^?_JO+uXMSyyzakefR{J}1;jVzCAV#|Ir1GM!78-lLef)|KIHsT-p+^8eRVaq^G?%j6ojl`ohqX*RVf#{ed@mf4hoyawfi#0@ze0AM*g0?VOEkcjIr3uk;d&m9bkN;^~;yTno!{rp9Z`5K<J!mN4!?=X;L8k|-P{>@1mzj%+ER_Z6JBP%?^K}UDterKN7Y*p}w5Bl$0|gSoc^<A7#a#zlTdZ4zj3W{hEki<EBK_GS$|YL3h_u)0Hx(prLCw@RY~;BoVtBA!boJ_&Oa1K^>B1Sm`6lC^i>uc^K_~kAch2vBa`pP}(*=iz|L~i&^X7kAmRNg#D__WVZQK0Bkr-h=70_C_+}JH3()I1-)BdlTvQiz`*Nhx~d1$SK&&Aqtq?Jeq2`pC7rygJo-sYqd)f*T{)}c{BKiau<+;Kd}qWVsSjmOjsY=wA9pcU}9u*%+XUMwIJQ>StnoF$ns*<`X_orAmK3!PD%J72~rIc%Eshs+LPC1+=0o|2hldVCxG|JEwmYw6rWW28H$Z+`9+Bqh!Yy7ym~&iv~2FJj1o!<+-r`FZS|##gU?S1PethBWleuOU}xc=gNDR^4uU_xd@FACao$V61b`xwW^qcXM-_{J5a*$Gzp&+JtY=TmqLrViAHo(iaK#(>ERL=f|gW`XjM^`o__JYZaX{>qrB~4LY256=V=ASp0h(tS;6`S=s5WC4eYU5S!8?+7i}5=a$W&EesRXr@R@b%{XhuNi)uwamtKrY0j^4>a3P4tEJI6Z^mh}{icUO8ES_T*%bbVbBvXo6}uz~kgjN1$f1KXg3pIsS@<|qIPhqsNn$DOW23ef#?->=vSLt6;%Fn2m^ZfS+C{789G0eUz_bX{2S|x_Jt5Ydik|E%KB&Y3bF=-ex3)tRs^}I*Dh9*TJe-u?IGdi^(fnFKo%j}LsRW71i_jRvpeP^dx$APa^P!~3azv$FrGj)U^mEBuKH%Q1Q0P!#M;w8EMk$EAFGPyeG7K0Je@ou^yIM*{d7~<bO0+z^)e+IDx{xs$odu#;D1+S*gT?Yb3~+uCo-U=bO%KkP<N?f^B4)j$DhMQ^xQN0jXw^=PK%`Vcmey0#=X#9JRy1r^Qkc5SfcWiJ&3Khiv|0>B3cHgw-1j6-Ysn=$ZJ~2NI(3V$KuE%`;zq&PtpfJa=CYY;(ipRKl02aVsb)YTbkrQVLaYqr+dS5IP?Y3XlkF=^xJZCOPlGX)B<P7K4!1^RqFCTbNSsnlYaZMOvv#n@dNpMiw=qmL^B{{0)D|IEQ0q;xOC4IKKylHj?D1VH6@ruKyo%Qj?YM26;ylJ9lfblcB_Kihpk4%`<&TF2O})acwJZR0;l5@YlW=Y2-XH?uavaas@V){O#TuMyz8e55!c|rc7iOf)Zy4`j|3G&;O(lLsTaa|Xx~GPQHSx!-I+~h+q>@N#29<<tSRp-2QPTvfc<a2Pj%&yCil_mP&wX)+Y*rw7wet!UR**d_!O%dCSxX~VB?Fo^6n2F=sA=ncEE8XKGc8RQ@_qq@S4v<bF4$vJN`~VvRymS3fK|9?H=IJwOEN{~RSKlzyl$mBb)K~1@qw~lI((3xt+GZ}DZO<Hj5sILuw2`F#(cPhum<>FbE9fvc~rync+j_TYQ0r;O~EBN_`i`<OsKP<{hfHn2Yo@jK5}3*pc~C)lA8^_ur!kaQrn0s)D%(aAs-CQQnyMJ^dW5**cmBOhxL{TgjYHNnR81YSW~&RvI3t|_rA&7`+R12luq8Qa@~5*2X{><;?u*=9y@4D5Rmwe^U3}17w$Dk!}9=b>qu*h9MmFfvtF#X-(+;^rK&O*O*Z9QynlplkbMvAj?ye-l0Ej)2XDjsTH|V*d7*)el>&I1Df_KlE)5lwdTiL6s<yhs5WrqJdB=IC;!P?CL2+8)OJ{Lgr9JMl^3hNWMlO_q5<;$v_jVWW>#&+isn+H=jVXr_1<JCjA2TyKXQ+tf0+?Q+4&u6V=Xe+XQai+tlcq6~m~b6?$<AC#s@B^ZHWI!K7*y9V=;AI(wBD#bw1b`0EPpyJ+)lk`5u@wOy<USUX4b^es;biI{T$w{-lc4Um{h^+R`%-Qh9mig+Fca)b?Uu_QJ<c_5v8=w*Sfx&KL5Y=MArdRLunR%MOI&B22g`bhViBXDe5%jXxiY0rkr;5Ffu46;N)<l2}f!kT%NO#(QNx&Jl?EKr2(pLt3%!MV_75@C|QQi=H6jioRJ6BXB6Fd;gs9Wr*1blck25No!@UAG0I08-OZNK;yoR6cqj_Rqv%stKVSJb3c4ZP@pVdd@m^O9`x@@QsRZ;|PoVQY2_Ck}-otpR95)p?sd1b(c!qUjWSvsV2<~6RgUyy&?JfJHmk;bgTtG)9{babiIuL#cQ=gnRGW$bgMH%Z~R~xc*ahD==;-Bx5&&Ax6XNjp5i0D>|xDOnl)|Fj>O8p~9aHQi}F1w0Vz^8IEFONygEH{^9A<zvV@X?s1luM>4JSscOZDVS%qpQLz6w(S>0d}c7mTHotYN}(JqOQxk=F>;B(gqp}RlI0s<oNfUy&JgAmCrdnuiR&<h8iC+y^h&rMudqC9%>}2ajQg|3<WB3*FtT5$H<y$F!-tufq~+%ao9{Y9x<#4tf`{7qTsmB_ul+^vH(Z?;Wt;We>UNK!++4Oe|Giizb1CR?lGRj_WQ#(sjA!?XD9*5nOgy=U?%o1dha9{G$gu3&4o8mMHtl(s&Q2KT^d4LQ*GERs~0@a62fa-OvMOV^Bw7M)-i8ab#PWY($RHfr_EGcuIgscFr5M!Su}?7IH`GHW0Ip!<1Pl_rEC6@tUz}{;8GzY+`187#VdwMO`SS3iKdqVc9WQrtVQN<Yk_Y%bv;d^p@_mr@g>bRq7O^Kj?6bY+mK~~-w`02n32gn(h=`=c~Ey`c%<UO!>WT;>Z3}#n`c`0j>rYxFp14)ysan!w%Rs~tcSfvN@fZiMXG5)=7b_A$qPp1#i*{dHozy9J*u>2ZQa$WII@4Vp;@fSrT>&EViati1)e9!GF;oM4>~i8HOkV83i5^?&JOEw#eW@>XwH=WqgrQ;0cX7weB5sHsFR(>=jT)p&*zJExb!b7ULppv0c5<Kza2oqKPVRA?K>Zg)NM{*etLY<V!B?g3X3+7C72YXdx}XMkGOPV;IjlpRN1|VUQXjVj5Tktkx$vy()2U&^HzsydsK_L^~3fpNeHw#(r}*qX=YR}-hEeJcN{wJzAK$Bw~QZ0-Hyy9$YXQafMYVP#{*ps65H3U+E+EIi@-SyFzeDUWjv9scNE0YEx&PoP@X0(W~ePFtNG<t<}v#VjW#kc{5v<pg@;I)!<Y;@nnS`TsKA7JC<zpqw^=$ZEW~QGFqcAW#IP}|dJu9qLrsq{*rBQ}LoEO<tP=QW9_4=hZjqi%-%6EBAh#X9%y90tiFwN8JEgL%J(iGDj-<TX0yH!$QCC}!ZU21JDijFR|MS2uQtMS&tI@OBmt~m|^?lh*lauvsZ1xHhRrF3fWPu$Cz`m{Rxs+RN^f=3sO~`ZuFl3<NzT2O-UWO-#Vn&EzpxR-Qhi3_8`#?~Nl#$$GXR>it7D2{=uVQ4CHpxwSBs+u2Y!*A_RyjM4T+hr8b;FjrPD7U-aIq@+q&}^$ep?&AE}|kBQxOBsf9xn&5;aUIDH*!7yj9&L!%~8Pwmb-M^d_^0?+%BIsyTeqhgZnLIg)i6=^h%vFi$G!B7Nr738Ip%&0tJ2^Lg=D0a-1QK(1)vzmRs%W|iu)HnZ3Hh!)uacrN!Pq0>fupPyDUaCU2{c%CMpo<84qszV7+xkg=6o^I>$wJ5I+pDp=%6I^w~g2>31D?gWQ^teR(jN-#1nMwR|aC0z5$syOQ?fHYAfgz-#@9sa@+$8Y+G6OV9LU`n#F2k1|-Hva}z9W6<gDSad%%ZkM@ro9W*0$Bg0G%SzBU)X3dEA&sZ$w6cCK|(40~AnO#)bn)q`+B2t;|%z6xJ5GQKOD(LY`h9$Mq|M5;bK1P46%0AIwnd$y1ZAfQbu*UM7mgxF%NhHP;FoSDRy!x;CZ5l6RLT<XzQNw|1x%QP&G=6?*b4AC0OGZD1V%e%6T^{R8@IbYp?kD8*|PJa&1l7lxqxl`0OYxF(twdb1okO77AIMtRO9=y+3yBdXPJgw7bjB0)yMBlH{lK*fSAYRz@jOP@n%I%wR*IwfnbfG3Av;Nfuyb2*~bDB?5WnQIjD(AI(MQsaP7oVkcx<6H&G;~6?=h=d-j!Z7Kax}%Ks>&W?zRA&psu8=MF`bapvCq>X)Vb>;R?Du`mfw`Ic8Po^WNy2Ytz?bAvA_I=SDfl8n85KQIS0&?nUAXpgWJ}vv-1#Rgs?C{iiL<au^-6)xO;DqIwy9Q{*0_coC{3gom%fpTQ0JjB+iptA-g_}fEJ|#{l6WOE{dpigimb~0w-Wz8BDdCz09^#{5-F;oHQ}yvdr+CNjbUBnm-*=!Y@HUQG+7<<ZCD)G(-=7e7#-~0RY!rK(wyt<E>`OJAeAR*_8jRiow8eJVngK3YJs8y5no%?yJ4&yxfa1$tOF#_9)T*U808Nrn4~G?D-b>99jX%;q+C<Mg;eW$@9%wxH{;y?OFqW#?ClX_bauYye6Y9I&5Qgx2+ETP0%EZ|C);}GK3$!YltSlFlB4P@L1h`NsoMJ`S~*jx<*cmu{!C;OcIQi?)Ld-nV7W_*xYlf+UhulA)UF`4?$|$t>cwi#R1K+($e!OPpLGvV55Z_Y<kT^y4?O*}Pjz)|<4|uBqTu@NB%%%*l}EC;y4fv_oS1qpZ4Y$|`73z5|83<#+#x{d0$`^bUrvVoSRE<}?@^gGWw&9x$4Rl56N*gzIna9X;&HQ*`0eo00B(ZV_0SHoG?jSgjeySRK1tGX+|}F;vSE8X4CdDjqm{Q~$z1Osy{xmGu3dCET|;^|TIDP4H>SB%9)^+-+mSZ!m!T;tZyte(aipC30_q{5N?GmQ!|Pa+ca0!t%2z0g6$l%MkE;)(ju3!-0!gwP)e-r~)w`uK;8b7$fe+H?OnEgNqo7CXH_B!Ly3}?)de2;1!AlyR!WK|x<*r2T?!t?o;2lH-Dw<8IRB{6_hoZ^J3Yq|62SiFwV=i&xNXQzuk#W-oUZ{TcgDf4})f3L-PGSfc@7g3#WGyW_(c`F&dfJTA=|Kq1ZH#K^e*FPCf<|+iaFVLO<g8z-be@WoIIN5&&2|)`3MlC81p$8#2K<J%l3s;KMf-{|wmRO<V6z5ShuaNYJg)hGpXwj<q{cw>v?GRcY3+^<Z_qaE*6YLZ)Cy^ka;`>H(Fxj=c6h_;%CSpLznjGx>GauE+@qN4E=dnIina}LXGg5K4Am@z0MUS0TW?o~>{}QO4LZ_WvRU#nfuedlff8X+-|`lq;>d*>K-5qya`FjO%CNfJB*Pehf@{C&$s%^OariOYzL0@OR9A6)$&}x=>xF7$Z<YuVq|P`K??CJNrwph{{)dj;LQ9k?FVDtFD>Y4nz6e3jo*K>JJbJME`SZt)F6#2zWrsF4Vv)F9cBt>nz)3axuKV>}aN&VUP`gk*1KsGgpMK@wdY-*Q2p)%L%s5yyPRWoiZIqOM?>6bx9;i$aeyM`F3YJn9$96bKcURpcxm@L*Y!H0MM#I=-CtSM|n0uGYAV^X5Zirf#IZ*biuL|F{tH{;vZC^~jr%)G#{OYWka%_!ibe9OsjOhsCYsb(MdBjDubjMW~XIBN4@d}ut(ihlubJw=;8+Z5RuD*IdSZ%{=9W}tV^3)*A+1w%E!9`=|^`Jph_HBh0q7XKoD0CD8LxZyQf;iDkI&L8|Z#rD}EoG*@9qp@r$95J^J*?PHbS!oNIf<f?+Rz$sXeqnFR%g&f<wn?VdCOFZF3h1diW(P?u*c>k&a@;l%_L~?OtpHh$*90~qS%dE+Q71v2JOp*bsgU#M@=8gv^uDuM8+&htx@Y$wMN@YYz2a<CY`@~n+Qe2=xo{*mU-E=U5W`5)_w7R3fwhuZ0TyF)BmET)4RfCJP2kJ?ClWgG@XdkA>(Tz&4i>whm8(3D`S;PP4cFmQ<KS!IaE5%DJM|p5-s72vRnnYCeLruw0m@=nFd?e_$4>C2lO7bHs!{;xsnpZm2Wi^p9qG(VR++gZtR6h;`f~0?J!|`zsJzFCtIDaq~<%F{%N62kr0^MgFjI-DXnA(rF^}Oz_hbhchE%lD?6#n&!L2Cukk1_{|vU{b=G0axRi_|SD$IkBpW$$mAbRv)+<+#=(TH95X`nAU8=|K1@fI`MRCH=HD5%&@Z~GKxsa~5nJ2nomV|TqE!Q@?6rD>YUQ$OxpqsT{Ra>pn3eD~ED{FjJJJmgDv%(5P>orsKeu2(nANzqRR<_!7U0wIQ?b@pfX{&n52G~(;*I1}7km{bY&!`^u-bNqllFwo7j`Og7v3Y1;SMHpdLw&?1^f#cQLj1sl_~!!P4^}0r!`6XNnd?K_+8);IW&lpY6=J-BstK}R5u1@uaB;G4>#`H;U68`_m-Ya92d@x<(<+=nw>2T#^FxHTkM<E8Kkgn1;o2~8hl1+0kyg?FS>5p^=}x?+T`CnRg*<hK3_`|@*RvTLUvcKY;I}0_dWtv2opD-R2jp1=a;B{~^(mk5H{9UK;GvL(UbKUvL3u08S;23xrBj1cos_!UdP2$ku|0vDj1qhH+7AelS7F|18TTz5h-2A;FOio6`kpEAlM>k-2#UUr4r5^%<`VYNM+TxQT&gJwdZ6BG$!7%x5mCCBADM-*g`>5mRlG>}xA@)-Clq<L;WU(^RLYxf*8-Gb5k-PnpR0dcqlRG}Mh6f}fwv)1L>SO)C4W4)9%g-Aod~v;4wzm=OJXW(6(gI@@_q>l?pDU-4Up0*xeJ@N-e!bwqlF8t<VyDSQ*<fhCPnK|s(@4tjV?fp5w4PSP275ENPU6a+%d6wz_T~vEDeYQGx_D#MmyI!?nJsI$~e@*re5btR~*PVa^KOle{cR}?fmh_|LC0X%bmVj<NZx2QMh{j+lf=6NnJy<Dti^cuCiWO>L~>QN4Ez&tWGcYQ!x$XWjao(Uip{c-BYJ;{u{`g<MmM){>2;k97%Td>YKQFJ;0FONcGbt-#fbNsJFb~D#)wf(JPYE_3r17oXOSeUxj$_@0*|F-aw83r{BWhnUL3+<IS}{iygeI>YJ3(s?G_$>I#c?u3rDMbMWE^yMO)S>66F&$~b3&kGpcQg=YNKIp0C~m6Kk*`t4G^AWPY${30!UTOrc%a!s$rYUA+xe*ld2?gXp<Nz8B78uCBlv-|ynp>@Nnz+<_aw5x9;6YvCxPBZmxlNt(LdfDggJHA$@KYse;y9b}%|2%m9@Y&PnhyHTf`d2*)t7d!ae*v-sHKG',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<%aYs3wcws#0n0G~P|ZR;<o2{FQ4U#>rxk8X3Q67ViRv<gMS?}RNP-Q3RVtTjeb;w6yNQ^X-7ID?tLw!qJTd>_KbiAPWC970CAIIkqeDdi`8;{@<ay_V2M^w_Cd*~8*`6=c>A_}lkpz?JG%1Q;nyogIJT0>IFgVY%>mtd6MKYNsc@d=R;9c_LHYv(t5M-M&&DN7;@IF~4>vX${A3S)_o9EdI+E`Dwd7iAxc)l&Sc>+~en=CJb4~rbydYfg-mmibqw#@QDFe%2^=`tyknW<kUAIoKWuJzX2<x<}E<khsi*(3#5NLG_{8E?yUX?lW56v=E-CgU<2<KHq}C0h65&*?^g{|Tn5zgLrTp}#X}djs#Rk-W8v<fg%5ft{PD?fE9p5S-@CjrnPhqa<*o>kIB`n=hfKn1-(X!Mi_Y>3V=aX30FCT+q*AvrJ2c)^eFlaWTx~pJ!Wu6~Gp1Ot#B%mQKrF@4J`p-oJYDIv54vCQq|GEpHBBoaf2G)$vi-`}*ax7f=E|-}VCdFMI|Vu_?o05F*?U7v*X>m~5cq=>&T_{D^P9_*j3tT0Z`9bQJ$;@bK^<{rV~#*bN?hGg)730p+meKQ9iRzaIQ~AytyKTjSf!MGm-k@M>KEy5z}$;vd#MhEjLEUJoW2q{a9uyNUoZ{UKFdWJL*6i4<1!DaN<!$tnr5JP5;nyv(kXJnB=8JONZ)OGSVYv=?8kLoAB#;$%H5uG4Z6g|U3<OXsV}WrE$0=SewTK!ekKa=k?OXWMcRJj8WaPHq6z%Q9JQaQ6V%v$7cTCMrg+vvnc^hh6k$odnMpu(k=V&pcfw0Ze`w6o64sJeb3f%0&Xtc{*99e@<q>cW=H6W*KaQb?~R`oVFr1#s~s+$J6;d4Gi8g5Mk@_M}5(wO4H>82*Gps@LP92u{_I2@xLvMX)JVNJO@lpa=@)Mg8oCkO-woH87K(Qi!wp{o6_t;^SLOq%{W_+bKJ1Ybd{P0Oc4b1c$}`&ay*WTWI4A+K}~EoxRkLeL$D{nk80o_RqOEZVdO3aF8A$SzwfpU-NhF;HlzkHG^7I9K|eSoL{SKiu~3X8C3G8@T)`rZ0tMtUo6N>oEOMLf#}`Q%h3l*wpC_=)NjM0O;(n0MYh#F$k0~rz)DMy+5R0SO=?;oR6R;`Lbx}^%(<Gt+41x%jW)RGw`?BB1sbVptdU0CJ5pGhFzVV=+1V=8QX|GID==HnFa+|!&^DK|TSL-W4MMlIa(Dzw(y$%Q(p$C?0ASmNWd8(W23}AzVF<xabqHLW`VP;>z$Ua6#@jym$z(w$%$hSa*%DP8JE@sE?`F&&|;z-eO5M04dn%|5N<N7YjNqd5EN+cTnId!JXtVpy*qiD>ZYyBt!WwcsQhsgXOpv|CojAVl73Bmx-PhQO65Lu?{#3mySWN2w%#zX|&&naz4#Y@1x@q9UpsOvc#s^bEv&$?U+Y!EH9EE2_LgVz(<ySj4t*IseLZ|s!tdJr_eDMrxuzOK227X3P=>+AvD37ruL!DKz-p*$T`1)7buVYbfkIsO$42R3E{1R@!0doae(LHFe2G)ZQKu4Qn(oxzkI!*j8PGiM1akk_{Zji^{A$tJ?_x*s<aps#~qy4bESfmjtl>fs4ExAV#La-86}1l9ycA#?NxSBfBfdIq2z5LBGD2~nk~h<flqW1X#+8I<yN-6)NPf-gn^V2rfbD1w9xOFd=8ZuUqAP<@8J<;Y;qVpB6%hI#mVpp|eYPS(LKHGBv2L#`4)4{Ts?ep4nz*!T7(Tl&a1)>hnV5GxDMK4Vf%9J}Y38C7Cn$bemGTLh`yvP@-u<JlHMo`moSxyD~Gm*@ny`Y*nW`}@Lh*fJ|*V1S7TIybGiZaTNBKN-QHb+4`52Ij`Bw;-rQt8^V*4Xh3igOfh7WmhO9)vEaxfcu9Ao5urEL88BaFxfSnV6!v_{=OLfcCtijptPTdQtxZP($l&eE5eQRhj(>cv@}5L-smb2R!D5l*`mz1Q<C}O=aV8q#(fR+KD|m_fDjVt%wuW7J0?>In3r;k;^8boxJJJ?`U>#u<X3bKeR*_*3+*lZB>3ggksrwry(^MEVrD3k;qVdkE`dS^&*ng`k;Wf|eLs32z2B4;8|3fN*Q-}N?HmORf><Az=4uo5U2dqgUm0Rljd2is0Kr*)z5yZ%0Jx^dzB9raOKB~S%P(pbs7=R}97uX}>DM+Q<XgUhBG@1}55pMcUnW#Fbdkxqj2^#4Rz4PC@uj{&c?UiZKXo6lI0Y^E8nBpcC^2Xs)%*9~2)O+;$DvwX4MeclurFYfev*NG5EAmK<C6OO3~P5|&$IQKwHkZP6_62@%5!iz;s!FFXG(PUltBoQo9C<y{Tj3=iBFOn1h=*rH?()Fwt6QrfNTX@;#SqdIw`NST<~vvnc|_Y$^`@mPk5G;z^miL1WJx#!-oTVpqGUQGQhAXmOPU1QyP{fZK9brKji{?mVHtu?|^c&_kilQM)Q;PO+?B~0VAH{^bCSH=!$?sWy>wL&Qk|k4D@+2n}MEVo{^M9OlZizf-(!J$`x=Go*~TEMGb~9s#ONc&$JK#Ky%h`P*p`D<1Cn(O%b3b3Rx1RpH_`lz(Tz-w^a3QH3>~6<-YB3;ZIqi_SJC~#?z;Ojd#X$0UJI6h0_?-*HLw1WV0L90U9>t_ELAm5Lh%^;L8E?YI@|tHeP``<&G*+S*V6>oeqzWPJR(Q4DipHv$5cCyG)^fAV`~%if2pUaN;7F<kLk|&F?&X5|wGWO#ZSs?*DLp`p2hd51;-}Jmdz7V6{bQF$t3O<Qz~4zC1bn;TZlr1-zmjg#$@GqM{YlTW0p3(rxtXA!@+Zvg$u>fx(>p?AQF|hri&!mf3bTUrzF*|Lec}2}d^xR-h_bmsldcJsv!|gD;`omlzOcU<|<4{sMGtdWp>OO}5R|mf%$PmI#<)iai3pp8oO4*~5N2Ok@@E1eIz44q}rPF(VXm2ql0g{Ob5uCnUX{hqG+D#p%Rco3Nf5btpZSjm|?9fe}_Q_fZwbz3F`3EGJW|F?;{hv9$#4xN2chd){jmy2?DkwZ%d)Rt3rqu;8gU1+QT50n(sWH4t72>&74P6pF&bL#%Rmh${T3e+oxZgxeGpuK93SO!IUDN=%$*?Mr#HO!{ZYbYjn&8jD$nIYL<mf`C3gsU%Y$2THR_Xqa!25zc)KQN0hNPZrAHa?8{C0f_x6p1(6@mvyU>GT-bM(&xJy7hW{?l`|At<|nk3o;tgs1#hT4=otZ#SBT$aya00x>lwvT(p6XbYVJ}O_Z$!@%aDVWC@B^yPex*t7@)2`9|NsimzAmE1spn)d`9L3IlrtEVFQ#UxL%+E0yQZ~eucC7`oo8}#ukBWdt&SjQ0gkpwn%cN+gOKTZ`NX9(Pph9TM_?mZXyt5(P|Z$ua3Y7XHN-3VbVSf@H{So_0htE;F87hOFqU~y^CluP4W`|PPXL&|AMp%!gI2|3Wf9oPJE;Rf`m3%;Xq+j*pE9JDb^Sz*J=(P8|gbBGHPIE;Fbnpq;QLEy>g|~_5)Ql2f87)8&dYxAm%M2Mn5!!+=k!63LiW}1Oa}v!j?O?1zjd-K_1;Q0v;`+&~nQVs7w(CoI6&8nwd=nLR#4bG@VySQ*enDAM_ZnC`loxA#Dz<in?J??>dGyVIKh*uJ~)SagwWKT~=y#V@Q&2T6E}C>of$WK0Mn`nQqtMnXa>Ik5tmDDcVn9^>j=iTNvgG$a=Plx9b$?Py|Zg_{+YTmQN#(S_SCRhz-=o7eRbnQ=>zf3c@mK0v;&T=D?)^okD3Ud@j=eA<EF%d~RvTRdeR*%ELNOSRLk0H?(9xRWq>3Qdc=xyl13A9|Vth{9etvG|Q&No^5ym9b1;qJgZ+!&^bYDxz(;PinxJ{9St)*uM)A?pj5+lM%ChRcVp1)jY9qn`e{R!-!Fjmng!&TAp;?-7ZdL+om{K|W728C+GwEO(8Oj-8rB3D!b?Mv&y$OE-G)SFO-EAh3z>`n3&*9jPvHO*KZipWP-mo=wk@Yfu$9{%q0OPvTf&nEE^x@A$tpC~=@2naDY=1u*k)k{>P?^&3uvgFhrZ5qkyoHy0(i5<Hv#c2pVFB+udR}_$uk(Un+(YrV{(}A_rdFX8)7HupMWZO7;k&?s2_mo%}@tOiUa0(rH?za4Vyr*te7F&cE~n~%XR8}JhKiB>ph>v?4yHf16o?4JbN8EVqyM-<+4fh=N?H8*NppPOzL;=FG-@WYV0IIe$2M9QPUjOSfi#Y$!i#?H9at^z^yn7z~wd>AQK!j4E?5JRJEE7yG5JPq|sk(RhXlIwc-AO{QwzliimT2*KgPLx%xE#uAVRk#K+ZPq8=4MK#u!f1?&ASX7@D3zH?3z0OJC5Y$Y21SfHen!(-^7g9RB$LjR#|3uF>`Qt(cG(K{?+)4jW1>yju*I#Yuj_0=4LkkmFR?JNZF5nx^Mp>QXTvU=Uttb<r+<EHI@U_fwuB#ICDGiWt5?D-%2G1wdCb6Yg#Fg=ZpH-!FZ-C|KgO;xuTfQ95e%ciZi19wq;%5H;XO5sS4HBw#e=@2YgXM@0~$j0P=HA4SJR_2b!sD~Vn_kdNUCm!um9eZ%AO^I&tU$AByJhGb1S7U=8C|(<utVR+PY<n1n0k0>l<)}F}NgI2rw7X_zI^j9#Q@gcU$Alc1(#9uEim0wApCe+ljN!A7U_Y>T>km2`5O_*`))ZHhC+Ir9feqUV7_ifIO97^>&Q4-<SNt?@LSjN?kw)fMtNQ?NQBo$=WSI_tRiwTl{j|tM_dM#HT^tU`!Xr2@ltV0H^71FN)wi1H9b&d9%gvZimysSa=6seWMyRCe)Vp_6Hdq~_z-Nt)I5`|3k;KDab+Gsw^;B##C)giM)P!Ew5yEpm0hYkS1p%lWU(XO;9GdeS-9G0JZk26LVOj6Sw+i1`dI6<aReKj-{P{tv^>9*VD-bNmJ1Me#O11j|xvIzN$wX~}Qyr4p(NTZL%4j1a{Skw!{U$5Yk0h+Y%VIG(`SO<|-@seiu9Hb#@mA*y`AqH@9<B$RejSfrDGmx-Xom=Ale#rsg$-TvdYieO47+wLdGnEW6S{v4nxbS=c|_o7Tt~iM8xe7VB+B*1bi0p@VH=tHM;s-4P~6WVapz!nz3jD<aUTs)HzwTSBS+Q$T=xD4+4~_XwUBig0|P%&-nE0GeAdS3Y`9MM41nDCs@`+()NG_jM<7@|Iy%9B9^pU#5&!v$%`^@qzEP^@bKOMs9!3p}G=`*N>!p#?s)4h9hZG}<oJx$j4e`ivWTi5LN9_}yttEvI{KG?MD;<Q^?d^q}w*9ly672JfKdBL?NAQfgSZ${X%%2CpLt=n>Hlh#g4n!gpE=me)p4#Eaa>L3x*3zX#d=FKE0G(Q)J*GGa;K?V`QjP#EXYUeZcvP#v_`$8v7HN}#jHu%u&juusUN;X`fgjuk$a`UAUfCR22WhOz_x2WV=8R&uf?8wC!k1d!0jz4P6%PCN@cX8k%6`p5T7T0J=sTkDab@7E`bAvy4JE956M`xFlb5-Jrb4r*LeqAC+(HaMd4sR}13~>!D7_(%+w(Cf-~w=AJWwO<z%A*LcAQ~N?be~h5%nq`;_BegqWo&e-|lQ>LQo>s1EelxA0HvJ+_MjBQ<`kVO|8<fx0mRlik>m6p2kM?6x#UD!6m+}x8K~tEs)3!Z7_q5JjZ!>+f)|Z4TD>FaThu>cUwjm&SK%!<=5o=QLBtkb_0~~#lYKijB^-UWA^Z{&=ffJK(XF2*zhYc*%S#5Y)i`W`L=Rqti|1N+ZZ<i?j=(WoY`S^L45@n_wfse5H~@-Wlt+H+=Ghl<l{%=5FQ(AIr^|{;B<u}02b;d5Uszlzwl6K1a{z9EIwigYkJ8=toN#C*Se!X-0hKKN^uN7TTQ^g2h-=Uvp#s{jxl7=Hi9<8wchCrr5*J^nqU+QM3$jxZkEo0qr=nLfWhfYLs?dn{F2<IaD0bBsL&g`BjfqZ(4nEcA$=?NJ%-kaMaUHf$mR3e!TZzGg3$w@C>Dyd%0*Agc`Qs}wbbFzT6b30{yZ=c0QwBX86%%($sr*7JcSTjgfUo4Rw<i7z^5;TvmD)$0ue6@BO1ig;)ew&vQqSbf+*euIsN4gvV`DJ1BimhUqGTL0y)L_+<d!1t*^m00`WxP8QboOj+t1&AfoVk0qC>Mz@1=y0dv`+v`7(D;q~=(-1fEKfT3-f;e|m#8O)rM9EsWmhi>wx!%*AvL_X5?>nvdtVXH3_)acUP%8k|^L0L7>5<a;jHnJ){p!mqn@W`rg|2|=n-vBTmrgyTfR@<OY{p>X?adWQLT6pFvk)l?i?l4ggt+whmJebkdKpB1u43(V{F;xPwD;)hiH1o&S3Wn_ny#4=xT%W#n!F?A`;9bMzt36Qoir!ygWN#I-WbXpyF8lWl{N0V)KQYD$scbH7d%!hUari)T0LKcnEZ{o!vr$c3smGS8EBihsx9fc6LuU3EmFPg?*U2><s>}gr`Hc-IjWNa%=|q<>!6@PbwmQoIkCxea6h4%*f4?JNd<l!LapVa)Ccu$<nS#iVkVK!3(=+3L#k<+s*Hzb|H?24nWwcsZzFv(uI}z@+cxE1uY9j2$>Mp7^wSn<wY2pNh-rIu^QHQrFwB1+Ek>4O`>l2zd0~olD6!%7wS1%5~efLcOOb2Y(>B7c|`j|wsLSwL`INaEFv8HV95q*V7h*reu#@6Bx)i~<2zT~G{71XvVcU*C`M)@QfI%ucH8C&R5wSmkpc*9Q6h5=qR%24jT5YQ5%FR*d2*$UYAx;U^+>;)lSjg<2lOgXBCfM$pDknF|4(k7FY+qszC`BRMXC{f+WPCgX&94jC8rwxm|?L=s+DG@PjAhJd?Ji8Yt7L#IeNb%Y&I9XG@*{IXdd6K({6QA+<*z@z?E&C-W$gLl!I$d2{fN#$p2N|gC@@t%^IiHpZ5Le*TfhaG@3sKhHQq@t633wtKf0>A#Mu)xj_&O>vAY$z~JLjZhOqVmHnpCtUQrzj4YSiK$m)>sCOV96nLj`16bVIe610XHX1?tkB-AMI!ZAM4GNryVN!AI7;8<3r?qiNaKO1ikadZG4M#vQ0{(5FrcYBa#fd*RXp9rzAv;M&)u*ZWY18qAPVXgh7xBC277PzE+H)JRFy7P3UuA}TfcO!#)wmz@?<UiP|;R9o+|lQy?vO#c>9zt!+&Z8)k5*>rHn$#(Cwu668Z_hVhd>fx8`N|v3>m8(y){O4JHpvp68i1#O;tvZm@UPH)G0$u_ObY%395jg=i*(M@KdvZ<f{bQ}_pQ1pwPby<O3?=Q6&BOIPGYi=bdrU$6;9?^|;$dUD`zz!hK2-N==Df#fU>n&wv0j_WQBuwMJ+iwjoxOIK77Meu{`^55;SPPc*2G+A!;1WJPv?f#O>l2zawXZc^BO8SlFz%!eP&rwRE3Ld#Aa`NBMc0h+T9gs)rqqdDw(YuO*e~utmsD9c(yaZQGnH8PgdIMXQTi>6ojY5aO?vFk{zfsW~9cQmh2t`nS2iBlM)VgLA(ItkU;!H6Hj5Z)oKm_pT28H(x{Axm9d(14@g5Q0fd}x*1+$%Zd0no*G4TbE#7n7Aci`V%5{CAQU?d}R$_=fG)FqovDsze=%$Q4u7qQkAYrC4PBj-=CGSE{<0d5#TlRG5_PBIdX3##7j3&-7f&XnwCP3HJl!Y79igBgWvwHwJR4b*`gYC>;^@PEqQYSP4pv`z~<h5_cRM^H~a!l^^B#j{A{@Xmez%6c2_bufJ6cY-_yqLLi1@=JFnS(iXyMhgZe8EO)5_6)MO8`LFoFmJSRM^|d5jc=A6WC}_aJF6ICS7J1aN+_f8@YlkYTwX|X1@1zjd8F54D$g)Tz3glRbgXW36Sz4cS`F23pW`UrH?S9#RqH1^jx(#>YSF_q{*$Sc`pR&@&t<nCpj($y*qL%GZH7b>m;XWR2#7(<^Cmk^;(O8C*LJH!fR^O7<^PNHCrAnFk{C}JqaXDA|k`oY`fYN;HAP)3WqLmLjw~1VqSFlA5_)GC0Kd@&92mYrqK*$SY^Z=LsEwgg#*c8I=e8-sZCnzEBPh#3qN(YCTjv!ID*Sw(oRRFu)eHL6ZMb#HN|Ah*A9K@-{EF0;P@I(ifNj1pd7k(Ew{zOvYuC%;-ap*1QotWU>}o%>HyvIv}3x4hc7!)HP2&LC1h~elZLU{ijXAlxcOb-dCEV|T8h=df7&=9Y>geb4O~LJCg!*4ZjKoN^*Mm%b{stYX8aVwq$LH9P@TpQ?zeNvhX^;5S&49;01{!?(GIM0k~e5;PG|N9()RAtCPxdIdL%y#bkiB1x4oJ|hQV&jVQdDJy&x~_0{8+_WdQ-*bFekcu$rpl%)Oe^J>7yTXffEYL5o)zwsSCyVf#1OXv=QqKJ-TUUUwJ&hsL=K2Pm<#wS#w5Bjzo8#GGu_X~!-)^|P*v)HKKWv`H`b7y)P5LH2E@gAF*=V2}cOynd;<YL8<p;qoIVa=O<M##QB#)VS)=@mG>bb`0h!NqOrHs_wPLX1a;C{a!#Yr?tqoQVKzf&MAZoHMY^)7uus?nqv%UW&|3l3y|&lW1Gu~;k>FdnrNFnwrsNk+zO!GmcH+~xb0zH<ru2-phaBCPN+WAM3SiSJHf%QRo}8OQ^KKLw&jsyTz*)KmG2(EWQgJkNI0hspZlRNIumOGb#u5Kt`B028FcmDo8CrdJze17tFodtWSRE|3Mu&kt*)JusUnyyL4Zx>D3g;6B@WrvM4FJLT}2b%7-lCgceC3URLBE0>~x)6Olpvm?bS$h%NQzO>v*b_o;X3j&B187c1Z|Z!#=GM@z$`ml<6JuZ|Ld9E^PY3!rH2)QmC|rns4tXNEYw7wo!6ok__b51y-Fxwk6e_S5d_1tv5zhcr+%@z46#{W`<I_tGd3anxSAo!CohT-qV5F>4t%czz^-b3CMQ+BQht1E=-{tCte;?f}&4eTUSi0+r0fe-D#G&Av;x6iE~5AK{7N6Jv0Kms6xO24MM^Ylw|w?%;Dw+a`TZ?cvHlJ2`CbRX$mv3$OnBK1P<cDSNrkKkTy~L>(w0$PXUhlej1M<+m=)LIqcB{l6}SGt}Xpm+6btPK-OqqH<xz1M#tC&W?+&@i9Kz}H{~9FDSPXVZN*TBN6OxQC9WTbx{T<PxXazor^LTD&y7!VmrIG-hHaur!7Spz=X5LKZ#x}OhQ|NdQLy(rA_O`r_|GM4MGXeXuhXHFKhq2+OBRz*Kd@azt&?rKD@u!+v`f^GBUhmr)za5k9nuc8`sXBh+NzLA68vfv>*;KqdKMeXBf9{{{0-$NtFHQq$Rg?464HvL0P&};a@M{zca@eBQMwBI<1ulzUEi`|B&4I_dx>JbNl7%9G)H5&$tI$<EUf=|fVbnIy#tsBnHji#0B&Buq|xpmwTBdyCETN651bwVU;ylv(t<2fiQU4;ElM9|nPn@j>5+Zg4WrJexox_gUwIm)AsPQ@*VRG0IFg!l<35r<sTm{3L2)nbX(wHLp-~Up!fMgq0`)}wp(nuevk0!$kptrnDr{Ky<&4s+LqSUR%ZA5$I29PrVolU%EbNCnjnmM1Uk#60-D`)X(QC>r;ZHfnSaFYxWBcV!smaRjqazGko~6FEFaq+h`su1bD~%r0$GEFdhdR`^_>o^dCb8#_1{!*9iq*3??1q+I+qCZw4Rgx+Y<i7d;mxjMO>!vf?<9t_%xaZt+h~BQ1UB3C7}XUVW`&Zx=q`BMTi54-O?u$DhL0pX)#1or@>;?apjM78xy}z*F8mO~O?MEx7gh9nAe4>>3H>y5q%bZvXWjINJHdMP)qs)o?Q^W7FX&MQ2cQKwiL?6+<hE!DbNzXs0H7=UD9|4R><b!*z~M0;Sl~qnw2KapLUT%CSv<gMjeb}pCKNyu9IikdXJsC<2H6Ue0AN>D2pO#^x|^Gm%WavcE|J_0pb4B>!8Aq9$g<XioAJwws%B4%O49LKIBgR)8f4$~7ioltG`QuxnlXpMndmKnc!PINf&%D@q8Ry($_@pl<lYu#pQCN;?N3}ayW_rE$I6@I9g7i~iCD7if@28`4^R*7`deVu-fqu-k30@ZR1jYVk!cu}F5X(k60Z6anU$+L=@9E`@B&3bUX)&4%0HjZs71|+eK%w1T<HdBJWoAdPtVhMy<ElXbP-=<SD|05Y`R0xMO(ZlpnLQKarmFmg{dRi+O<f1<Xr|3u4Fj}d?HnNM>-GpSnrO1B#)G8l)DrMc3E~U=NDZIsQ1DiFb?ghAuxg_iZ~tW0((RQ9gef+0;`U7Lhw$>Tn|o-lhTB_9MzPCCez*y1w%Ryx{&N=9Y{9p_H+N|QeFI{VKJYh!e}ZJI;fJa@zaqO8Sa;B#GeLN^VeE}x(4ctJ79hLc4v;U6Vzg?8?tou2HXt0cGjF-=Tv*r-g=<_QRgu0Id1n#VSA4ptJS8{UsimJM_GSliN(kJUgz4Gf#sfQTwA<_eK}p!qZ+OR;M|wgYIR%V{+VDqlTqF?F~&YZq%8Y=^2+Xv`xL_MNhzWm)sYHY0^kPXp$OP5mov-y`{Ufa8cZ_hr5wgeZs2G(Sr?NzU^CqE1Lz#~`#T7<9wn;oa!7@7rAH2K6yAd@P=b{j){N{)G;s>kxc6O;;`%WsJhDj<sh6CoQxmUA#hDW0$sP{k#-l<FnN!yX6S~+NlN?mO$iSF7H^aOBRxK;(IvD%c)$$kKXXP5_zd6m*`Oa9$rgN=T;XeMgGT@rB{64MU&qcZ|hV!4?OPbw~t-*hyC$f49yO5~6I%%(38dpbAxw$x|IwgJ5I<|*h--lK;=ehP~xq!!!n(u*wYZEl7tGh#`@Q&SVFN8^>gLJ`UiWPP?GD5M)s2~i(VK1E_e}xxp8rOfg)uFok4DvEA>6#T5agxb&F&5nraA0E%PoN7}mdmi)Z-Rju{N~-8@87?CH~#SQ{fF`I-@JHf3!pd@l-+O?9!8%pK)@Icl&W%OM?n}OMn31OT4|1V0!dUL?QKsqd?EF~_h@)SS2awTb*J42Nez2;A9J5|tnC;KOKy5UfV5A)!TWiZd}QtDkv({`B)GstQhb<8afqhq4s0Rroys)C7hEr&g{it+6bcBsQ^R`im(YL+?o%y5xn5WqB;jX{8MIRC2z-%oo;r|WQEp9FQ0FkEawh3x!8%#g#_%N_(iLP1VqFfOX?hk`)f{}o1QWP)I<R58u2hOZeR7>e*{z7M{%I(m012(%omX=GF1e{+58K}<l;V{=XvE?k7(e)J?Q#J`47<=O0S}3pSdU+GisM)z0-CS{)I6NcKyC@*!vKT#xXud&?r>Mq2XoW1-IBRAu`x`$6s_;88DqI31z{tg37}Lyxu`BQHWD!3)q8q$hPA4K2EXuA1CJRM6qWUbgUsk7ZAzXV9G{))UO(&Gw<cSDF*QaH_&88IC+_jFKhNMf(KUgiC!=QbpsOWP{EgiA=O$G29idf#W`hnQ+Xmrn$pK&6iv>XJnGC&GZz>$>IkkNZbv<=v>yTJ^I5`L44$qg9^<}7xRE@bb4%@>vV!xhTVRC9rntnaW@p`0Ycx`PaRSk7(szg^$b7!qEVGhdBESUI*{BQqby$EL8o4^0{U&`Qe@wfj4+R5eL|Lwou1pnt>|L_0&`(OXtG+6)rum2^5GVta6Z~rxgEU16~xBp$n?iHRUA`rn@!%1=tLu(X7!sf8+mFn=-ULYRROVD4TnI;*t4;|U|u<2V`rlFy=F+GFA95_MiEhXLfwfh{WaLrx*0SX2~An}?L)M_9wA@7onRa1};%zI=wYj=WLiD~)rO&49=>jTBbtaitr@49j1GxxetkVcNB`O-VQTa&^;7h6cjXG%Dfh1`DeCF;hGzdUN@Wvwlms9Y**3a?Vla{UmURjT`T2Z4QSS3NB?1*xiJjK1rTWzxFBK`+e|*I;AL^VGdg&IM_T?FK8w+f=K6pPav-Z{NdEdV5^G(yjZ`KEtkCv1Fd=wc)wBsHWeFiDe_@jbFK>(@RR;J~osi`=L>V$c#;^d?ld0i%5|TjfGaJ1`}d6+%j6>HtH+q)nRp!F6(-0zWBg5jyj#V20j(6^k_JZ`d#yIFe6fYwlTI3H|dq>GHOz;opbe1lF1_9Wd~-zRI{o(mt!;%wo00FnXg!xnQwK@E-gV1R_+I7V;ZD&I@^q=s~Na<3Y+#@EcyY`JusJa%KQ+$IFkqbk<OfBh39575NG7)<YB%$QLZ}2s<ZaY;>~-tj7riNt%@Xr0&OIS7{Ha4^MkL#9cd`cUjEcH6hC3c19#DXqI(=LXNQ@w5&BI!kCyx!lUt!%ckyO@QO#CnhBf<ml_wsnt4pv+Mf^~RA)O3Q<Lt5)Pz9WiQZdMPU=1U}J#Mw#_wSv08WN%o@MM8zulyi%gdgI4#E%1tvV(XdMi(#seL`-JW*&>~q}pbNvUx;bk7@=#8}DSaZr&58?J=`Uw>IFfjx1iccit>rJF~v?g1bsKw2%FXm8o5NZac601J)|GiX&F4_XiF8k~fQjD613_uEOnX*n0Cpt)NP0zCJ!HvA5yD|9P@{TQ_&L=iz1tJFimmQ*`UqU9Wn>ce`GzQvhle^xCAIM)cRtB#6$1H}%?I?wv^yuEo(vOEH|fo7<e}BfjIa%AbK=jo?E;7m!IAy)q?0-wBX0SSNc3j(Ryf%hh&;WutNOW^n&_g$3!5)SB({bfE&xRWT_bRD7t4?UjcTt99*Pt6NV*derx~$dY(Wz5`Jn{xSOXP_C>tu0`T0@)vT%L+J`+rd*-g|KWU}7;|fIEG^QYs+X=_p>EN@!zO;^pMd7&tlDbN%GtAh2{O#1VJ)kol#{PVpk~e%tb=Y|%I30fd)0`%9#G~m73<O9*sGEza;hyze33IQc`)eKGPDGb$Xf}s5y8U=J;<3c436(;&dsH-!sCxTHf)-Ly(T_8k}u2-n25Y&OVZwW*KFnhMe1GE1ZL^nhCcbvsW+d4Ib=N+byLlR+-vt|X7o1*k>$1ZQ7nfbarc9ebB`MByJv64FTQ>AobOsQ60o#Xk$>&9y9LXCMh=&DGl)#y?_T4m)nqV!?b+|Xe)IZ2jrW*_UFrGP-#!~Z|N7Z~ZXdKFw^i1){j}O@$p28Ui)J%vZIFF$udcHzt3uIp2+u;t*R?u2USVwXav6vj7Lw9qC19TE#&Nj0AcxFLKpc^K*^~$8;5-fK>p`jh(cCQ_pm$qg-*D0JyMz&;zJ+`lTf-iS-V&cuzf%{pa_~44Oh7liL{kV}261+0n~LrB9P8K;$Lq2fIG&2D#@pW3oGH|dZ;z~tW-l^`H7=1`?>Ei)_?q5Ua0QT<axhLvVRd<M93RC+79YV3BY6j}4vxxS{(>$ytlrr9Ikfh9o`S3R!AY|b`E=4~%l=Sra$OXUpdoDrAA!OT<l83yh4T@f@vGCq#rX<Ha;&A``Pek^@$y`>d-h<>+oMKne?V~=U4Ij&(U0bJ)%G7|6ST~^<@nukv)y_65lk*n_K_(|ud3<(0q(h;<yhpz6rooqjn=@{4y?3OlwX}#fae!=fKM?t!mfxftZMXFue1R-T5!Kg^Q?hA?x%waqIcCHtxgyQFKw?1B@M<fsm4hzO^HKIMVG{S-Rh<&$G{z<FTVLEk|t^ijahDzyOx^`$beJ;YqTNzO^Wd>t4Vm~+OuhbewS+p&Z;U+Rc&fO=Z<oMmxE8if=%grA(SLt#@CA!Sk4fsD2A2ewOfr3R3-G@Py(Oz0H;!IZnN*07Kz)$?_IKP=KMI2aU7a%gDpr;AR{323$C2t)q)5Y(^Y{o$r?`$2Mz2H7cVq6jBKXKHl0Oep*`|snAy&<26XUc=#4fb0uNuiE{AvLE=QYh`qiZec}97q>1I0+4rfaY>&1S2sfVixV+nxH!Mb+6WwO44jkaF#tsi7108q%=LGa?`Z(conJ^t;xH?KdueEkAld)IXL565}|z`uDue)i2bqY#exo~zMD2^*Y%<sgh8*)d70g^2_7c~Ywj3q)o~Vpt3gi)12S1Xe2r#5+O89joNqS8rd^dyvBH_wPTvc=PRt{jX|A_F46RsDp(S8HK7=BMFZbLvZp=lW_lL0jeVl-h=wx@6Gk&C^<R2g*}=jNJL;<fEJj$3FHAv*&R;8=5Q$n4hs2^5I!)I5QVyNnt<-(#)3RC=rtxDD-3y&M&z@14@h~YZ>Y@Pe=|5c07*7s(ZuA5g~*7Ybxh!10AlCxtcH>{I-U~A3t8ucgyZ1HA5mKg*_vuZL)f<_*{B^UUc##7q!s7rIhD$1eyAAY*4z%KL;x`-2YGSB@TL8;)@`vjmKu-ymR;1jJMP;{V>7jHl1E1^QP9B9khzh3K)|Xv$w^c4T;C6WC>};Xy#JyPts9uS_|ld$y((UzFlR}lhz!%S^AuCKXmre;K9JH0G+$wRdUeGw<qqVzJ>1tHtlo?z>DpbdGdiCYD=Sn(T*YnW<#Vj9P-LiIr96xJwszQhHk&-sZX-X-gM)Q;Aa4&)CWI`yYf5JJP+L74RGOi>G~|)7&d5`W9~#n=a!P!o`dJ5WW$2f;qAz!D2lc}98**8L(AvSVv~=ZI)tWLO*>A&rmG|sT;41GWE9-t*`~FWTW15KF&uYpQRW<1!h6xzOz8f?9&`wDpk*#vFo`azb(&K6Nyvh$@g!%m1bK&iyzi2vos85IS_vrKm;kIMED<+*LTK7Fa^2o%~WK^{nGaA0(FM`@qE(Rysu^#df>_p6;h0Np#BFn<=qL7*$QTSk%TpbjP)xnk=G`s8M$qLjjQ007k0p2GU2b*-Ws}6fV9N-NbqvMmWjt0LvI;#BHs%5NjysafHd%#PM+pelwc?Vj$7$V%z`J<C?chf+@3LyRG*~e;V4nj4X;P;sEfcY8r3~?&CfT`SMpdjG(1$qpG3B1iiry%;&OedSkc}fP{@FD}=57<0}I2v8ZvDI-@06GNxyN0?|y#HzV{^j`jv*%yG9KU_`;cMf+f@m)WbKIh_(PD}ySY3?cZSlja-@gPdf~glUX`w{2GFp!mqbZ3w^t3@v##q#a1s3Ec-nWh8*dR|lzPyH5KOzTcVbg4{7Ekm1-OFbmUXEYC8Grxk^@}&(-!)KSWKe+>o9b`NgYfLONG3CqzXM&Ik><5&ishf~5!LX1QEy~63e_BhcavI+^-^+s>wwsltWpFjiBh~n!NgYme<vkti{<9_>vohBJ81^`Rj5SiC>zZ&a&(PHO2|!Kp$%9}r)ND{Fsnld7HKPfz?!yhy9S~CQk4!(ZxNY<MdsUYub07tv~Q=$;M20^dJ6k_nO-c)`U&??5zQ#gzhhrY)s>T`S#y9CnuQFfj%ss^05H~8R5Mg|R>gS}gXY0v@>TI4z~<Pr2kG_kBPVF%+cJ1UX2+-0(1DrvlS6*}1iiDK%H)V6dU8mwLZ8D9)gcOBYcAZItus{u+^wKBVKXfvP~6QU*>{PDjfFHjuFAqK93DB^)~yV8dwOkr`Z#)?eEc{l!!(*a)jPdC(BrXd7WWxwV+(c+)l=suANO9U!Z3S`nhXpf4}8om3zD@%F2>8Qku8KF^}OkA403yYTZNR^39<dGv7JullXzs;;dFG2Z#>zwPkgzmn|GSv6+y7CfKORM&_=eTplan(4J}LPhV#b(GYfomP=Wk7%H;$$L(3#suG8?RiDyiv>RS32laJcDLZuo0r_mve{MGHZs95<xi$-1{6J1&tjoj<gTGI-3XMwcr$DP?4^?uuW9!5Ukev&hymNag(Bk#Ja3UYANd8#36lc(BAMb(Y*_TPoL+=FHFP(ym?-;X}D-e!?-T;1N64OqzFEd7pLo_j<y=znm14dvSLYGm~iRrHv|Ts_foMSAG+$u@^c%SCm+0FhRVx`!InA(-9(6F2a}0|r$b-d+xC*M5tyhFtoMXmD9YK9i3S`i}4w@e~t^?y0TDEU&wyMH5j}Qb7APk$%l3Ep@R>k`3nEiX?%_39yJJ#-`ul$y-~qVkBTO0OYu}5fV1})Y;xY#QpRhYP0*_huKJP*-RZ$rMvXhZDhOjl<_Z;9a!C&%A>)u__OgfsnY)gCW2^-',
 'pubmed_efetch_snapshot.py': 'c-p-^%aYs1c6)vWEH3~^9L$XTsB22pEm<1htBNhDr15<e&7eRMoPh@fcmZ%m<GEa<vdB8Ce862~o2q0Jdy$1x`49g|&ZE20-Dr@M?Zj2qkU&39pFVxwr+M$*y|?LKHfhmhN$$Pg%)Uw&-q{yvGr#nHcXsaGWX+}b`l}buy>H8`sJ*0Ec#BPy%<|NmCD>|yna;1)@Up3+d-v`Qvemk*8gE`-50+KA^5$ipr}HK&p|^Oqz!~}NB5BelTczri_&)UT-|xyov`be>mPeZ=%hf3Bq^eVFA2($z`^o7qzRT9~b5_bv-_~UzKd+MJQhqLLo@EWs@@yt&c&#<5)X){+pj^psG%0o7R5|pG2rN0mUp6Jcx6U{95Z~A9JZlE+kbs7|snvsbt2|1ZDovxaJYB(_oWo}?slBuF!C;Xtu|eFV@0y?}7il;ec<>+K{B@PqX?2}?tLz<YUsmK<k$TvW_OIMDUb0wZ#f4X|lX(glEb45LdN*aYs2>3wO_CMfEHCF*-mKgdi=@g@-c6h#TY5O3m(|`&*btr;|AUU3s_=ZDo>2eV8+&*D+8=s8AR@uPi|iWz<Qe~4H%D2MuJFsO^0(9#*LIWSS3%0_4%<1_S&9&ky@_}i(YneCrk$!*OZYy&%yL}3CmOb6DCl4qY85rfMO4vl1pXF4ISnUAr_(4eZ_+9VJ&BFl7y|G>PtAc_AfsfxPK!n0e^Y!@_~D?x1+A4J3B&)RDdak{oN*O)cP+vr9dI{(Enm^-{Mz?qyQFF&Mp+mV!tn{BidI7uDc<`C8?J@>G*z~SK1{-}c~vrR;wqg3^#%M#>^Jn_@3=qSc<lMWkXGs9Y1L%&JoQ_w%(EszmI50pdMq<U3ve%_{b%B}w(5k(Knf5qMHe|Z0j$OaS})3m=pE<@4m5shiE@=KVmcF@;gDCp*H}%~<f$A?Kxu+@jxBghxr*IGCbOE5Ovh$w^=tb~f&vcVQ_<MxO)YRobuf}V541+@nd@1iNxBM@2sR$+lIDEXT&9(4Vgs2pv(=P06v1!jS(Nh)9(bD=J}=9w=i;443Hs*AuAR+OW4_1cS%Y<V@4!L3+oyLtWd96Ec5R>wqF=Ou-I<*mh<gC5Q}RshN!Usp6p+z!ljkecGlI&WoF08Pot(fw_km?&7fnu7Nd_qSLy~XOv#Kg9*#4is|6Ac*TxQ;%zWUN@F0-G$|E~?}GM7L7@VB*>|MbH@3A&(t)B$QYmK)SUg&BItUwCJR@c%$cABm?Su-oh+<6|Uo*MgZR#l<GM0R83)5QHaQw23ZKV4Poy*C-wRaA>_fzg>$rYP^f8+^pjkCPN~!)NbMElEXQy)ekL6SuSpMRJ2HF7con7Xo7Df-0*}!ll<Q3F$eYRqO5L>PA~J-Gx2;!*F^?GCAp2|r2H3F7pOsz6UomXfYkW<<<qxsU;OUn*>j&yif@TZxtmg0-ub*~{z&W|YASLoM%t$N4Q|1RHgDLG(Z+v10ZP5$FLW*PRV=}T$MPc%in}l(#BaO$06cZ5168`r-XYV`(E~aSVbCAJnS9{Em_D6y1BC)3ogp<+l6VobJRl%KopET5M>fRb0HFLf4}W(V#%$bLQL7L0220-y?j-)VAuUIKReMWIwYHc?7CcLXmRrd=h{6Gjy2hh5bG#*`N7sHP#43j7kW;Ds@Hm%A9n1H4=8M}vkl=(je@a7&XE>^)P{DmbDsJ$A`$ZYI_z`T<7@cqu_^Yh0LE(=HIMtHb9HT9n&(n32E__4v1Kl0?Di99-LDHoDBTp>SSF4l&!n>9?N=n8*=JD`(3D}xGKwBMbc&>kiw<$7lBw#QRNO2^)Do&SZn!+yJ=4Ao^3L7e2#0YTRDv~q*@dsxw&tIJX*++be4$%fk4K(~+as#Es{KtRyxVLw8`Qv|p<@DeG{6GH&Qe%#?<6pd{g3l6jWr>@5b{?S_hzGE^kN^sG1qUYn+%i>7X5s)5*hFM`z($dEA^@dRboS=Wt2e^hfd)L>t|bTqp86pVfb;Q4>hat=RDzafM}7MPTi<@0>PNOS%aSlft5-Ps@%!Rp<lV_Jw~DyLE&&c|VEy*4iAJcq0c9*UU_~H%<hO9NigK0Y8Hu>N?U05D4e2XPK8ekd&Z?B%2y4VckRg?uhQ&~>992g7aBC?-5{Oo4pcFt7-4yA&H5>w14{orS5J!bMp@9a0#oy;8I9`g4rchQEvqxJQmTj3$(oR-Q(|359we3N-0X#f-4OwvwZx&(#!o05?Lm=w_h70$>FBCh3dcv&4&^PI|*aSrcKL^heK8;)L15Hj(QxmC~6I)~#fy;#S((q95+5wVvwg*Dfg3-u-f(@qvg^Io<;1-O!E5_{I$H#rYhl!&)E7><R>ep>-zWqf4OBrem-ie1D<0(#0YI+4)yH|j+{=+|PyqQwr=Rf`@Fc?(cA9?eizW>((RIs0Z_{SA=NPsonM8cCK#F*kGYP$x5!5_}vynXTNC5q8Co%SP^CP&w&{^0!0)0b~wzj|{X_cnv6iW(htN7oO1&-)PViesLMsEy<=py9TLuPg8+X6Pj)4{DuU1M|B+WuGY!&*1Fk^VhFlygbKgE}LdukB*O1@^41Pe3nI^Ek;Fl8C{gu$8gfC^gHqTn4yW5^NY+MJU{#5>DOPLzojjw!Q(pn4tM(W<OFsFTzpq)h1+rp-!76h?s-`(@JaLu(n14twr=qICzi-Y*CzN5kvL9IqM`TMNrXf)+bk|nGI$@I;8RdFirAdl8fTTfi^(w*OOGc`hR`j+7?~yWt5}Ws8IOr43)L6c&gwQ^Cv~0HJtM<vU`c?Q>oi`1y9=0uKr(^|W3>)ycyQ;Z1v;-6l5)4(!B<cJ6u&ux&42m!EdKn@=Vyq`(@(wofWwC$i$4R_LWoHsX~VS3>>}VlLZ9F-xE1&B%Z(X&_wV!bt(3TELx_$Oeb1($G{XT_kL<U`JM|uqJy&CV4tson8i0QMwx_q+)QvX-zcMNxV1t0l4V>jPieR+J51mj-GU|ZXX@F|P=>b4(U`Ihf!I8;tJ#=aC2z>x3GVO(+ZQ(sW2m;xmMToa9>kJ7=B1S@kEchw~Qw?Z6TWwaP_Xoyt8p3e`og|RQ1HPY}hEqcmA*ewu2Od$()!JZSI935nIrNsGG&bN~6`PMSiYTkWYzJwew-2?I<9ArH0L>CAU8U^>3wjykxr6T8NV!4i#Ct8pNgkk-&pa&x@i4@_eu$8uL1QiM`k*CU)AMVj<rBj4HKJCIf50Y6h>J8`)LwH_dOUKwmBnT?1Ipa9l@=4jhyZS;gLToXesLDKdDqZ8Q3LMaSU9*zGx{muop+M|uSf*wO%c0qVP)z)akC=r7&=#v9Y5N&n-U(mDI8iPc7@sxkLl5v9*^k-CsFiKr~%!3U^-&PI6mFCM2!c~p4CzD5kY|%DwR|y!IX1qGHFUdDfu;Cm5WWD24UouG-g-Bbms;_PLV?u?a>&*jsg5H4ig7WVM-Z|q<8d++^pn7H@vRLf#!D&kAI7<R~jV3XsN=!cy?LVO_4x6_XDLqb@Ulu;|24}U5FO(=&#>=iS7aD!}6E(z{qr5&{z>UdV<qXV4;He@Vp7GaY%sx;nG2Sh94`gKwtwm0LSFB&tJq(U%!a|?(EN~FVpw4ub#g6Qumd@8HTxGtP=c8#dTIe*aH2UIF9M01@v%Z^IQs0@K02w;GqMy$?c%~gket?Q}Bz*w{19CGz_hlI!18Z$>{Wxlar}J2cF4f39beyw@lpV#wWy-H#x#b3|vQ+f%SI*Ds0@C+!y8%lrPBVbP^3TTQz8Rdg!5Rn?7m??%2ZCq=7R7KtSppsa%4nC~pFpP5?Os^DvB>vPp7<gs@{syMY^euddSy5OQO~s(}ZqWOS3gkb&|Ze|&PHJVGjJk5uK=<%(dey_+goLu4NGk?SmlFtieF=)m`2gh!sR(WT{#XAWo$cC#k41NdaMN(%HesL(wp7F;%~T&I3G;`L?*hx_s5@1IQXKS3h*;2&%kft6j)6OcO6XSM(=Ez-c>G|QvkU{gVk5X=dqc~Ym#GG7F(LkR{?lZu$fH?!ckBLVhbH^mj~L;-(m@NNNI=_355egG}Lxp@E|!n>2<!!3MO>-y^kx`74$%?zeR*uzfp0V(l1+0V){4>~XiWd0TZE?_SFjN0EVd<Gg_qe#}KK?X9<LC|=gBVYLfr1YjrgEyN32hcYkH#&}y*y9)uWxgDG=`|!*fDWYdiVb=5BGppL5EgW-1w2AVSHn71n2sTY9=q@4kP~4oPeY9j<;zIEM^BOXXf~juPgy%{3y#95H?Vd>rZqIHKOpKLH)7H0ZM^Ln0Z;@`J37l3Z9hIZ4skfr$(jFPvJG0qLasF^yD(!Xh~p)oGmb;d#X+y&v2Fy$bpZ$M5*(E4TVK~5a0mz(K!OS|gwg&H%WA!;<91**UJX7fbl@0JM);H^GK~dU9u(;ge75Jcfoc-5QnoS%y5sS49udb5@Sjit5aH1U3nqOB(+Vu|A5(17BSq~@mCy1_<<B_idk+4VJOyS5hiE9&4s^;w#Gx$47(zCrBv63rLEoqwKbHoxYCz~l-CquD5hG7RZe;#MvQ>Pvqv}#;kKS^NAasrvU8abt8hq|3GQgwvHKbD=J-tZDW#B(8=7^6!!fC9JSS2}PL>-?-C&&<=QY;1Af>V%uUoh-3_=5edHRT2Q?wGVBGLV{BRgzXljLjm<R`bp7u)cT({I)`Z2n5z>t7B{(m~(`D%-T+9hD716xFw-eA48o8DZ;G<u^^S02B=O3if-n4S*HY(O%M|E+#(!kSzu&qC5u}@Yzz2|5xH<Quwm4YaYWKGBKaSBPsXa7e6uQ{by;NdAaof==PVvf{T_1VV9Sr+AcH)5idqwC2WV(A+m>>e_5Vkt1%VEk1MZaT0ROvzT&&c9@UguDTa!Q$fF{q*mDqvTwwr0e{C}=hx)Gu}Pt$b(23>F(4R>$Yu!TI-Xo01opaWrVCaZOxf?pS%t@!u;ki%g^Psy>d@JJh?Nj{V!hresI02Z)3lJZzBvI|&ofCnpDY*y<!uz;w6AIH$K9-mjBa)K2=ijuBPwsQH-7yU5OZ;nI0dVG%Ak-;LvObuY`IC5Yba`nWAKWM;GyqFRd^|3<BHQ{R>SBWy^a2M7QeGN&ilbXP&#U^_84H-W&5o^t-MEXdf8siGma~^~Gj0YcHcO34}4vo@SdJvL^yd$7@VQWCJHC573>{;6|Q0<YJe9?{g7qnSQ?#ieVZ|h_BdJvc0I^|V#Q-KYmaaLy5tukgzG}Sb}_UNPcKX7n*3zkdRc}NB!XmYUlO29(kWBx1=72snm^zRTOdtkn%iRI?uXAHAWv=S&Bszzv&dY)zC0AOg?$RuSRVQ9VWs@kJTQF!Ocyx;O{(glWSuPFgxs|Q5+nAP*kDZd>P%?D0IUd&J^Gs@X=X+YzlKp0AActSxke<-s%C>)Ix3s!_J`&slqW7^P2Mmi6MtwAQdikjm9&mHEl`{V|+Jq#EqgGKwg^G{~ebiE6qBOi#?JW&8l4Zp+$d$oaBB~r%da0}SLhY0Xeq$$eRMz*1RI_!ucpdn40oA}DhrdjgLdO*i)MC#mKi$7E@GgFChs1~~6Cs~upi9F5FAbuhKZB0a#tjDf@3I#(lQ5u{M%_V5Tga_b~?QDup(uUrNPCu7kbktoXXes(I-94SbdA>3QYvo7<qv<rgNoh<*OX1hhg}*aJfW4Gp(1WEeH)5Bf<=pwM=E2qmH5OvMN1&558q0J8U95LSK+k`s;K!|z3)1Ajd}Lwin)8V7d^kmyCGiFbpZP4C;F}EFwvkhKC_ZU|oD)~vz+$8veIIeZ0K<TAtODvtSX+zL(+KE1+@%t#EU9&}LFK3a<kTCmU}`*hGoOrv;f@nTBytJf98`h8MNjTO@}R^5upx%zV+?}lz>LI+KN-qgf$nfl!tT+6#T)~N>9S7&-lGJwrYpSp>AOq78cbwsw`QQMfn<l2)ZA-LjRisfFl+-_#vxH5weGMVdRj3S(A^_+d0%D=5Wlo}m;wc30{v!?cK|#D#-U0cNSqWZya%a~kOA-J5*AZ>0HNAWL(W&wjWiTHE#0%E8S|f^#Kc(qJ?Ma98VyojtQMklcG6*3oksQ~3Cq2eDlJkri}*>5YPfszEkYHN!r7L1f%FMzCvq$^7%}Gqy}KBq8!Hf`Gl5L+XWZnd&D?fd-*iA}Azcr>dt}X0>l>7dqTrlz%99uc6Afbldhn@CZwN=d8%I$%AQg=K(s2C_1hmYt^%Jj~Q!&~}8M=S6%N}-e_+5LrH-;zCqO4uW`}ivR9PQi1W}a$!6D~HO$7yY)=!xQ4qXR&5@*@j4SgOvUb^*lfxvd53-8CU1nDMg?%zeWi=4r6?Uc4hA@ZrQ1fs>Pd6zCf-x?qCl$0xrTdLKXhY>WI3LRiSq#p}o3r|=yM0Oa`}V|avAFL%hLA5S(Rwa}0vr<MSa^o~J%LOAAQ$^}H_+iyW3-U0CV8d@}%+c`<kUb6JR<Az&*a4=0T{okLTzxHI$N5I8Xk$GkjR3GVd)8=3$?!;>?akqn1rA&0_Lq^w>XZ8($IHTtvY=)j5c!Qkk*EqY3!)b>k?A2cN{h+N7evYMoN68eZS3)2LO`<s7LP{7s*g6mbo2BKmK%-&nP);Sh;9GwCr7IZdq;HGb+>8!WmPzywpVw7E$_9TZ$VNuza#1uRi&rBOZ8eJVAb>1*jOl^s!F+fq&O!`Om=A<Tk5JtUq}asIA@Zm8?eN{`?t$A+X*`8!GC<_e5lQJYW%}sXVs!Oo(&G{|vin_Kg@=FX!xxdTd3-0voQxh$x9ZzAy7<n0#*K-Srd(z7cy`;QweC)mlMSijJwo0L_^8`+x<bJ;KI!3shK?i08n=TAARm76=~#T^X9UL>H3EqWQ{^A)U&6ueX_bZ7XP=CXZ{6WA38!2xF(iVy0fx4%=jct9CG-%oGQB{$0gk}L2Rok&`yu|Ff#$TL6a&cV+ZQfD`%-c|lbz!?ImP2%|3W9MqSt=$!bb8Gv3f|hzFHG9IIP(ZMB}0NQP_J5a|nHc<=O4rQD2muwQD>~@~AJ_l%3jVg^XFZ9b$yBbcezN@4yWdi|rY|&ws0;THt@u!(t;FtZnGFi-P``UHwaZxBGp#Ddtih;=nb1Zc+uL0lJNZLyq3sJ}yDjqdsTOo!>h1TNnQRAjdA>2&J7L6fEfPFS!?$Ue<VO<O9rJZ#kOrUObxa8rKm@QF;=3f#he?SF?tCFe8X`CkaOc{#||~(*eX_>d&V8>}(a}!M?0X@RLyN1aRjKg|~@`lQOq;5*TLiZt6?6$p@}j@vjnx6_Ew$zx|c`j^xeeK}QFwU*vT~SB$z<1)VGOsZx%_N05?5p-&kHMJ0qhG8t4^mSr?>NQqW5XfEjSxKA?da|pk$Pt~RbAcQYO2CGzTkLfBN+X3c>JE`q`<+tX9l^@!C%|uq91!k5BuSq4bwy8OkhpsbYvS568w7hr6y(SpsnR2A>UVp}qjaYRjgDAo4W?L8uumLt>{AP$Uul<g$<B+oD105o{DjC^2AmzAC67pU=KgPURDD>sjS2OvOs>c|qtK=%}AeCokAVmmIB9|-ptmvBo=s?`ML5^~xvaBs2Q#)eTp=l41y=Tj;EyXdY%oi1pTpDI_Of0g5nyp=xY<0P*=BcVvgY2q=s-R9(6%MZ4xI;whqqLwgS1|J8rkoWhpHacKR`4z77Vo<-xw*9q6=Dd<<tdB$&~iHGlE=WE0%Y;E1Y8TC_QD(L6}fcD0NBrxWbWSwWKmg@I1qo?fY@sw@Kj>JCM_;7^efWZFqKW5OgO39<3TS%aFxQ-YGFg{nkZnGCGNc#5rard0zy+GplW^?s>NgiHe*gz($m5unwe#qPT+Kh@<S}uGrY-m9vKup;%=tRSSH~@Qf>~ph;Sw>9n&=#na_BT$HqYh8ir+pdPp2)-1_*HB}k0H(4YprwlW%tT6?8Yd2PH)y_c#Klod=TdO&R21HLpf93#NRU6v){8pASu#%%WB>2TUbct#qPWT4N7(gqKdcp>%OX^4iBb7p2*ESrBlF#4ubnHH0~E6`L)0$uT;zvA)+5!w_nO)ZjXDV~_JU_qp^hr;bn1+*XTsZp;(M+e&42OYuGueumI&!#3ZmpR@TrVb$HOf&bI_#L*Q>4~z9U|}&11Mj8PgkQtn#tPgq8V8uQzmvg??dv<3WmM_6RMPHu=4pLqttug~Z?VER>V5^f-bipMkFF&&3ca9WIr6)O(n3=}rgO`LYN=oJOll1^xPM}l=4B!D3O%cb+!j=egYZfdj<g?PyB(nCaGZlY&k#c8F^~m9t_GyZZz#8SUc!r8E{`bE8?V|FN(oAC&>5sFh#gLbRW)^)U#Ee2=M>bEXEm$~)O#$@Ys$bF2S@O=zK3XS2oCA2Tv$;fsPvGdpP2WhRVOl%wZo%WjF|$f?*$=S*DVR({9sQppowxZIV+dP>k4pOp#NJUUjq7}_td@zG(l=wCmCOjlcB%XLq`TkHLOfI0>adC+}oQe0qc0}4}`nfY7+NPkv|hlPlQJkH-_v3_JOt+m{`79x~d3Q?~p|Y?iJLUT{;wN5gUo^m<Y}|l+&@7WBPvENj3;4I_x!pjH^v#zs}X1hWfVC0ubvW3fiUghtf<i^FX?uwx4B|NjUc3kU^KB@UnGOC|+dtQIF+KlY-SjXpEsk9^yna?DQJBdWoTaC#QoVt2Zg~BB{M=To!udw~xqmfV>G({X9~DVG%?$n8*RQmHz%5nu;pNs=R6MT-BzgG!sIqd>CqV5`1KCV`h0GtIni5v7Qz&ICBrBwevd24f~;BgTKt(oz`4vwy~ro-)N*RpR%r=qjf;yNdtS7K;d8j!<Y`d2{3LUoplIt#ab)`%RuSK@qj2Q4HN$$4A3Z`pz4{W<akgUp#@-Uabmj}Y?-1Bu|uS|S=AdU`#OS!Z9rWV?%=2dSC^&kVXJCKx=eKswEH+Ld99P`f{JRF_w>+NXHpSF>paJB+P#6zos8%id+&K7J8Z_Ryc;u{iQ4FWK5rr!hgSZJGW==K(539_zHYf7)5G0C(9m#X^J}G?TORGn)Hx^bXUaf}yP*_oYVCp#y;u2yZ^D6lao86LqUcuoLKAV*126_~i$B7H*ZN!eRQk2n0@yxU0N|0ijm(YFmw?HJ<gRE8!;=<j*qMe^Nv{nXGHh)a3rSO}W~9GdgsI_xG7*OqbyN$aEI)m%ZlNQS_p0!BHOpPtwwA&K0dF-yr)n{AkUjw^Yya+Zl0Qf~k@7DkYs?HuJT;{#q=T0iaf;2xaFviwYOS8A%UewnD#@oR2zxH(8E6UHD2VM`?SRq|#9RAr4cY0jkpkZ3aI#(hMKU-V&M8?PIPJ;kLu&2rgU){OLIsT}jCWkCFqOIhIdI(@E#}?o*SSy+;_c)p6iuK6w!6~j(g^ps{1prZ8dtD(hF`mr!=eMd3UG)1B_QaKkwKp7YBZF8xHKv#4|)y`ltSj^Eb#C1xzS~8|B>nO_zykX{uP%Ri)%`(J(XLX?Tl}PQiq=ky5`)8MH?qKHU|P?6u9Fl#r)k!BRgZ!gRb1MJ_UR65r?k);d_T%eF=sLA1bKuwk-*`{mz28z2r;AAjOMAqmb@HtyLDO-4LH9Bhg-9n2Mb7;!dsRIdIiB^L!BX)2qFWTU+Gb4NT@`wOLv{%wlEN9Wbv!rv~Tl{F!>^1kiOUjM|4G+$RCd2tTNMNKe!qn!;d!;<GiDkJMCKO-oU=Uf=}4Jb)N4)|4WR+?Fw4t)v~L0<0dWYtP%+)hm#m8@ePN<!CUWaWu_ZD@k&c7xiA%)B681xcXS8Vj$jWJNrtA)S2qSsl}kp_%Ijo(lShqSKR)tpLuE(txFht+3>BdhaY{*dcfgF-P<hjKe4ega3vUi3r6?mZdITFH&N$#tzGoxxan#NM>cRG5?&Yx5BC>yI8&sx`~WIzFoEl@;SfNHs5dTiNMSz?bRQblwu_5giDj(z%lXLN!_%o6$F?rcsB^Ry6S^}f+bYZ61WIE!N8$`m#9mLy8#0eNmwxQXKG`V~I7%ToGa3tc(y<^|JMr#x08~+otP&zWtF9P<)oY7onu@*%<@xTJ$13kNyP74r@0-%1{AzP`Yl)iztx`j08<k1&RHTm_-C(Vbs;&cAO42?s&$=mvvj{Hep*xk?iQe7NUr;bh8{%9rCgfT-y`Xm7<_2juzuA|<^#+0{A>TgpkJ}@U?+;=x&}kQ&>Pciq-Yf2#rZr4P?Q)KfA5Ptgo%FjoaK{UBbe*O1Zz<RptF=gmCOF}Ae*x6{j4G~5rjgx<W>F*VxtGBA)O!3ZUO?CuzvqPUY30dK$UL;>%O~*E023VHZol_q%U>?&_hG~Wr`tV;*eNgf+3U>N_nYg#uE~Co+3vPD+SLj*M%HO{CBIXQpo${Zh9RjSW~UFm^Got37HGoGh>O)2>A}McY91_AW-0JB?BwTTqKdq?P&olVHv*#`=jlZ<zqQ6j!N_-)WqFSJHFChL(zRr`&MZOmnOFsi7GjcEAX&+{6U2_v$f`|MS2BUSX$FM>Py-Wy;;%B&7h{Da@@reFJ*Q-MF1u^7d?Yl;HkEDAy+DstT)OpN8X)9<8qvR+QmgY)p}aNXw<WkZ9kg}Ra&q1sZ2t6%F0R{=_FU50Ep->ylcGT4yM1huQl55RZI=@R+Ks~ez1Bom0Wx7Qz5>LgCxa0<#w%ob=YFmh)EeFJYe6+!dUY)>>B66DQTCKuDXD>e&}wt_wd`JzZUIf{z7uzQq1*Q`SGQV*s+QC``%HA8-HvZTd$gCYI?q<PEAbXpdt=j{m{!M*>u{%iS@OEI>a%hcOTEfBCg%i+9dq=nZk=T3cG{7n1>a6<8iLXZai9S>=+&bHzosz1<kxrG(7wWtV{fQ6SayPS;a)iHO;C0%tz~7@<QFnB8e=yb>(qM*JyG5lri^luzL0hRGP~|^Bbp&hlhu!U2pWK&r>3(>-~GYQRjITa4}Bhs;wNh4U8<B~PwMLW1C5&xsT%v`*&kVig20u$O2!NE-bu&>F6Q}$$TblMR%WR`QnwE*N{=tpC@Ae6v)ym9xe&lYPU`GSL=krGleoaA-^nj;N$UO*jQj~0Vf0a)7s<58b89$^gIyOgbsWa+Z_S7%n$Ovoeu!!&B-^iMi^QY5Xh-~qk6_Y>djLj;yl$iQVlZc(h>nbjo<%Vgf)QSH9ODOi#7CWA=dOfz^QTaR`9X2FMBWHR38W&%+y=gJQzQ5V5sZ+$lfNXIsPq_tBo&0DfWQ2mr5o)x?)IcOs9cfv%BK;%oY^$uL?}Ghot!J;$OF5)N^?pEJceZ2!hWU8$R~H(Dg6emJLAPjz}B)Z#=A0hhK`JC<#gZ0Y1oaX1FLF>j8ihEyLP!9yfZUB4x6g=z?ryF0&VE&9v!2V;p*9Gh3d)H`kpvSthMClZc7zz*Nj0*+mXKobTyJcLE25b&GeL8y*Q%%s^zG{Lo0LUP&2y~7-m+h`V}Bqr_oLKRqIopxuJ9e$gB7af1Z8Up8R<z$OFn-FBM`4(`fkD#^4b1ub2LEgh(K#1M?a-76W_*n@T{#d&sd5LO^%0Ia2zG-&AZs@o$OUP%yS~W)V{`)~E=W?w~?7p_$kGp3~`_51OrY<-X@kvg1Lsi1g>)NjfRhALvP7Q~lbz*=7kk5+*&D(Odd4y%(?0cqeDOeycv|6xa}cYCd`n4DyfkAm9OpOoT`f^_c0=?#jL!zEvl1zr-a_VOJd<l2ina9-MF+0Z8cRAxMxFkOB}WUlNyU9M#!{<Ro%Xsv)){Hf~KTYq`o;Q-#El!(8uqndhR2N8XkF3{}3gqBf9kf!6}=s$SNAuCHcqzOwr<U7);h6wFa+G`^zB=Yxx$JJVnp#3|7aUq4ugZZtExl+;g&ALM}(g%>dDDiK#}0^aU=!`Ol|7$Ix~sppV<pdAvE=(Q801-W+&H9lx4P5(z79Dj7@XN>p$(f&4BQ?8*~0ihq~{uOY0$}J;}Mi6J;GR$%Gf#66f%8ivjh3>7@X-eFzspqueMz=?28xH;#nM~J{',
 'pubmed_resume_snapshot.py': 'c-qBU>u=n+5&yn_g`E2!>7r<J0a_Frs6cw>pe~ZsaGb*xTSJf~u9s)MKvIsKApd)3h7Xau>vgU`(FKxSQA2Vbzj={<_UzfG@kUe)i-x{^h}4!v_0Wx#;@2&UPTw+BZzCnDUXFuMSu_eJnYiQqO~mfGRQTh!_wS;rZ=$Mgqoz_-RBu^*I}W^8GJp2$*@AaCIjW?xKX1P9(f;06a@+E2`>4NQAfFU(?db1v=ocF?bP@1|<c2Ofhet~!xBe8xJEo^hqQ&*bN=Rns!I&Idt`>{)Hy0mIFUq%{KKy)n_RkL=qasS;`>rh&yH{}*#qcGnT9ty;gJ^JsRw~&BwG*d{#iC&w*u6?n{B#uwrY6x_$#Qt}>Q8^xFY_(CZ}<(9DoJUYU{#Z#Q1d(#HgAWjkqL^Y-{we2JSD8JhXyoX#FN^*_)DDXl`1uKye8@_x*nQ`bhU`!g99Mcyp$V7m9Spr-EG4~q90^&DJGVYb<5$Fp7adGx)Ii)iJ!UJmU7x`_&u$e>)+4gTy<kS)90TBSFF5#P)sJo)BH-Xv8`%mctQz?*a&b}wXpFjkSZa~G*dUqgx+VYmy=*+C2P)$w^b`y#`^}uC|;${<A3*Ynv=Fzl3F{F@kCh<J()G}G^GS%ARz!%gN5GZ4Wk8SWh3ck>TAV^UIP8sKJTlJt!N^0m-N6z-A)Z~rj+2<6Ns<bOQX#oqCVn1?Uv&X5#K~y@?NT{2YvJzuJhJx@C)0d-)Kvq>Ia2xmf&`{#LNY)YDRYvjmWAvVE<Rt*{KkNNaFWgO7vdi<LJa%Gm+IyZtQ4K`ITgJJFfBZ>gc}TjK&elWi@gGb6Ql6tOnJE-k#%Vi9TCyc+1Lms4Ltoi6QJvopZTthH*DEQ_J>O(&Kt9Ts6V}Qnxj^7ZW+VhYdAtqQ3%}^^g$*5XZrXS&5FE3=@Py5X#5{aqa>f^aX{{J0|#sYu>X+LjiSNR}dPTNCy%L(Y69}3PxrHZxBPxqy&Oy5=gFvrVxr-z>e09;inP5>iGtO3A~^{0xy=a9n8^Pj1dWE5ZQ=8R^8Y#dfmn*vS=4gEz&TWk!PF~cDc2gSL%2cUmb$g1@RndN{WC{$7NT6*({#)J9M127wKBw8|T1Z2&Y5Nms8MA$g`16q|GI`*qEeOTF{4*H|Ugd9rFg5)V?8cMYFSWr<U~z(Swxu+q_RB<idLnbii6xE7p9E_-DV!Wj##2+S9?<@rFVNASlNNja0}=3GfYH@m0FVH?nN=)=ZXf2dU5niy|JU>y9;fUtjaQZ@awb+x%v@i?zg}`dktO3Am+&A#m|6gMUPWKxcOxLMTKcO<@r3r4mAsez}fuRUAiF0I@@R#|(_TphY=?OW3mouef8S8WN2~m>zo*(<531Qq2KOl)z#nrrs*l>_x9}L(rNNcjg~In7XI{$%heaOo=5H*MO%=MgS#Nb;!f#LjNZ3NYG60-~3iyoPIoq$aY%(?O&ItAD{n}21^@A!Q}L>hi4p-4kIwTuNecANdAQZwmYtT`S3%v70?&xpt#X41D4>V@D21GE{obd=&$zU^tnczT_FzaW*NcQtBR1dj6j~QUtZ1YxCv1+)UYem8yLF6l^6eWaZGr3JUHmP7~fUn2r+oy1+*^~6gKi3rVIuB#DJ3qcr<~4%jm-ZQ4JlMMvU`Xsq}0dI)?wbu7=36{B-f|plq;fOr<ON04-1DIRVD#kltU-rBTQSzO^b&S34MZgg2b?q`tN6x_$yxkh21lIuPjCWup?rgqwD_PU0U;7F&6XAw@A4&bt8s5&{FjfdB3r=ZJdWoS*qB#tH<(>hkQDgEAQ)JcgJ#W`Y+K2g1CMgFpnIXV%|9hKs>Kv8jC!Lq5e;sj-C0QB4xbUcG#|OJNj?+am>jf2UDVBL+c$U@AVTJ!}~yKwCLtmC^=r%DGjsn@N0<!jTc1$kwB3Fd|^qvZ6w^>vRdLC0p$VLRj8%ZGCSS<VU^Dtp4KsJpjN8au5luowu`F!J~DvedmhPnS`sB?a|QS_qrZ3xWw#1K6r>hFWawe2*BFt%Ir#nb=40&6s>LX@(~aH|GB2)?T<L<lb-pAuNBVMb_@Wc0zpuC`t(wO*6ChCcGG-#RQ-dmfLn^dhR%$T0d7hA8Z;Tjip)=dSs1H#5OWkYPNmD8xOYgEj`(@;)A8_qjZ@h%j1a{m5TSS48qh;Yjx$dvc>a2N@ez8;C+a7fADOAxc)7DIUz&|e!_;Qr9NXJSs3~?0IH!6Wv&qhYVtPEnl4j>fu6FT}D8qJp?le|@FOtPB0>>+7mn?=_7JJN)_{B^RNM)_%ErruOT*E$i28LME-K?c?wOVT1`$?Z5>W;W)pG$1qGCW4u@S809F^j-W!32#yOLG14KL(H!T^eeTrKZLbydtNBO>qu@jEmQ>Dm?4sA=^DKfsv&_+fu?(7@SR)CjFkd2s>s~2{2{hV^({6Df?=a+X2pg9%9`g;z1Hu57gf18L;&@$x?Hxq3xf|7(rLNg%220r>#}oa7i?AaHtb(mDAxHG~+AZH+EyNCDLcF5KoQ==_j+v&sC3{7;AzVT6R;_57=Z;&xFSh`{KWjCdsVD9)hH!5AL+SGmDG#h-#1qb~Bp!UBkmuzA=~Kurvgo70ieZF8+KvEAR(q?J+o%v(4{Jq+^g*YXDz8xtDJeL%b%=ZBKGSt(lYeP<S}&6%%3_v5K<#)_Rgr!#9)%l&!DOJ%ak)K&5cI!=d~gAHXj5ReTlZIM7A_u++B5Za51>nb&y|HHSpe@R$Uhe?e-g`Wt-Pv-^<(S{vYGXdBX0LHKoi;4q5roh|H~Qh6s;T6qD<EMu^wwNiG_cK?;Q#!fbT92FSRw%Rx40Tb=(=YWvAda$&Roa|~r0c|6l?d=Xo__4a1YkMct!Ojkkt<5fYE~fA{2oh+p<K$hG*m4}xfEOb7Sh%$JO-{vhsw$@u2jzV8WIK?lmzF1c9-ZB?2eke=iEGFdEu8rK@jaVIjV`jp26uGzJ#sr}srL#k`heJi4}y#+hHJL7h#50$T&;G^M;?IOhiLb#xdyQgo_c)hE$o0(y!v{?cB?3UPCG1IRUTf4co5osHOI6g#`7-{G)Fo`R9G6gM1fpprlWM%i~8`~*xVZQhv*%g57W&gE`TAyrK1fOl1|Ry9trV)M3V69*36?bKn&G^y^cn%(A~0j1TF!TP<lfb3rF+0?SvZY9CX^TiY&JPyFom}%Z+IeN=I3K4TAnRS3{UA=m>Wb;SY0{Vk#gy4fojeAr#r;(?|^@tiGSn3dUz!s!`H5^1-;k34<0+kD6-c;4X(5H+O}PPRjJcA#fL_`z+iQkXui`(?U~)zoUpmf0-)-h1S@=fWiA0BJi_T(Qp8(%WO6<g~NWp%7W-vn2obTF*D5V!EMQ*98`q#RH_|_^+n<HzgImH<D`O2S0qJTPXetGXAbD4>3T?)mltn7z_IhiWoh~*%_;gel~bENP%n*`UxPHVhhv0|T_tm5M4>NlAN81^bky5;x4E;mP!tGq`Wo81z5S-^yv$OjLB;1(gvznCS~e&AA`mN^bAD)`ET}pPC+U8)->bP?h|U5m`VSez{~HK2(zR3tS=HB{U!>X^C0#sKPwTk>r~)<uWvy*pdsVnzGX5?W9>=2z+RYyTq;4KfwkoYR>MpWjk_*=OcdM$xD{~j%p{;#!K3%`ZSO-&V1NnOXJC?s+UY_IOByTyLzSte;#?|uc8bz|V>lB}C^ujjHDMqtxZaINX@0i06IIdCwpl9@l4EmN2X5?UjZnyOVvXdgL;bMv<rpPh-(vbO0g~CZei$7J=!lBd)*KAM{vb98T5sjq-tol5DETPYTFn}k*#wSszod9cD+>z+Yro_vwSG^cZ!I4>u%6xDMvsw0o=-}ZC)8zqp*EmS8cs2UYJ*M{dEQrCqVIS^#=Hfr{KxCf'}

EMBEDDED_SHA256 = {'audit_snapshot.py': 'ae45721c3bdde5fe22803aba7d7cbfd16221de26aa346540c5df52035186cad6',
 'checkpoint_snapshot.py': '8ecff6c935274743509f8ca1b1a0a85dd612b58f02240c79e6ae9cab03e18ff6',
 'corpus_runner_snapshot.py': '4ffa504f75c366f4884e0c66424a830cdbd56d8f764315038cbc01604e6a08c6',
 'crawler_snapshot.py': 'e05b4d47a9ba09eff94c32bb61d5c99bf9939f157b5d6aa1fa8bb14ceacb7d15',
 'diagnostics_snapshot.py': '894a6027eae99385aef49755d02eac66ad2053cd8e60c6c0e4573f66534e16de',
 'export_snapshot.py': 'e924cfa0a104928aa7d90826118955bbe49bbdca833994bb42746bf7e9c8ba30',
 'longchau_snapshot.py': '287b8cd009ff5e558f2a0e7c5592dd1b430a5e144da23619087c745f57287a3f',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '626e493ad05d4f6c96b257167c2cb81a815614122ce35272317ced8d5240f37d',
 'pubmed_efetch_snapshot.py': '983c7c3c1350c4e3a157a5b1b6b004a85f81074454ced5ffb7476f531f4cc3c7',
 'pubmed_resume_snapshot.py': '2c43d37a923335a61cc03d41632c340668ac8cc45afca15f24ebdcddc36220db'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'pubmed_resume_snapshot.py', 'checkpoint_snapshot.py', 'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}
HAS_VOV_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_VOV_SAMPLE = any((urlsplit(json.loads(line)['url']).hostname or '').lower() == 'vov.vn'
            or (urlsplit(json.loads(line)['url']).hostname or '').lower().endswith('.vov.vn')
            for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_VOV_SAMPLE and VOV_USE_CURL and (RUN_DIR / 'corpus_config.json').exists():
    saved_vov_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name,key in (('priority_probe_snapshot.py','probe_sha256'),('corpus_runner_snapshot.py','runner_sha256')):
        if saved_vov_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run VOV cũ cần RUN_TAG mới + RECOVERY_RUN_PATH để chuyển sang curl/Chrome; giữ raw/records/cooldown.')

HAS_LONGCHAU_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_LONGCHAU_SAMPLE = any(urlsplit(json.loads(line)['url']).hostname in
            ('nhathuoclongchau.com.vn', 'www.nhathuoclongchau.com.vn') for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('longchau_snapshot.py', 'longchau_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256'), ('priority_probe_snapshot.py', 'probe_sha256')):
        if saved_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run Long Châu cũ cần recovery: dùng ZIP đầy đủ, RECOVERY_RUN_PATH + RUN_TAG mới; '
                             'không ép RESUME vì parser/runner đã đổi. Raw và records cũ được giữ nguyên.')

for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (name == 'longchau_snapshot.py' and POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE) or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
resume_spec = importlib.util.spec_from_file_location('pubmed_resume_upgrade', RUN_DIR / 'pubmed_resume_snapshot.py')
PUBMED_RESUME = importlib.util.module_from_spec(resume_spec)
resume_spec.loader.exec_module(PUBMED_RESUME)
if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and PUBMED_UPGRADE_EFETCH_SNAPSHOT:
    upgrade_status = PUBMED_RESUME.upgrade(RUN_DIR,
        zlib.decompress(base64.b85decode(EMBEDDED_FILES['pubmed_efetch_snapshot.py'])),
        {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}})
    print('EFetch snapshot:', json.dumps(upgrade_status, ensure_ascii=False), flush=True)

CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

PubMed mặc định dùng **EFetch GET**, **100 PMID/batch**, **1 worker**, nghỉ ít nhất **0,5 giây giữa request**.
`PUBMED_USE_EFETCH = True` hoạt động cả khi `BROWSER_TEST_MODE = 'off'`; run chỉ có PubMed không khởi động Chrome.
Giữ ID/URL gốc, ghép theo PMID trong XML; `text` có title và toàn bộ abstract cùng nhãn.
Các website ngoài PubMed, Long Châu và VOV dùng Requests rồi Selenium theo cấu hình browser; `RESPECT_ROBOTS` áp dụng lượt crawl website.

Checkpoint XML và batch được lưu trước các record ID. Resume sửa các record còn thiếu mà không GET lại batch đã lưu.
`MAX_NEW` tính số URL corpus mới đã lên lịch, gồm cả PubMed; retry không tính hai lần.
HTTP 400 chỉ retry khi body báo Timeout/Empty Response; HTTP 400 invalid UID/parameter không retry mù quáng.
5 attempts, read timeout 90 giây, budget 300 giây/request batch, backoff + jitter và Retry-After.
Batch còn lỗi chia 100 → 50 → 25 → … để cô lập, giới hạn tổng request/time; sau đó tiếp tục batch khác.
Cuối lượt thử pending thêm một vòng. Cooldown server dài hoặc 3 batch liên tiếp không lấy được bản ghi
thì hoãn để resume; không ném RuntimeError vì riêng một batch lỗi, không đánh dấu complete khi còn pending.
PMID không được API trả về hoặc không có abstract được ghi riêng, không đưa vào corpus accepted.
Xem `pubmed_results.csv`, `pubmed_missing_pmids.csv`, `pubmed_no_abstract.csv`, `pubmed_review.csv`,
`pubmed_pending.csv`, `pubmed_requests.jsonl`, `pubmed_efetch_summary.json` và `raw/efetch/` trong ZIP đầy đủ.

Run mới có hậu tố `_efetch`. Nâng cấp run Selenium cũ bằng `RUN_TAG` mới và `RECOVERY_RUN_PATH`:
reuse raw hợp lệ của domain khác; PubMed HTML cũ được lấy mới qua EFetch.
Muốn thử lại PMID thiếu/no abstract cần run mới; resume giữ những kết quả XML hợp lệ đã hoàn tất.
`POSTPROCESS_ONLY = True` vẫn audit/export run cũ mà không crawl lại.
Resume EFetch cũ tự nâng cấp transport tương thích, giữ nguyên XML/record/schema và các snapshot crawl khác.
Checkpoint full ZIP/HF chạy định kỳ trong lúc crawl, khác với upload corpus cuối cùng sau audit/export.

Tham khảo [NCBI EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

Long Châu mặc định dùng `curl_cffi`/Chrome, 1 worker, delay 5 giây; không chạy Selenium cho domain này. Parser giữ sapo/nội dung chính; `/benh/` lấy đầy đủ các trường HTML y khoa từ JSON đã xác minh slug/canonical. Xem `longchau_results.csv`, `longchau_summary.json`; parser v2 loại nhóm liên kết “Xem thêm”. Run mới có hậu tố `_longchau_v2`; dùng recovery từ ZIP đầy đủ để tái trích HTML 200 cũ, giữ cooldown. Checkpoint thuộc runner chung: lưu theo cadence cho mọi engine và khi dừng/ngắt; log có tiến độ chung theo ID.

VOV (`vov.vn`, `vov2.vov.vn` và subdomain) mặc định dùng `curl_cffi` với `impersonate="chrome"`, một request VOV tại một thời điểm. Giữ parser hiện có, timeout/budget/robots, raw và ID/URL. Không chạy Selenium cho URL VOV khi `VOV_USE_CURL=True`; 429 dừng nhóm VOV và lưu `vov_request_gate.json` qua resume/recovery. Run VOV cũ dùng RUN_TAG mới + RECOVERY_RUN_PATH.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        # Pass credentials to the subprocess through its environment; settings/log contain none.
        for name, value in [('NCBI_EMAIL', NCBI_EMAIL), ('NCBI_API_KEY', NCBI_API_KEY)]:
            if value:
                os.environ[name] = value
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'pubmed_use_efetch': PUBMED_USE_EFETCH,
            'longchau_use_curl': LONGCHAU_USE_CURL,
            'vov_use_curl': VOV_USE_CURL,
            'longchau': {'workers': LONGCHAU_WORKERS, 'delay': LONGCHAU_DELAY, 'attempts': LONGCHAU_ATTEMPTS,
                          'timeout': [LONGCHAU_CONNECT_TIMEOUT, LONGCHAU_READ_TIMEOUT],
                          'budget': LONGCHAU_REQUEST_BUDGET, 'impersonate': 'chrome',
                          'retry_failed_on_resume': LONGCHAU_RETRY_FAILED_ON_RESUME},
            'pubmed_efetch': {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
                               'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
                               'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT],
                               'budget': PUBMED_EFETCH_BUDGET},
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        PUBMED_CONFIG_CURRENT = {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}}
        corpus_settings['pubmed_efetch'] = PUBMED_CONFIG_CURRENT
        if PUBMED_UPGRADE_EFETCH_SNAPSHOT:
            corpus_settings = PUBMED_RESUME.runtime_settings(RUN_DIR, corpus_settings)
        if CHECKPOINT_ENABLED and CHECKPOINT_UPLOAD_ENABLED and not os.environ.get('HF_TOKEN'):
            try:
                from kaggle_secrets import UserSecretsClient
                os.environ['HF_TOKEN'] = UserSecretsClient().get_secret(CHECKPOINT_HF_SECRET_NAME)
            except Exception:
                print('Chưa có HF_TOKEN: checkpoint local vẫn lưu; backup HF chưa khả dụng.')
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
